In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1

/content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
!pip install numpy pandas scipy

In [ ]:
!python smoke_test.py

0 1.723291039466858 128 0.1523188352584839
1 2.386245012283325 18623 0.27441704273223877
2 2.4616775512695312 25882 0.37187179923057556
3 2.655210018157959 39833 0.4496717154979706
4 2.5280051231384277 52504 0.5120396018028259
5 2.036310911178589 65246 0.40964215993881226
6 1.6486269235610962 77109 0.32772764563560486
7 1.3415077924728394 86562 0.2621987760066986
8 1.0969291925430298 93947 0.20977771282196045
9 0.901048481464386 99734 0.16784214973449707
10 0.7432432770729065 103987 0.134294331073761
11 0.6153632402420044 106962 0.10745608806610107
12 0.5111439824104309 108834 0.08598501235246658
13 0.42575833201408386 110055 0.06880727410316467
14 0.3554697632789612 110467 0.06048284471035004
15 0.2973681390285492 110173 0.05476268380880356
16 0.2491716593503952 109441 0.04918661713600159
17 0.20907552540302277 108191 0.04387429356575012
18 0.17564086616039276 106580 0.03890303149819374
19 0.1477099359035492 104479 0.034317418932914734


In [ ]:
%%writefile benchmark_memory_v01.py

import numpy as np
from scipy import sparse
from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


SEED = 20260914
rng = np.random.default_rng(SEED)

print("Cargando MaleCNS...")
g = MaleCNSGraph()

print("Neuronas:", g.n_nodes)
print("Conexiones:", g.n_edges)


# -------------------------------------------------
# 1. Crear control aleatorizado
# -------------------------------------------------
# Conserva:
# - número de neuronas
# - número de conexiones
# - pesos anatómicos
# - grado de salida aproximadamente
#
# Pero destruye gran parte de la organización
# específica del conectoma.

print("\nCreando control aleatorizado...")

A = g.A.tocoo()

random_targets = A.col.copy()
rng.shuffle(random_targets)

A_random = sparse.coo_matrix(
    (A.data.copy(), (A.row.copy(), random_targets)),
    shape=A.shape
).tocsr()

A_random.sum_duplicates()


class RandomGraph:
    pass


gr = RandomGraph()

gr.A = A_random
gr.nodes = g.nodes
gr.n_nodes = A_random.shape[0]
gr.n_edges = A_random.nnz

gr.in_strength = np.asarray(
    A_random.sum(axis=0)
).ravel().astype(np.float32)

gr.out_strength = np.asarray(
    A_random.sum(axis=1)
).ravel().astype(np.float32)


print("Conexiones control:", gr.n_edges)


# -------------------------------------------------
# 2. Entradas y neuronas de lectura
# -------------------------------------------------

N_INPUT = 256
N_READOUT = 4096

input_nodes = rng.choice(
    g.n_nodes,
    size=N_INPUT,
    replace=False
)

remaining = np.setdiff1d(
    np.arange(g.n_nodes),
    input_nodes
)

readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)


# -------------------------------------------------
# 3. Generar secuencias binarias
# -------------------------------------------------

N_SAMPLES = 300
SEQUENCE_LENGTH = 20
DELAYS = [1, 2, 4, 8, 12, 16]

sequences = rng.choice(
    [-1.0, 1.0],
    size=(N_SAMPLES, SEQUENCE_LENGTH)
).astype(np.float32)


# -------------------------------------------------
# 4. Obtener estados del cerebro
# -------------------------------------------------

def generate_states(graph, sequences):

    reservoir = ConnectomeReservoir(
        graph,
        leak=0.2,
        gain=1.2,
        sign_mode="biological_fast"
    )

    all_states = []

    for n, seq in enumerate(sequences):

        reservoir.reset()

        states = []

        for value in seq:

            external = np.zeros(
                graph.n_nodes,
                dtype=np.float32
            )

            external[input_nodes] = value

            x = reservoir.step(external)

            states.append(
                x[readout_nodes].copy()
            )

        all_states.append(states)

        if (n + 1) % 25 == 0:
            print(
                f"Procesadas {n+1}/{len(sequences)} secuencias"
            )

    return np.asarray(
        all_states,
        dtype=np.float32
    )


print("\nEjecutando MaleCNS real...")
states_real = generate_states(
    g,
    sequences
)

print("\nEjecutando red aleatorizada...")
states_random = generate_states(
    gr,
    sequences
)


# -------------------------------------------------
# 5. Ridge regression simple
# -------------------------------------------------

def ridge_train(X, y, alpha=1e-2):

    XTX = X.T @ X

    XTX.flat[::XTX.shape[0] + 1] += alpha

    return np.linalg.solve(
        XTX,
        X.T @ y
    )


def evaluate_memory(states, sequences, delay):

    X = []
    y = []

    for sample in range(N_SAMPLES):

        for t in range(delay, SEQUENCE_LENGTH):

            X.append(
                states[sample, t]
            )

            y.append(
                sequences[sample, t-delay]
            )

    X = np.asarray(
        X,
        dtype=np.float32
    )

    y = np.asarray(
        y,
        dtype=np.float32
    )

    split = int(
        len(X) * 0.7
    )

    X_train = X[:split]
    y_train = y[:split]

    X_test = X[split:]
    y_test = y[split:]

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-6

    X_train = (
        X_train - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std

    w = ridge_train(
        X_train,
        y_train
    )

    pred = X_test @ w

    predicted_binary = np.where(
        pred >= 0,
        1.0,
        -1.0
    )

    accuracy = np.mean(
        predicted_binary == y_test
    )

    correlation = np.corrcoef(
        pred,
        y_test
    )[0,1]

    return accuracy, correlation


# -------------------------------------------------
# 6. Resultados
# -------------------------------------------------

print("\n")
print("=" * 60)
print("MEMORY BENCHMARK v0.1")
print("=" * 60)

print(
    f"{'Delay':<8}"
    f"{'MaleCNS':<15}"
    f"{'Random':<15}"
    f"{'Δ':<10}"
)

results = []

for delay in DELAYS:

    acc_real, corr_real = evaluate_memory(
        states_real,
        sequences,
        delay
    )

    acc_rand, corr_rand = evaluate_memory(
        states_random,
        sequences,
        delay
    )

    delta = acc_real - acc_rand

    results.append(
        (
            delay,
            acc_real,
            acc_rand,
            delta,
            corr_real,
            corr_rand
        )
    )

    print(
        f"{delay:<8}"
        f"{acc_real:<15.4f}"
        f"{acc_rand:<15.4f}"
        f"{delta:+.4f}"
    )


print("\nCorrelaciones:")

for r in results:

    print(
        f"delay={r[0]:2d} | "
        f"MaleCNS={r[4]:.4f} | "
        f"Random={r[5]:.4f}"
    )

Writing benchmark_memory_v01.py


In [ ]:
!python benchmark_memory_v01.py

Cargando MaleCNS...
Neuronas: 165122
Conexiones: 25563197

Creando control aleatorizado...
Conexiones control: 25475215

Ejecutando MaleCNS real...
Procesadas 25/300 secuencias
Procesadas 50/300 secuencias
Procesadas 75/300 secuencias
Procesadas 100/300 secuencias
Procesadas 125/300 secuencias
Procesadas 150/300 secuencias
Procesadas 175/300 secuencias
Procesadas 200/300 secuencias
Procesadas 225/300 secuencias
Procesadas 250/300 secuencias
Procesadas 275/300 secuencias
Procesadas 300/300 secuencias

Ejecutando red aleatorizada...
Procesadas 25/300 secuencias
Procesadas 50/300 secuencias
Procesadas 75/300 secuencias
Procesadas 100/300 secuencias
Procesadas 125/300 secuencias
Procesadas 150/300 secuencias
Procesadas 175/300 secuencias
Procesadas 200/300 secuencias
Procesadas 225/300 secuencias
Procesadas 250/300 secuencias
Procesadas 275/300 secuencias
Procesadas 300/300 secuencias


MEMORY BENCHMARK v0.1
Delay   MaleCNS        Random         Δ         
1       0.9936         0.9825    

In [ ]:
!pip -q install igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 21.2 MB/s eta 0:00:00


In [ ]:
%%writefile build_degree_matched_v02.py

import gc
import numpy as np
from scipy import sparse
import igraph as ig

from loader import MaleCNSGraph


print("=" * 70)
print("MaleCNS-AI — construcción del control degree-matched v0.2")
print("=" * 70)

g = MaleCNSGraph()

# ------------------------------------------------------------
# 1. MaleCNS binario
# ------------------------------------------------------------
# En este experimento ignoramos temporalmente el número de
# contactos sinápticos. Cada conexión vale 1.
#
# Así aislamos la pregunta:
# ¿aporta algo la TOPOLOGÍA por sí misma?
# ------------------------------------------------------------

print("\nConstruyendo MaleCNS binario...")

A_real = g.A.copy().tocsr()

A_real.data[:] = 1.0

# Para este control trabajamos sin autoconexiones.
A_real.setdiag(0)
A_real.eliminate_zeros()
A_real.sum_duplicates()
A_real.sort_indices()

A_real = A_real.astype(np.float32)

n = A_real.shape[0]

out_degree = np.diff(A_real.indptr).astype(np.int32)

in_degree = np.bincount(
    A_real.indices,
    minlength=n
).astype(np.int32)

print("Neuronas:", f"{n:,}")
print("Conexiones:", f"{A_real.nnz:,}")
print("Suma out-degree:", f"{out_degree.sum():,}")
print("Suma in-degree :", f"{in_degree.sum():,}")

assert out_degree.sum() == in_degree.sum()
assert out_degree.sum() == A_real.nnz


# ------------------------------------------------------------
# 2. Grafo aleatorio EXACTAMENTE degree-matched
# ------------------------------------------------------------
# igraph genera un grafo dirigido simple conservando:
#
#   out-degree(i)
#   in-degree(i)
#
# para TODAS las neuronas.
# ------------------------------------------------------------

print("\nGenerando red degree-matched exacta...")
print("Esta es la parte pesada; puede tardar varios minutos.")

null_graph = ig.Graph.Degree_Sequence(
    out_degree.tolist(),
    in_degree.tolist(),
    method="edge_switching_simple"
)

print("Grafo generado.")
print("Vertices:", f"{null_graph.vcount():,}")
print("Edges   :", f"{null_graph.ecount():,}")

assert null_graph.vcount() == n
assert null_graph.ecount() == A_real.nnz
assert null_graph.is_simple()


# ------------------------------------------------------------
# 3. Pasar a scipy sparse
# ------------------------------------------------------------

print("\nConvirtiendo a matriz sparse...")

A_null = null_graph.get_adjacency_sparse().astype(
    np.float32
).tocsr()

A_null.sum_duplicates()
A_null.sort_indices()

del null_graph
gc.collect()


# ------------------------------------------------------------
# 4. Verificación rigurosa
# ------------------------------------------------------------

null_out = np.diff(
    A_null.indptr
).astype(np.int32)

null_in = np.bincount(
    A_null.indices,
    minlength=n
).astype(np.int32)

out_equal = np.array_equal(
    out_degree,
    null_out
)

in_equal = np.array_equal(
    in_degree,
    null_in
)

print("\n" + "=" * 70)
print("VALIDACIÓN")
print("=" * 70)

print("MaleCNS edges :", f"{A_real.nnz:,}")
print("Control edges :", f"{A_null.nnz:,}")

print("Out-degree exactamente igual:", out_equal)
print("In-degree exactamente igual :", in_equal)

print(
    "Máxima diferencia out-degree:",
    int(np.max(np.abs(out_degree - null_out)))
)

print(
    "Máxima diferencia in-degree :",
    int(np.max(np.abs(in_degree - null_in)))
)

assert A_real.nnz == A_null.nnz
assert out_equal
assert in_equal


# ------------------------------------------------------------
# 5. Guardar ambos
# ------------------------------------------------------------

print("\nGuardando matrices...")

sparse.save_npz(
    "adjacency_binary_real_v02.npz",
    A_real,
    compressed=True
)

sparse.save_npz(
    "adjacency_degree_matched_v02.npz",
    A_null,
    compressed=True
)

print("\nLISTO.")
print("Control degree-matched v0.2 generado correctamente.")

Writing build_degree_matched_v02.py


In [ ]:
!python build_degree_matched_v02.py

MaleCNS-AI — construcción del control degree-matched v0.2

Construyendo MaleCNS binario...
Neuronas: 165,122
Conexiones: 25,563,096
Suma out-degree: 25,563,096
Suma in-degree : 25,563,096

Generando red degree-matched exacta...
Esta es la parte pesada; puede tardar varios minutos.
Grafo generado.
Vertices: 165,122
Edges   : 25,563,096

Convirtiendo a matriz sparse...

VALIDACIÓN
MaleCNS edges : 25,563,096
Control edges : 25,563,096
Out-degree exactamente igual: True
In-degree exactamente igual : True
Máxima diferencia out-degree: 0
Máxima diferencia in-degree : 0

Guardando matrices...

LISTO.
Control degree-matched v0.2 generado correctamente.


In [ ]:
%%writefile benchmark_memory_v02.py

import numpy as np
import pandas as pd

from scipy import sparse

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


SEED = 20260914
rng = np.random.default_rng(SEED)

# ============================================================
# CONFIGURACIÓN
# ============================================================

N_SAMPLES = 160
SEQUENCE_LENGTH = 70

N_INPUT = 256
N_READOUT = 512

MAX_DELAY = 30

TRAIN_FRACTION = 0.70

RIDGE_ALPHA = 1e-2


# ============================================================
# CARGAR MaleCNS
# ============================================================

print("=" * 70)
print("MaleCNS-AI MEMORY BENCHMARK v0.2")
print("=" * 70)

base = MaleCNSGraph()

A_real = sparse.load_npz(
    "adjacency_binary_real_v02.npz"
).tocsr().astype(np.float32)

A_null = sparse.load_npz(
    "adjacency_degree_matched_v02.npz"
).tocsr().astype(np.float32)


# ============================================================
# OBJETO DE GRAFO PARA DYNAMICS.PY
# ============================================================

class GraphProxy:

    def __init__(self, A, nodes):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)

    @property
    def n_nodes(self):
        return self.A.shape[0]

    @property
    def n_edges(self):
        return self.A.nnz


real = GraphProxy(
    A_real,
    base.nodes
)

null = GraphProxy(
    A_null,
    base.nodes
)


print("\nNeuronas:", f"{real.n_nodes:,}")

print(
    "MaleCNS connections:",
    f"{real.n_edges:,}"
)

print(
    "Degree-matched connections:",
    f"{null.n_edges:,}"
)


# ============================================================
# COMPROBAR GRADOS OTRA VEZ
# ============================================================

real_out = np.diff(
    real.A.indptr
)

null_out = np.diff(
    null.A.indptr
)

real_in = np.bincount(
    real.A.indices,
    minlength=real.n_nodes
)

null_in = np.bincount(
    null.A.indices,
    minlength=null.n_nodes
)

assert np.array_equal(
    real_out,
    null_out
)

assert np.array_equal(
    real_in,
    null_in
)

print("Degree sequence: EXACT MATCH")


# ============================================================
# MISMAS NEURONAS DE INPUT Y READOUT
# ============================================================

input_nodes = rng.choice(
    real.n_nodes,
    size=N_INPUT,
    replace=False
)

remaining = np.setdiff1d(
    np.arange(real.n_nodes),
    input_nodes
)

readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)


# ============================================================
# DATOS
# ============================================================

sequences = rng.choice(
    [-1.0, 1.0],
    size=(
        N_SAMPLES,
        SEQUENCE_LENGTH
    )
).astype(np.float32)


# ============================================================
# EJECUCIÓN DEL CEREBRO
# ============================================================

def generate_states(graph, name):

    print(f"\nEjecutando {name}...")

    reservoir = ConnectomeReservoir(
        graph,
        leak=0.2,
        gain=1.2,
        sign_mode="biological_fast"
    )

    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )

    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )

    for sample in range(N_SAMPLES):

        reservoir.reset()

        seq = sequences[sample]

        for t, value in enumerate(seq):

            external.fill(0.0)

            external[input_nodes] = value

            x = reservoir.step(
                external
            )

            states[
                sample,
                t,
                :
            ] = x[readout_nodes]

        if (sample + 1) % 20 == 0:

            print(
                f"{name}: "
                f"{sample+1}/{N_SAMPLES}"
            )

    return states


states_real = generate_states(
    real,
    "MaleCNS"
)

states_null = generate_states(
    null,
    "DegreeMatched"
)


# ============================================================
# SPLIT POR SECUENCIAS
# ============================================================
#
# No mezclamos estados de una misma secuencia entre train/test.
# Esto evita leakage.
# ============================================================

n_train = int(
    N_SAMPLES * TRAIN_FRACTION
)

train_ids = np.arange(
    n_train
)

test_ids = np.arange(
    n_train,
    N_SAMPLES
)


# ============================================================
# MATRICES X,Y
# ============================================================
#
# Un único readout predice simultáneamente:
#
# x(t-1)
# x(t-2)
# ...
# x(t-30)
#
# Todos los delays usan EXACTAMENTE los mismos ejemplos.
# ============================================================

def build_dataset(
    states,
    ids
):

    X = []

    Y = []

    delays = np.arange(
        1,
        MAX_DELAY + 1
    )

    for sample in ids:

        seq = sequences[sample]

        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[
                    sample,
                    t,
                    :
                ]
            )

            target = seq[
                t - delays
            ]

            Y.append(
                target
            )

    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            Y,
            dtype=np.float32
        )
    )


# ============================================================
# RIDGE MULTIOUTPUT
# ============================================================

def evaluate(
    states,
    name
):

    print(
        f"\nEntrenando readout de {name}..."
    )

    X_train, Y_train = build_dataset(
        states,
        train_ids
    )

    X_test, Y_test = build_dataset(
        states,
        test_ids
    )

    print(
        "Train:",
        X_train.shape,
        Y_train.shape
    )

    print(
        "Test :",
        X_test.shape,
        Y_test.shape
    )

    mean = X_train.mean(
        axis=0
    )

    std = X_train.std(
        axis=0
    ) + 1e-6

    X_train = (
        X_train - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std


    # -------------------------------
    # Ridge
    # -------------------------------

    XTX = (
        X_train.T
        @ X_train
    ).astype(np.float64)

    XTY = (
        X_train.T
        @ Y_train
    ).astype(np.float64)

    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA

    W = np.linalg.solve(
        XTX,
        XTY
    )

    pred = (
        X_test.astype(np.float64)
        @ W
    )


    # -------------------------------
    # Métricas delay por delay
    # -------------------------------

    results = []

    for d in range(
        MAX_DELAY
    ):

        y = Y_test[:, d]

        p = pred[:, d]

        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )

        accuracy = float(
            np.mean(
                binary == y
            )
        )

        corr = float(
            np.corrcoef(
                p,
                y
            )[0, 1]
        )

        memory = (
            corr ** 2
            if np.isfinite(corr)
            else 0.0
        )

        mse = float(
            np.mean(
                (p - y) ** 2
            )
        )

        results.append({
            "delay": d + 1,
            "accuracy": accuracy,
            "corr": corr,
            "memory": memory,
            "mse": mse
        })

    return pd.DataFrame(
        results
    )


res_real = evaluate(
    states_real,
    "MaleCNS"
)

res_null = evaluate(
    states_null,
    "DegreeMatched"
)


# ============================================================
# COMPARACIÓN
# ============================================================

comparison = pd.DataFrame({

    "delay":
        res_real["delay"],

    "acc_malecns":
        res_real["accuracy"],

    "acc_control":
        res_null["accuracy"],

    "corr_malecns":
        res_real["corr"],

    "corr_control":
        res_null["corr"],

    "mc_malecns":
        res_real["memory"],

    "mc_control":
        res_null["memory"],
})


comparison[
    "delta_acc"
] = (
    comparison["acc_malecns"]
    -
    comparison["acc_control"]
)

comparison[
    "delta_corr"
] = (
    comparison["corr_malecns"]
    -
    comparison["corr_control"]
)


# ============================================================
# MEMORY CAPACITY
# ============================================================

MC_REAL = float(
    comparison[
        "mc_malecns"
    ].sum()
)

MC_NULL = float(
    comparison[
        "mc_control"
    ].sum()
)


# ============================================================
# RESULTADOS
# ============================================================

print("\n")
print("=" * 78)

print(
    "MEMORY BENCHMARK v0.2 — "
    "TOPOLOGY ONLY"
)

print("=" * 78)

print(
    f"{'D':<4}"
    f"{'Acc Bio':<11}"
    f"{'Acc Null':<11}"
    f"{'ΔAcc':<10}"
    f"{'Corr Bio':<12}"
    f"{'Corr Null':<12}"
)

print("-" * 78)

for _, r in comparison.iterrows():

    print(
        f"{int(r.delay):<4}"
        f"{r.acc_malecns:<11.4f}"
        f"{r.acc_control:<11.4f}"
        f"{r.delta_acc:+.4f}    "
        f"{r.corr_malecns:<12.4f}"
        f"{r.corr_control:<12.4f}"
    )


print("\n" + "=" * 78)

print("MEMORY CAPACITY")

print("=" * 78)

print(
    "MaleCNS:",
    round(MC_REAL, 4)
)

print(
    "Degree-matched:",
    round(MC_NULL, 4)
)

print(
    "Δ:",
    round(
        MC_REAL - MC_NULL,
        4
    )
)

if MC_NULL > 0:

    print(
        "Relative advantage:",
        round(
            (
                MC_REAL
                /
                MC_NULL
                - 1
            ) * 100,
            2
        ),
        "%"
    )


# ============================================================
# GUARDAR
# ============================================================

comparison.to_csv(
    "memory_benchmark_v02.csv",
    index=False
)

print(
    "\nResultados guardados en "
    "memory_benchmark_v02.csv"
)

Writing benchmark_memory_v02.py


In [ ]:
!python benchmark_memory_v02.py

Traceback (most recent call last):
  File "/content/benchmark_memory_v02.py", line 7, in <module>
    from loader import MaleCNSGraph
ModuleNotFoundError: No module named 'loader'


In [ ]:
import os
import shutil

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

print("Carpeta existe:", os.path.exists(PROJECT))

# Copiar los archivos que acabamos de generar a la carpeta del proyecto
archivos = [
    "adjacency_binary_real_v02.npz",
    "adjacency_degree_matched_v02.npz",
    "benchmark_memory_v02.py"
]

for nombre in archivos:
    origen = f"/content/{nombre}"
    destino = f"{PROJECT}/{nombre}"

    if os.path.exists(origen):
        shutil.copy2(origen, destino)
        print("Copiado:", nombre)
    elif os.path.exists(destino):
        print("Ya estaba en proyecto:", nombre)
    else:
        print("NO ENCONTRADO:", nombre)

print("\nArchivos principales:")
for nombre in [
    "loader.py",
    "dynamics.py",
    "nodes.csv.gz",
    "adjacency_weighted.npz",
    "adjacency_binary_real_v02.npz",
    "adjacency_degree_matched_v02.npz",
    "benchmark_memory_v02.py"
]:
    print(nombre, "->", os.path.exists(f"{PROJECT}/{nombre}"))

Carpeta existe: False
NO ENCONTRADO: adjacency_binary_real_v02.npz
NO ENCONTRADO: adjacency_degree_matched_v02.npz


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/malecns_ai_v0_1/benchmark_memory_v02.py'

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

BASE = "/content/drive/MyDrive"

buscar = [
    "loader.py",
    "dynamics.py",
    "adjacency_weighted.npz",
    "nodes.csv.gz",
    "adjacency_binary_real_v02.npz",
    "adjacency_degree_matched_v02.npz",
    "malecns_ai_v0_1_core.zip"
]

print("\nBUSCANDO ARCHIVOS...\n")

encontrados = {}

for objetivo in buscar:
    encontrados[objetivo] = []

    for root, dirs, files in os.walk(BASE):
        if objetivo in files:
            ruta = os.path.join(root, objetivo)
            encontrados[objetivo].append(ruta)
            print(objetivo)
            print("   ", ruta)

print("\nFIN DE BÚSQUEDA")

Mounted at /content/drive

BUSCANDO ARCHIVOS...

loader.py
    /content/drive/MyDrive/malecns_ai_v0_1/loader.py
dynamics.py
    /content/drive/MyDrive/malecns_ai_v0_1/dynamics.py
adjacency_weighted.npz
    /content/drive/MyDrive/malecns_ai_v0_1/adjacency_weighted.npz
nodes.csv.gz
    /content/drive/MyDrive/malecns_ai_v0_1/nodes.csv.gz
adjacency_binary_real_v02.npz
    /content/drive/MyDrive/malecns_ai_v0_1/adjacency_binary_real_v02.npz
adjacency_degree_matched_v02.npz
    /content/drive/MyDrive/malecns_ai_v0_1/adjacency_degree_matched_v02.npz

FIN DE BÚSQUEDA


In [ ]:
loader_paths = encontrados["loader.py"]

if loader_paths:
    PROJECT = os.path.dirname(loader_paths[0])

    print("PROJECT =", PROJECT)

    os.chdir(PROJECT)

    print("\nContenido:")
    for x in os.listdir(PROJECT):
        print(x)

else:
    print("NO se encontró loader.py")

PROJECT = /content/drive/MyDrive/malecns_ai_v0_1

Contenido:
README.md
dynamics.py
nodes.csv.gz
metrics.json
smoke_test.py
adjacency_weighted.npz
LICENSE_DATA.txt
loader.py
__pycache__
benchmark_memory_v01.py
build_degree_matched_v02.py
adjacency_binary_real_v02.npz
adjacency_degree_matched_v02.npz


In [ ]:
from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir

g = MaleCNSGraph()

print("OK")
print("Neuronas:", g.n_nodes)
print("Conexiones:", g.n_edges)

OK
Neuronas: 165122
Conexiones: 25563197


In [ ]:
!python benchmark_memory_v02.py

python3: can't open file '/content/drive/MyDrive/malecns_ai_v0_1/benchmark_memory_v02.py': [Errno 2] No such file or directory


In [ ]:
!pip -q install igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 47.2 MB/s eta 0:00:00


In [ ]:
%%writefile build_degree_matched_v02.py

import gc
import numpy as np
from scipy import sparse
import igraph as ig

from loader import MaleCNSGraph


print("=" * 70)
print("MaleCNS-AI — construcción del control degree-matched v0.2")
print("=" * 70)

g = MaleCNSGraph()

# ------------------------------------------------------------
# 1. MaleCNS binario
# ------------------------------------------------------------
# En este experimento ignoramos temporalmente el número de
# contactos sinápticos. Cada conexión vale 1.
#
# Así aislamos la pregunta:
# ¿aporta algo la TOPOLOGÍA por sí misma?
# ------------------------------------------------------------

print("\nConstruyendo MaleCNS binario...")

A_real = g.A.copy().tocsr()

A_real.data[:] = 1.0

# Para este control trabajamos sin autoconexiones.
A_real.setdiag(0)
A_real.eliminate_zeros()
A_real.sum_duplicates()
A_real.sort_indices()

A_real = A_real.astype(np.float32)

n = A_real.shape[0]

out_degree = np.diff(A_real.indptr).astype(np.int32)

in_degree = np.bincount(
    A_real.indices,
    minlength=n
).astype(np.int32)

print("Neuronas:", f"{n:,}")
print("Conexiones:", f"{A_real.nnz:,}")
print("Suma out-degree:", f"{out_degree.sum():,}")
print("Suma in-degree :", f"{in_degree.sum():,}")

assert out_degree.sum() == in_degree.sum()
assert out_degree.sum() == A_real.nnz


# ------------------------------------------------------------
# 2. Grafo aleatorio EXACTAMENTE degree-matched
# ------------------------------------------------------------
# igraph genera un grafo dirigido simple conservando:
#
#   out-degree(i)
#   in-degree(i)
#
# para TODAS las neuronas.
# ------------------------------------------------------------

print("\nGenerando red degree-matched exacta...")
print("Esta es la parte pesada; puede tardar varios minutos.")

null_graph = ig.Graph.Degree_Sequence(
    out_degree.tolist(),
    in_degree.tolist(),
    method="edge_switching_simple"
)

print("Grafo generado.")
print("Vertices:", f"{null_graph.vcount():,}")
print("Edges   :", f"{null_graph.ecount():,}")

assert null_graph.vcount() == n
assert null_graph.ecount() == A_real.nnz
assert null_graph.is_simple()


# ------------------------------------------------------------
# 3. Pasar a scipy sparse
# ------------------------------------------------------------

print("\nConvirtiendo a matriz sparse...")

A_null = null_graph.get_adjacency_sparse().astype(
    np.float32
).tocsr()

A_null.sum_duplicates()
A_null.sort_indices()

del null_graph
gc.collect()


# ------------------------------------------------------------
# 4. Verificación rigurosa
# ------------------------------------------------------------

null_out = np.diff(
    A_null.indptr
).astype(np.int32)

null_in = np.bincount(
    A_null.indices,
    minlength=n
).astype(np.int32)

out_equal = np.array_equal(
    out_degree,
    null_out
)

in_equal = np.array_equal(
    in_degree,
    null_in
)

print("\n" + "=" * 70)
print("VALIDACIÓN")
print("=" * 70)

print("MaleCNS edges :", f"{A_real.nnz:,}")
print("Control edges :", f"{A_null.nnz:,}")

print("Out-degree exactamente igual:", out_equal)
print("In-degree exactamente igual :", in_equal)

print(
    "Máxima diferencia out-degree:",
    int(np.max(np.abs(out_degree - null_out)))
)

print(
    "Máxima diferencia in-degree :",
    int(np.max(np.abs(in_degree - null_in)))
)

assert A_real.nnz == A_null.nnz
assert out_equal
assert in_equal


# ------------------------------------------------------------
# 5. Guardar ambos
# ------------------------------------------------------------

print("\nGuardando matrices...")

sparse.save_npz(
    "adjacency_binary_real_v02.npz",
    A_real,
    compressed=True
)

sparse.save_npz(
    "adjacency_degree_matched_v02.npz",
    A_null,
    compressed=True
)

print("\nLISTO.")
print("Control degree-matched v0.2 generado correctamente.")

Overwriting build_degree_matched_v02.py


In [ ]:
!python build_degree_matched_v02.py

MaleCNS-AI — construcción del control degree-matched v0.2

Construyendo MaleCNS binario...
Neuronas: 165,122
Conexiones: 25,563,096
Suma out-degree: 25,563,096
Suma in-degree : 25,563,096

Generando red degree-matched exacta...
Esta es la parte pesada; puede tardar varios minutos.
Grafo generado.
Vertices: 165,122
Edges   : 25,563,096

Convirtiendo a matriz sparse...

VALIDACIÓN
MaleCNS edges : 25,563,096
Control edges : 25,563,096
Out-degree exactamente igual: True
In-degree exactamente igual : True
Máxima diferencia out-degree: 0
Máxima diferencia in-degree : 0

Guardando matrices...

LISTO.
Control degree-matched v0.2 generado correctamente.


In [ ]:
import os

archivos = [
    "adjacency_binary_real_v02.npz",
    "adjacency_degree_matched_v02.npz",
    "benchmark_memory_v02.py"
]

for a in archivos:
    print(a, "->", os.path.exists(a))

adjacency_binary_real_v02.npz -> True
adjacency_degree_matched_v02.npz -> True
benchmark_memory_v02.py -> False


In [ ]:
!python benchmark_memory_v02.py

python3: can't open file '/content/drive/MyDrive/malecns_ai_v0_1/benchmark_memory_v02.py': [Errno 2] No such file or directory


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1

import os

for f in [
    "loader.py",
    "dynamics.py",
    "adjacency_binary_real_v02.npz",
    "adjacency_degree_matched_v02.npz"
]:
    print(f, "->", os.path.exists(f))

/content
loader.py -> True
dynamics.py -> True
adjacency_binary_real_v02.npz -> True
adjacency_degree_matched_v02.npz -> True


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_memory_v02.py

import numpy as np
import pandas as pd
from scipy import sparse

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir

SEED = 20260914
rng = np.random.default_rng(SEED)

N_SAMPLES = 160
SEQUENCE_LENGTH = 70
N_INPUT = 256
N_READOUT = 512
MAX_DELAY = 30
TRAIN_FRACTION = 0.70
RIDGE_ALPHA = 1e-2

print("=" * 70)
print("MaleCNS-AI MEMORY BENCHMARK v0.2")
print("=" * 70)

base = MaleCNSGraph()

A_real = sparse.load_npz(
    "adjacency_binary_real_v02.npz"
).tocsr().astype(np.float32)

A_null = sparse.load_npz(
    "adjacency_degree_matched_v02.npz"
).tocsr().astype(np.float32)


class GraphProxy:

    def __init__(self, A, nodes):
        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)

    @property
    def n_nodes(self):
        return self.A.shape[0]

    @property
    def n_edges(self):
        return self.A.nnz


real = GraphProxy(A_real, base.nodes)
null = GraphProxy(A_null, base.nodes)

print("Neuronas:", f"{real.n_nodes:,}")
print("MaleCNS connections:", f"{real.n_edges:,}")
print("Degree-matched connections:", f"{null.n_edges:,}")


# Validación grados
real_out = np.diff(real.A.indptr)
null_out = np.diff(null.A.indptr)

real_in = np.bincount(
    real.A.indices,
    minlength=real.n_nodes
)

null_in = np.bincount(
    null.A.indices,
    minlength=null.n_nodes
)

assert np.array_equal(real_out, null_out)
assert np.array_equal(real_in, null_in)

print("Degree sequence: EXACT MATCH")


# Input / readout
input_nodes = rng.choice(
    real.n_nodes,
    size=N_INPUT,
    replace=False
)

remaining = np.setdiff1d(
    np.arange(real.n_nodes),
    input_nodes
)

readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)


# Secuencias
sequences = rng.choice(
    [-1.0, 1.0],
    size=(N_SAMPLES, SEQUENCE_LENGTH)
).astype(np.float32)


def generate_states(graph, name):

    print(f"\nEjecutando {name}...")

    reservoir = ConnectomeReservoir(
        graph,
        leak=0.2,
        gain=1.2,
        sign_mode="biological_fast"
    )

    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )

    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )

    for sample in range(N_SAMPLES):

        reservoir.reset()

        for t, value in enumerate(
            sequences[sample]
        ):

            external.fill(0.0)
            external[input_nodes] = value

            x = reservoir.step(external)

            states[
                sample,
                t,
                :
            ] = x[readout_nodes]

        if (sample + 1) % 20 == 0:
            print(
                f"{name}: "
                f"{sample+1}/{N_SAMPLES}"
            )

    return states


states_real = generate_states(
    real,
    "MaleCNS"
)

states_null = generate_states(
    null,
    "DegreeMatched"
)


# Train / test separados por secuencia
n_train = int(
    N_SAMPLES * TRAIN_FRACTION
)

train_ids = np.arange(n_train)
test_ids = np.arange(n_train, N_SAMPLES)


def build_dataset(states, ids):

    X = []
    Y = []

    delays = np.arange(
        1,
        MAX_DELAY + 1
    )

    for sample in ids:

        seq = sequences[sample]

        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[sample, t, :]
            )

            Y.append(
                seq[t - delays]
            )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(Y, dtype=np.float32)
    )


def evaluate(states, name):

    print(
        f"\nEntrenando readout: {name}"
    )

    X_train, Y_train = build_dataset(
        states,
        train_ids
    )

    X_test, Y_test = build_dataset(
        states,
        test_ids
    )

    print(
        "Train:",
        X_train.shape,
        Y_train.shape
    )

    print(
        "Test:",
        X_test.shape,
        Y_test.shape
    )

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-6

    X_train = (
        X_train - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std


    XTX = (
        X_train.T @ X_train
    ).astype(np.float64)

    XTY = (
        X_train.T @ Y_train
    ).astype(np.float64)

    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA

    W = np.linalg.solve(
        XTX,
        XTY
    )

    pred = (
        X_test.astype(np.float64)
        @ W
    )


    results = []

    for d in range(MAX_DELAY):

        y = Y_test[:, d]
        p = pred[:, d]

        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )

        accuracy = float(
            np.mean(binary == y)
        )

        corr = float(
            np.corrcoef(p, y)[0, 1]
        )

        memory = (
            corr ** 2
            if np.isfinite(corr)
            else 0.0
        )

        results.append({
            "delay": d + 1,
            "accuracy": accuracy,
            "corr": corr,
            "memory": memory
        })

    return pd.DataFrame(results)


res_real = evaluate(
    states_real,
    "MaleCNS"
)

res_null = evaluate(
    states_null,
    "DegreeMatched"
)


comparison = pd.DataFrame({
    "delay": res_real["delay"],
    "acc_malecns": res_real["accuracy"],
    "acc_control": res_null["accuracy"],
    "corr_malecns": res_real["corr"],
    "corr_control": res_null["corr"],
    "mc_malecns": res_real["memory"],
    "mc_control": res_null["memory"],
})

comparison["delta_acc"] = (
    comparison["acc_malecns"]
    - comparison["acc_control"]
)

comparison["delta_corr"] = (
    comparison["corr_malecns"]
    - comparison["corr_control"]
)


MC_REAL = float(
    comparison["mc_malecns"].sum()
)

MC_NULL = float(
    comparison["mc_control"].sum()
)


print("\n")
print("=" * 78)
print("MEMORY BENCHMARK v0.2 — TOPOLOGY ONLY")
print("=" * 78)

print(
    f"{'D':<4}"
    f"{'Acc Bio':<11}"
    f"{'Acc Null':<11}"
    f"{'ΔAcc':<10}"
    f"{'Corr Bio':<12}"
    f"{'Corr Null':<12}"
)

print("-" * 78)

for _, r in comparison.iterrows():

    print(
        f"{int(r.delay):<4}"
        f"{r.acc_malecns:<11.4f}"
        f"{r.acc_control:<11.4f}"
        f"{r.delta_acc:+.4f}    "
        f"{r.corr_malecns:<12.4f}"
        f"{r.corr_control:<12.4f}"
    )


print("\n" + "=" * 78)
print("MEMORY CAPACITY")
print("=" * 78)

print("MaleCNS:", round(MC_REAL, 4))
print("Degree-matched:", round(MC_NULL, 4))
print("Δ:", round(MC_REAL - MC_NULL, 4))

if MC_NULL > 0:
    print(
        "Relative advantage:",
        round(
            (
                MC_REAL / MC_NULL - 1
            ) * 100,
            2
        ),
        "%"
    )


comparison.to_csv(
    "memory_benchmark_v02.csv",
    index=False
)

print(
    "\nGuardado: memory_benchmark_v02.csv"
)

Writing /content/drive/MyDrive/malecns_ai_v0_1/benchmark_memory_v02.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1

/content


In [ ]:
import os

print(
    os.path.exists(
        "/content/drive/MyDrive/malecns_ai_v0_1/benchmark_memory_v02.py"
    )
)

True


In [ ]:
!python benchmark_memory_v02.py

MaleCNS-AI MEMORY BENCHMARK v0.2
Neuronas: 165,122
MaleCNS connections: 25,563,096
Degree-matched connections: 25,563,096
Degree sequence: EXACT MATCH

Ejecutando MaleCNS...
MaleCNS: 20/160
MaleCNS: 40/160
MaleCNS: 60/160
MaleCNS: 80/160
MaleCNS: 100/160
MaleCNS: 120/160
MaleCNS: 140/160
MaleCNS: 160/160

Ejecutando DegreeMatched...
DegreeMatched: 20/160
DegreeMatched: 40/160
DegreeMatched: 60/160
DegreeMatched: 80/160
DegreeMatched: 100/160
DegreeMatched: 120/160
DegreeMatched: 140/160
DegreeMatched: 160/160

Entrenando readout: MaleCNS
Train: (4480, 512) (4480, 30)
Test: (1920, 512) (1920, 30)

Entrenando readout: DegreeMatched
Train: (4480, 512) (4480, 30)
Test: (1920, 512) (1920, 30)


MEMORY BENCHMARK v0.2 — TOPOLOGY ONLY
D   Acc Bio    Acc Null   ΔAcc      Corr Bio    Corr Null   
------------------------------------------------------------------------------
1   0.9771     0.7969     +0.1802    0.8590      0.6945      
2   0.7594     0.7464     +0.0130    0.6013      0.6078      

In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1

/content


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_null_ensemble_v03a.py

import os
import gc
import random
import numpy as np
import pandas as pd
from scipy import sparse
import igraph as ig

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# CONFIGURACION
# ============================================================

BASE_SEED = 20260914

N_NULLS = 5

N_SAMPLES = 160
SEQUENCE_LENGTH = 70

N_INPUT = 256
N_READOUT = 512

MAX_DELAY = 30
TRAIN_FRACTION = 0.70
RIDGE_ALPHA = 1e-2

SIGN_MODE = "biological_fast"


# ============================================================
# CARGAR MaleCNS
# ============================================================

print("=" * 76)
print("MaleCNS-AI v0.3A — NULL ENSEMBLE")
print("=" * 76)

base = MaleCNSGraph()

A_real = sparse.load_npz(
    "adjacency_binary_real_v02.npz"
).tocsr().astype(np.float32)

n = A_real.shape[0]

out_degree = np.diff(
    A_real.indptr
).astype(np.int32)

in_degree = np.bincount(
    A_real.indices,
    minlength=n
).astype(np.int32)

print("Neuronas:", f"{n:,}")
print("Conexiones:", f"{A_real.nnz:,}")
print("Controles nulos:", N_NULLS)


# ============================================================
# GRAPH PROXY
# ============================================================

class GraphProxy:

    def __init__(self, A, nodes):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)

    @property
    def n_nodes(self):
        return self.A.shape[0]

    @property
    def n_edges(self):
        return self.A.nnz


real_graph = GraphProxy(
    A_real,
    base.nodes
)


# ============================================================
# MISMAS ENTRADAS / READOUT PARA TODOS
# ============================================================

rng = np.random.default_rng(
    BASE_SEED
)

input_nodes = rng.choice(
    n,
    size=N_INPUT,
    replace=False
)

remaining = np.setdiff1d(
    np.arange(n),
    input_nodes
)

readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)


# ============================================================
# MISMAS SECUENCIAS PARA TODOS
# ============================================================

sequences = rng.choice(
    [-1.0, 1.0],
    size=(
        N_SAMPLES,
        SEQUENCE_LENGTH
    )
).astype(np.float32)


# ============================================================
# EJECUTAR RESERVOIR
# ============================================================

def generate_states(graph, name):

    print(f"\nEjecutando {name}...")

    reservoir = ConnectomeReservoir(
        graph,
        leak=0.2,
        gain=1.2,
        sign_mode=SIGN_MODE
    )

    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )

    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )

    for sample in range(N_SAMPLES):

        reservoir.reset()

        for t, value in enumerate(
            sequences[sample]
        ):

            external.fill(0.0)

            external[
                input_nodes
            ] = value

            x = reservoir.step(
                external
            )

            states[
                sample,
                t,
                :
            ] = x[readout_nodes]

        if (sample + 1) % 20 == 0:

            print(
                f"{name}: "
                f"{sample+1}/{N_SAMPLES}"
            )

    return states


# ============================================================
# TRAIN / TEST
# ============================================================

n_train = int(
    N_SAMPLES
    * TRAIN_FRACTION
)

train_ids = np.arange(
    n_train
)

test_ids = np.arange(
    n_train,
    N_SAMPLES
)


def build_dataset(states, ids):

    X = []
    Y = []

    delays = np.arange(
        1,
        MAX_DELAY + 1
    )

    for sample in ids:

        seq = sequences[sample]

        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[
                    sample,
                    t,
                    :
                ]
            )

            Y.append(
                seq[
                    t - delays
                ]
            )

    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            Y,
            dtype=np.float32
        )
    )


# ============================================================
# EVALUACION
# ============================================================

def evaluate(states):

    X_train, Y_train = build_dataset(
        states,
        train_ids
    )

    X_test, Y_test = build_dataset(
        states,
        test_ids
    )

    mean = X_train.mean(
        axis=0
    )

    std = (
        X_train.std(axis=0)
        + 1e-6
    )

    X_train = (
        X_train - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std


    XTX = (
        X_train.T
        @ X_train
    ).astype(np.float64)

    XTY = (
        X_train.T
        @ Y_train
    ).astype(np.float64)

    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA

    W = np.linalg.solve(
        XTX,
        XTY
    )

    pred = (
        X_test.astype(np.float64)
        @ W
    )

    result = []

    for d in range(
        MAX_DELAY
    ):

        y = Y_test[:, d]
        p = pred[:, d]

        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )

        acc = float(
            np.mean(
                binary == y
            )
        )

        corr = float(
            np.corrcoef(
                p,
                y
            )[0, 1]
        )

        if not np.isfinite(corr):
            corr = 0.0

        mc = corr ** 2

        result.append({
            "delay": d + 1,
            "accuracy": acc,
            "corr": corr,
            "memory": mc,
        })

    df = pd.DataFrame(
        result
    )

    MC = float(
        df["memory"].sum()
    )

    return df, MC


# ============================================================
# MaleCNS REAL
# ============================================================

print("\n" + "=" * 76)
print("EJECUTANDO MaleCNS REAL")
print("=" * 76)

states_real = generate_states(
    real_graph,
    "MaleCNS"
)

real_delay, MC_REAL = evaluate(
    states_real
)

del states_real
gc.collect()

print(
    "\nMaleCNS MEMORY CAPACITY:",
    round(MC_REAL, 6)
)


# Guardar desde ya
real_delay["network"] = "MaleCNS"
real_delay["null_id"] = 0

delay_results = [
    real_delay
]

summary_rows = [{
    "network": "MaleCNS",
    "null_id": 0,
    "memory_capacity": MC_REAL
}]


# ============================================================
# GENERAR 5 CONTROLES
# ============================================================

out_list = out_degree.tolist()
in_list = in_degree.tolist()


for null_id in range(
    1,
    N_NULLS + 1
):

    seed = (
        BASE_SEED
        + null_id * 1000
    )

    print("\n")
    print("=" * 76)

    print(
        f"NULL {null_id}/{N_NULLS} "
        f"| seed={seed}"
    )

    print("=" * 76)


    # --------------------------------------------------------
    # Semilla de igraph
    # --------------------------------------------------------

    random.seed(seed)
    np.random.seed(seed)

    try:

        ig.set_random_number_generator(
            random.Random(seed)
        )

    except Exception:

        pass


    # --------------------------------------------------------
    # Grafo degree-matched
    # --------------------------------------------------------

    print(
        "Generando topología "
        "degree-matched..."
    )

    null_ig = ig.Graph.Degree_Sequence(
        out_list,
        in_list,
        method="edge_switching_simple"
    )

    print(
        "Edges igraph:",
        f"{null_ig.ecount():,}"
    )


    A_null = (
        null_ig
        .get_adjacency_sparse()
        .astype(np.float32)
        .tocsr()
    )

    del null_ig
    gc.collect()


    A_null.sum_duplicates()
    A_null.sort_indices()


    # --------------------------------------------------------
    # VALIDACION
    # --------------------------------------------------------

    null_out = np.diff(
        A_null.indptr
    ).astype(np.int32)

    null_in = np.bincount(
        A_null.indices,
        minlength=n
    ).astype(np.int32)


    assert A_null.nnz == A_real.nnz

    assert np.array_equal(
        out_degree,
        null_out
    )

    assert np.array_equal(
        in_degree,
        null_in
    )

    print(
        "Validación degree sequence: OK"
    )


    # --------------------------------------------------------
    # EJECUTAR
    # --------------------------------------------------------

    null_graph = GraphProxy(
        A_null,
        base.nodes
    )

    states_null = generate_states(
        null_graph,
        f"Null-{null_id}"
    )

    null_delay, MC_NULL = evaluate(
        states_null
    )

    print(
        f"\nNull-{null_id} "
        f"MEMORY CAPACITY:",
        round(MC_NULL, 6)
    )

    print(
        "Δ MaleCNS - Null:",
        round(
            MC_REAL - MC_NULL,
            6
        )
    )


    # --------------------------------------------------------
    # GUARDAR RESULTADO
    # --------------------------------------------------------

    null_delay[
        "network"
    ] = f"Null-{null_id}"

    null_delay[
        "null_id"
    ] = null_id

    delay_results.append(
        null_delay
    )

    summary_rows.append({
        "network":
            f"Null-{null_id}",

        "null_id":
            null_id,

        "memory_capacity":
            MC_NULL
    })


    # Guardar tras CADA control
    # por si Colab se desconecta

    pd.concat(
        delay_results,
        ignore_index=True
    ).to_csv(
        "null_ensemble_v03a_delays.csv",
        index=False
    )

    pd.DataFrame(
        summary_rows
    ).to_csv(
        "null_ensemble_v03a_summary.csv",
        index=False
    )


    print(
        "Resultados parciales "
        "guardados en Drive."
    )


    # --------------------------------------------------------
    # LIBERAR MEMORIA
    # --------------------------------------------------------

    del states_null
    del null_graph
    del A_null

    gc.collect()


# ============================================================
# ESTADISTICA FINAL
# ============================================================

summary = pd.DataFrame(
    summary_rows
)

null_values = summary.loc[
    summary["null_id"] > 0,
    "memory_capacity"
].to_numpy()


null_mean = float(
    np.mean(null_values)
)

null_std = float(
    np.std(
        null_values,
        ddof=1
    )
)

delta = (
    MC_REAL
    - null_mean
)

relative = (
    (
        MC_REAL
        / null_mean
    ) - 1
) * 100


if null_std > 0:

    z = (
        MC_REAL
        - null_mean
    ) / null_std

else:

    z = np.nan


wins = int(
    np.sum(
        MC_REAL
        > null_values
    )
)


# ============================================================
# RESULTADO
# ============================================================

print("\n")
print("=" * 76)
print("RESULTADO FINAL — MaleCNS v0.3A")
print("=" * 76)

print(
    "\nMaleCNS MC:",
    round(MC_REAL, 6)
)

print("\nNulls:")

for i, value in enumerate(
    null_values,
    start=1
):

    print(
        f"  Null-{i}:",
        round(float(value), 6)
    )


print("\nNull mean:",
      round(null_mean, 6))

print("Null SD:",
      round(null_std, 6))

print(
    "Δ MaleCNS - mean(null):",
    round(delta, 6)
)

print(
    "Relative advantage:",
    round(relative, 2),
    "%"
)

print(
    "Descriptive Z:",
    round(z, 3)
)

print(
    "MaleCNS superior a:",
    f"{wins}/{N_NULLS}",
    "controles"
)


# Añadir resumen agregado

aggregate = pd.DataFrame([{

    "MC_MaleCNS":
        MC_REAL,

    "null_mean":
        null_mean,

    "null_std":
        null_std,

    "delta":
        delta,

    "relative_advantage_percent":
        relative,

    "descriptive_z":
        z,

    "wins":
        wins,

    "n_nulls":
        N_NULLS

}])

aggregate.to_csv(
    "null_ensemble_v03a_aggregate.csv",
    index=False
)


print("\nArchivos guardados:")

print(
    "null_ensemble_v03a_summary.csv"
)

print(
    "null_ensemble_v03a_delays.csv"
)

print(
    "null_ensemble_v03a_aggregate.csv"
)

print("\nFIN.")

Writing /content/drive/MyDrive/malecns_ai_v0_1/benchmark_null_ensemble_v03a.py


In [ ]:
!python benchmark_null_ensemble_v03a.py

MaleCNS-AI v0.3A — NULL ENSEMBLE
Neuronas: 165,122
Conexiones: 25,563,096
Controles nulos: 5

EJECUTANDO MaleCNS REAL

Ejecutando MaleCNS...
MaleCNS: 20/160
MaleCNS: 40/160
MaleCNS: 60/160
MaleCNS: 80/160
MaleCNS: 100/160
MaleCNS: 120/160
MaleCNS: 140/160
MaleCNS: 160/160

MaleCNS MEMORY CAPACITY: 5.499105


NULL 1/5 | seed=20261914
Generando topología degree-matched...
^C


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1

import os
import pandas as pd

for f in [
    "null_ensemble_v03a_summary.csv",
    "null_ensemble_v03a_delays.csv",
    "null_ensemble_v03a_aggregate.csv"
]:
    print(f, "->", os.path.exists(f))

if os.path.exists("null_ensemble_v03a_summary.csv"):
    print()
    print(pd.read_csv("null_ensemble_v03a_summary.csv"))

/content
null_ensemble_v03a_summary.csv -> False
null_ensemble_v03a_delays.csv -> False
null_ensemble_v03a_aggregate.csv -> False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
os.chdir(PROJECT)

print("Directorio:", os.getcwd())

for f in [
    "loader.py",
    "dynamics.py",
    "adjacency_binary_real_v02.npz",
    "adjacency_degree_matched_v02.npz",
    "memory_benchmark_v02.csv"
]:
    print(f, "->", os.path.exists(f))

Mounted at /content/drive
Directorio: /content/drive/MyDrive/malecns_ai_v0_1
loader.py -> True
dynamics.py -> True
adjacency_binary_real_v02.npz -> True
adjacency_degree_matched_v02.npz -> True
memory_benchmark_v02.csv -> True


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/null_ensemble_resume.py

import os
import gc
import random
import numpy as np
import pandas as pd
import igraph as ig

from scipy import sparse
from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
OUTDIR = os.path.join(PROJECT, "results_v03a")
NULLDIR = os.path.join(OUTDIR, "null_graphs")

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(NULLDIR, exist_ok=True)

os.chdir(PROJECT)


BASE_SEED = 20260914

N_NULLS = 5
N_SAMPLES = 160
SEQUENCE_LENGTH = 70
N_INPUT = 256
N_READOUT = 512
MAX_DELAY = 30
TRAIN_FRACTION = 0.70
RIDGE_ALPHA = 1e-2

SIGN_MODE = "biological_fast"


print("=" * 72)
print("MaleCNS-AI v0.3A — RESUMABLE NULL ENSEMBLE")
print("=" * 72)


# ============================================================
# CARGAR CEREBRO
# ============================================================

base = MaleCNSGraph()

A_real = sparse.load_npz(
    "adjacency_binary_real_v02.npz"
).tocsr().astype(np.float32)

n = A_real.shape[0]

out_degree = np.diff(
    A_real.indptr
).astype(np.int32)

in_degree = np.bincount(
    A_real.indices,
    minlength=n
).astype(np.int32)


class GraphProxy:

    def __init__(self, A, nodes):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)

    @property
    def n_nodes(self):
        return self.A.shape[0]

    @property
    def n_edges(self):
        return self.A.nnz


# ============================================================
# MISMAS ENTRADAS Y DATOS QUE v0.2
# ============================================================

rng = np.random.default_rng(BASE_SEED)

input_nodes = rng.choice(
    n,
    size=N_INPUT,
    replace=False
)

remaining = np.setdiff1d(
    np.arange(n),
    input_nodes
)

readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)

sequences = rng.choice(
    [-1.0, 1.0],
    size=(N_SAMPLES, SEQUENCE_LENGTH)
).astype(np.float32)


n_train = int(
    N_SAMPLES * TRAIN_FRACTION
)

train_ids = np.arange(n_train)
test_ids = np.arange(n_train, N_SAMPLES)


# ============================================================
# FUNCIONES
# ============================================================

def generate_states(graph, name):

    print(f"\nEjecutando {name}...")

    reservoir = ConnectomeReservoir(
        graph,
        leak=0.2,
        gain=1.2,
        sign_mode=SIGN_MODE
    )

    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )

    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )

    for sample in range(N_SAMPLES):

        reservoir.reset()

        for t, value in enumerate(
            sequences[sample]
        ):

            external.fill(0)
            external[input_nodes] = value

            x = reservoir.step(external)

            states[
                sample,
                t,
                :
            ] = x[readout_nodes]

        if (sample + 1) % 20 == 0:
            print(
                f"{name}: "
                f"{sample+1}/{N_SAMPLES}"
            )

    return states


def build_dataset(states, ids):

    X = []
    Y = []

    delays = np.arange(
        1,
        MAX_DELAY + 1
    )

    for sample in ids:

        seq = sequences[sample]

        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[sample, t, :]
            )

            Y.append(
                seq[t - delays]
            )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(Y, dtype=np.float32)
    )


def evaluate(states):

    X_train, Y_train = build_dataset(
        states,
        train_ids
    )

    X_test, Y_test = build_dataset(
        states,
        test_ids
    )

    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-6

    X_train = (
        X_train - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std

    XTX = (
        X_train.T @ X_train
    ).astype(np.float64)

    XTY = (
        X_train.T @ Y_train
    ).astype(np.float64)

    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA

    W = np.linalg.solve(
        XTX,
        XTY
    )

    pred = (
        X_test.astype(np.float64)
        @ W
    )

    rows = []

    for d in range(MAX_DELAY):

        y = Y_test[:, d]
        p = pred[:, d]

        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )

        acc = float(
            np.mean(binary == y)
        )

        corr = float(
            np.corrcoef(p, y)[0, 1]
        )

        if not np.isfinite(corr):
            corr = 0.0

        rows.append({
            "delay": d + 1,
            "accuracy": acc,
            "corr": corr,
            "memory": corr**2
        })

    df = pd.DataFrame(rows)

    return df, float(df["memory"].sum())


# ============================================================
# RECUPERAR MaleCNS + Null-1 DESDE v0.2
# ============================================================

v02 = pd.read_csv(
    "memory_benchmark_v02.csv"
)

MC_REAL = float(
    v02["mc_malecns"].sum()
)

MC_NULL1 = float(
    v02["mc_control"].sum()
)

print("\nResultados reutilizados de v0.2:")
print("MaleCNS:", MC_REAL)
print("Null-1 :", MC_NULL1)


SUMMARY = os.path.join(
    OUTDIR,
    "null_ensemble_summary.csv"
)

if os.path.exists(SUMMARY):

    summary = pd.read_csv(SUMMARY)

else:

    summary = pd.DataFrame([
        {
            "network": "MaleCNS",
            "null_id": 0,
            "memory_capacity": MC_REAL
        },
        {
            "network": "Null-1",
            "null_id": 1,
            "memory_capacity": MC_NULL1
        }
    ])

    summary.to_csv(
        SUMMARY,
        index=False
    )


print("\nEstado actual:")
print(summary)


# ============================================================
# NULLS 2-5
# ============================================================

for null_id in range(2, N_NULLS + 1):

    if null_id in summary["null_id"].values:

        print(
            f"\nNull-{null_id} ya terminado."
        )

        continue


    print("\n" + "=" * 72)
    print(f"NULL {null_id}/{N_NULLS}")
    print("=" * 72)

    null_path = os.path.join(
        NULLDIR,
        f"null_{null_id:02d}.npz"
    )


    # --------------------------------------------------------
    # CARGAR O GENERAR GRAFO
    # --------------------------------------------------------

    if os.path.exists(null_path):

        print(
            "Cargando grafo nulo ya guardado..."
        )

        A_null = sparse.load_npz(
            null_path
        ).tocsr().astype(np.float32)

    else:

        seed = (
            BASE_SEED
            + null_id * 1000
        )

        print(
            "Generando topología "
            f"degree-matched | seed={seed}"
        )

        random.seed(seed)
        np.random.seed(seed)

        try:
            ig.set_random_number_generator(
                random.Random(seed)
            )
        except Exception:
            pass


        null_ig = ig.Graph.Degree_Sequence(
            out_degree.tolist(),
            in_degree.tolist(),
            method="edge_switching_simple"
        )

        A_null = (
            null_ig
            .get_adjacency_sparse()
            .astype(np.float32)
            .tocsr()
        )

        del null_ig
        gc.collect()


        # Validación
        null_out = np.diff(
            A_null.indptr
        ).astype(np.int32)

        null_in = np.bincount(
            A_null.indices,
            minlength=n
        ).astype(np.int32)


        assert A_null.nnz == A_real.nnz

        assert np.array_equal(
            null_out,
            out_degree
        )

        assert np.array_equal(
            null_in,
            in_degree
        )


        # GUARDAR INMEDIATAMENTE
        print(
            "Grafo válido. Guardando en Drive..."
        )

        sparse.save_npz(
            null_path,
            A_null,
            compressed=True
        )

        print(
            "Guardado:",
            null_path
        )


    # --------------------------------------------------------
    # SIMULAR
    # --------------------------------------------------------

    graph = GraphProxy(
        A_null,
        base.nodes
    )

    states = generate_states(
        graph,
        f"Null-{null_id}"
    )

    delay_df, MC_NULL = evaluate(states)

    print(
        f"\nNull-{null_id} MC:",
        MC_NULL
    )


    # --------------------------------------------------------
    # GUARDAR DELAYS
    # --------------------------------------------------------

    delay_file = os.path.join(
        OUTDIR,
        f"null_{null_id:02d}_delays.csv"
    )

    delay_df.to_csv(
        delay_file,
        index=False
    )


    # --------------------------------------------------------
    # GUARDAR RESUMEN
    # --------------------------------------------------------

    new_row = pd.DataFrame([{
        "network":
            f"Null-{null_id}",

        "null_id":
            null_id,

        "memory_capacity":
            MC_NULL
    }])

    summary = pd.concat(
        [summary, new_row],
        ignore_index=True
    )

    summary.to_csv(
        SUMMARY,
        index=False
    )

    print(
        "CHECKPOINT GUARDADO."
    )


    del states
    del graph
    del A_null

    gc.collect()


# ============================================================
# RESULTADO FINAL
# ============================================================

summary = pd.read_csv(SUMMARY)

null_values = summary.loc[
    summary["null_id"] > 0,
    "memory_capacity"
].to_numpy()


if len(null_values) == N_NULLS:

    null_mean = float(
        np.mean(null_values)
    )

    null_std = float(
        np.std(
            null_values,
            ddof=1
        )
    )

    delta = MC_REAL - null_mean

    relative = (
        MC_REAL / null_mean - 1
    ) * 100

    z = (
        delta / null_std
        if null_std > 0
        else np.nan
    )


    print("\n")
    print("=" * 72)
    print("RESULTADO FINAL")
    print("=" * 72)

    print(
        "MaleCNS:",
        round(MC_REAL, 6)
    )

    print("\nNulls:")

    for i, x in enumerate(
        null_values,
        start=1
    ):
        print(
            f"Null-{i}:",
            round(float(x), 6)
        )

    print(
        "\nNull mean:",
        round(null_mean, 6)
    )

    print(
        "Null SD:",
        round(null_std, 6)
    )

    print(
        "Δ:",
        round(delta, 6)
    )

    print(
        "Relative advantage:",
        round(relative, 2),
        "%"
    )

    print(
        "Descriptive Z:",
        round(z, 3)
    )

    print(
        "MaleCNS wins:",
        int(
            np.sum(
                MC_REAL > null_values
            )
        ),
        "/",
        N_NULLS
    )

else:

    print(
        "\nTodavía faltan controles:",
        N_NULLS - len(null_values)
    )

Writing /content/drive/MyDrive/malecns_ai_v0_1/null_ensemble_resume.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python null_ensemble_resume.py

/content
Traceback (most recent call last):
  File "/content/drive/MyDrive/malecns_ai_v0_1/null_ensemble_resume.py", line 7, in <module>
    import igraph as ig
ModuleNotFoundError: No module named 'igraph'


In [ ]:
!pip install -q igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 42.5 MB/s eta 0:00:00


In [ ]:
import igraph as ig
print("igraph OK:", ig.__version__)

igraph OK: 1.0.0


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python null_ensemble_resume.py

/content
MaleCNS-AI v0.3A — RESUMABLE NULL ENSEMBLE

Resultados reutilizados de v0.2:
MaleCNS: 5.499104660492246
Null-1 : 4.856009765044483

Estado actual:
   network  null_id  memory_capacity
0  MaleCNS        0         5.499105
1   Null-1        1         4.856010

NULL 2/5
Generando topología degree-matched | seed=20262914
Grafo válido. Guardando en Drive...
Guardado: /content/drive/MyDrive/malecns_ai_v0_1/results_v03a/null_graphs/null_02.npz

Ejecutando Null-2...
Null-2: 20/160
Null-2: 40/160
Null-2: 60/160
Null-2: 80/160
Null-2: 100/160
Null-2: 120/160
Null-2: 140/160
Null-2: 160/160

Null-2 MC: 4.6469699335774
CHECKPOINT GUARDADO.

NULL 3/5
Generando topología degree-matched | seed=20263914
Grafo válido. Guardando en Drive...
Guardado: /content/drive/MyDrive/malecns_ai_v0_1/results_v03a/null_graphs/null_03.npz

Ejecutando Null-3...
Null-3: 20/160
Null-3: 40/160
Null-3: 60/160
Null-3: 80/160
Null-3: 100/160
Null-3: 120/160
Null-3: 140/160
Null-3: 160/160

Null-3 MC: 4.566144768853

In [ ]:
import os
import pandas as pd

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
SUMMARY = f"{PROJECT}/results_v03a/null_ensemble_summary.csv"
NULL5 = f"{PROJECT}/results_v03a/null_graphs/null_05.npz"

print(pd.read_csv(SUMMARY))
print("\nGrafo Null-5 guardado:", os.path.exists(NULL5))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/malecns_ai_v0_1/results_v03a/null_ensemble_summary.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

print("Proyecto existe:", os.path.exists(PROJECT))

if os.path.exists(PROJECT):
    print("\nContenido:")
    for x in os.listdir(PROJECT):
        print(x)

Proyecto existe: True

Contenido:
README.md
dynamics.py
nodes.csv.gz
metrics.json
smoke_test.py
adjacency_weighted.npz
LICENSE_DATA.txt
loader.py
__pycache__
benchmark_memory_v01.py
adjacency_binary_real_v02.npz
build_degree_matched_v02.py
adjacency_degree_matched_v02.npz
benchmark_memory_v02.py
memory_benchmark_v02.csv
benchmark_null_ensemble_v03a.py
null_ensemble_resume.py
results_v03a


In [ ]:
RESULTS = "/content/drive/MyDrive/malecns_ai_v0_1/results_v03a"

print("results_v03a existe:", os.path.exists(RESULTS))

if os.path.exists(RESULTS):
    print("\nContenido results_v03a:")
    for x in os.listdir(RESULTS):
        print(x)

results_v03a existe: True

Contenido results_v03a:
null_graphs
null_02_delays.csv
null_03_delays.csv
null_04_delays.csv
null_ensemble_summary.csv


In [ ]:
import pandas as pd

SUMMARY = (
    "/content/drive/MyDrive/malecns_ai_v0_1/"
    "results_v03a/null_ensemble_summary.csv"
)

if os.path.exists(SUMMARY):
    print(pd.read_csv(SUMMARY))
else:
    print("SUMMARY NO ENCONTRADO")

   network  null_id  memory_capacity
0  MaleCNS        0         5.499105
1   Null-1        1         4.856010
2   Null-2        2         4.646970
3   Null-3        3         4.566145
4   Null-4        4         4.882394


In [ ]:
!pip install -q igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 36.1 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python null_ensemble_resume.py

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install -q igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 29.8 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python null_ensemble_resume.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python null_ensemble_resume.py

/content/drive/MyDrive/malecns_ai_v0_1
MaleCNS-AI v0.3A — RESUMABLE NULL ENSEMBLE

Resultados reutilizados de v0.2:
MaleCNS: 5.499104660492246
Null-1 : 4.856009765044483

Estado actual:
   network  null_id  memory_capacity
0  MaleCNS        0         5.499105
1   Null-1        1         4.856010
2   Null-2        2         4.646970
3   Null-3        3         4.566145
4   Null-4        4         4.882394

Null-2 ya terminado.

Null-3 ya terminado.

Null-4 ya terminado.

NULL 5/5
Generando topología degree-matched | seed=20265914
Grafo válido. Guardando en Drive...
Guardado: /content/drive/MyDrive/malecns_ai_v0_1/results_v03a/null_graphs/null_05.npz

Ejecutando Null-5...
Null-5: 20/160
Null-5: 40/160
Null-5: 60/160
Null-5: 80/160
Null-5: 100/160
Null-5: 120/160
Null-5: 140/160
Null-5: 160/160

Null-5 MC: 4.833407354205353
CHECKPOINT GUARDADO.


RESULTADO FINAL
MaleCNS: 5.499105

Nulls:
Null-1: 4.85601
Null-2: 4.64697
Null-3: 4.566145
Null-4: 4.882394
Null-5: 4.833407

Null mean: 4.75698

In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python null_ensemble_resume.py

[Errno 2] No such file or directory: '/content/drive/MyDrive/malecns_ai_v0_1'
/content
python3: can't open file '/content/null_ensemble_resume.py': [Errno 2] No such file or directory


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir

g = MaleCNSGraph()

test = ConnectomeReservoir(
    g,
    leak=0.2,
    gain=1.2,
    sign_mode="all_positive"
)

print("all_positive OK")

[Errno 2] No such file or directory: '/content/drive/MyDrive/malecns_ai_v0_1'
/content


ModuleNotFoundError: No module named 'loader'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

print("Proyecto existe:", os.path.exists(PROJECT))

os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Directorio actual:", os.getcwd())

Mounted at /content/drive
Proyecto existe: True
Directorio actual: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir

g = MaleCNSGraph()

test = ConnectomeReservoir(
    g,
    leak=0.2,
    gain=1.2,
    sign_mode="all_positive"
)

print("all_positive OK")

all_positive OK


In [ ]:
!pip install -q igraph

from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Colab listo:", os.getcwd())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 51.0 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
sign_mode="all_positive"

In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_all_positive_v03b.py

import os
import gc
import numpy as np
import pandas as pd
from scipy import sparse

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# CONFIGURACION
# ============================================================

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
OUTDIR = os.path.join(PROJECT, "results_v03b_all_positive")

os.makedirs(OUTDIR, exist_ok=True)
os.chdir(PROJECT)

BASE_SEED = 20260914

N_SAMPLES = 160
SEQUENCE_LENGTH = 70
N_INPUT = 256
N_READOUT = 512
MAX_DELAY = 30

TRAIN_FRACTION = 0.70
RIDGE_ALPHA = 1e-2

SIGN_MODE = "all_positive"


print("=" * 76)
print("MaleCNS-AI v0.3B — ALL POSITIVE")
print("=" * 76)


# ============================================================
# DATOS
# ============================================================

base = MaleCNSGraph()

A_real = sparse.load_npz(
    "adjacency_binary_real_v02.npz"
).tocsr().astype(np.float32)

n = A_real.shape[0]


class GraphProxy:

    def __init__(self, A, nodes):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)

    @property
    def n_nodes(self):
        return self.A.shape[0]

    @property
    def n_edges(self):
        return self.A.nnz


# ============================================================
# MISMA INTERFAZ Y SECUENCIAS QUE v0.2/v0.3A
# ============================================================

rng = np.random.default_rng(BASE_SEED)

input_nodes = rng.choice(
    n,
    size=N_INPUT,
    replace=False
)

remaining = np.setdiff1d(
    np.arange(n),
    input_nodes
)

readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)

sequences = rng.choice(
    [-1.0, 1.0],
    size=(N_SAMPLES, SEQUENCE_LENGTH)
).astype(np.float32)


n_train = int(
    N_SAMPLES * TRAIN_FRACTION
)

train_ids = np.arange(n_train)
test_ids = np.arange(n_train, N_SAMPLES)


# ============================================================
# FUNCIONES
# ============================================================

def generate_states(graph, name):

    print(f"\nEjecutando {name}...")

    reservoir = ConnectomeReservoir(
        graph,
        leak=0.2,
        gain=1.2,
        sign_mode=SIGN_MODE
    )

    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )

    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )

    for sample in range(N_SAMPLES):

        reservoir.reset()

        for t, value in enumerate(sequences[sample]):

            external.fill(0.0)
            external[input_nodes] = value

            x = reservoir.step(external)

            states[
                sample,
                t,
                :
            ] = x[readout_nodes]

        if (sample + 1) % 20 == 0:

            print(
                f"{name}: "
                f"{sample+1}/{N_SAMPLES}"
            )

    return states


def build_dataset(states, ids):

    X = []
    Y = []

    delays = np.arange(
        1,
        MAX_DELAY + 1
    )

    for sample in ids:

        seq = sequences[sample]

        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[
                    sample,
                    t,
                    :
                ]
            )

            Y.append(
                seq[t - delays]
            )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(Y, dtype=np.float32)
    )


def evaluate(states):

    X_train, Y_train = build_dataset(
        states,
        train_ids
    )

    X_test, Y_test = build_dataset(
        states,
        test_ids
    )


    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-6

    X_train = (
        X_train - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std


    XTX = (
        X_train.T @ X_train
    ).astype(np.float64)

    XTY = (
        X_train.T @ Y_train
    ).astype(np.float64)


    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA


    W = np.linalg.solve(
        XTX,
        XTY
    )


    pred = (
        X_test.astype(np.float64)
        @ W
    )


    rows = []

    for d in range(MAX_DELAY):

        y = Y_test[:, d]
        p = pred[:, d]

        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )

        acc = float(
            np.mean(binary == y)
        )

        corr = float(
            np.corrcoef(p, y)[0, 1]
        )

        if not np.isfinite(corr):
            corr = 0.0


        rows.append({

            "delay": d + 1,

            "accuracy": acc,

            "corr": corr,

            "memory": corr ** 2
        })


    df = pd.DataFrame(rows)

    MC = float(
        df["memory"].sum()
    )

    return df, MC


# ============================================================
# ARCHIVOS
# ============================================================

graphs = {

    0:
        "adjacency_binary_real_v02.npz",

    1:
        "adjacency_degree_matched_v02.npz",

    2:
        "results_v03a/null_graphs/null_02.npz",

    3:
        "results_v03a/null_graphs/null_03.npz",

    4:
        "results_v03a/null_graphs/null_04.npz",

    5:
        "results_v03a/null_graphs/null_05.npz",
}


SUMMARY = os.path.join(
    OUTDIR,
    "all_positive_summary.csv"
)


# ============================================================
# REANUDACION
# ============================================================

if os.path.exists(SUMMARY):

    summary = pd.read_csv(SUMMARY)

else:

    summary = pd.DataFrame(
        columns=[
            "network",
            "null_id",
            "memory_capacity"
        ]
    )


print("\nEstado inicial:")
print(summary)


# ============================================================
# REAL + 5 NULL
# ============================================================

for graph_id in range(6):

    if graph_id in summary["null_id"].values:

        name = (
            "MaleCNS"
            if graph_id == 0
            else f"Null-{graph_id}"
        )

        print(
            f"\n{name} ya terminado."
        )

        continue


    name = (
        "MaleCNS"
        if graph_id == 0
        else f"Null-{graph_id}"
    )


    print("\n")
    print("=" * 76)
    print(name)
    print("=" * 76)


    path = graphs[graph_id]

    print(
        "Cargando:",
        path
    )


    A = sparse.load_npz(
        path
    ).tocsr().astype(np.float32)


    print(
        "Neuronas:",
        f"{A.shape[0]:,}"
    )

    print(
        "Conexiones:",
        f"{A.nnz:,}"
    )


    graph = GraphProxy(
        A,
        base.nodes
    )


    states = generate_states(
        graph,
        name
    )


    delay_df, MC = evaluate(
        states
    )


    print(
        f"\n{name} MC:",
        MC
    )


    # --------------------------------------------------------
    # GUARDAR DELAYS
    # --------------------------------------------------------

    delay_path = os.path.join(
        OUTDIR,
        f"{name.lower().replace('-', '_')}_delays.csv"
    )

    delay_df.to_csv(
        delay_path,
        index=False
    )


    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    row = pd.DataFrame([{

        "network":
            name,

        "null_id":
            graph_id,

        "memory_capacity":
            MC
    }])


    summary = pd.concat(
        [summary, row],
        ignore_index=True
    )


    summary.to_csv(
        SUMMARY,
        index=False
    )


    print(
        "CHECKPOINT GUARDADO."
    )


    del states
    del graph
    del A

    gc.collect()


# ============================================================
# RESULTADO FINAL
# ============================================================

summary = pd.read_csv(SUMMARY)

print("\n")
print("=" * 76)
print("RESULTADO v0.3B — ALL POSITIVE")
print("=" * 76)

print(summary)


if len(summary) == 6:

    MC_REAL = float(
        summary.loc[
            summary["null_id"] == 0,
            "memory_capacity"
        ].iloc[0]
    )


    null_values = summary.loc[
        summary["null_id"] > 0,
        "memory_capacity"
    ].to_numpy()


    null_mean = float(
        np.mean(null_values)
    )

    null_std = float(
        np.std(
            null_values,
            ddof=1
        )
    )

    delta = (
        MC_REAL
        - null_mean
    )

    relative = (
        MC_REAL / null_mean - 1
    ) * 100


    z = (
        delta / null_std
        if null_std > 0
        else np.nan
    )


    wins = int(
        np.sum(
            MC_REAL > null_values
        )
    )


    print("\nMaleCNS:",
          round(MC_REAL, 6))

    print("Null mean:",
          round(null_mean, 6))

    print("Null SD:",
          round(null_std, 6))

    print("Δ:",
          round(delta, 6))

    print(
        "Relative advantage:",
        round(relative, 2),
        "%"
    )

    print(
        "Descriptive Z:",
        round(z, 3)
    )

    print(
        "MaleCNS wins:",
        wins,
        "/ 5"
    )


    aggregate = pd.DataFrame([{

        "MC_MaleCNS":
            MC_REAL,

        "null_mean":
            null_mean,

        "null_std":
            null_std,

        "delta":
            delta,

        "relative_advantage_percent":
            relative,

        "descriptive_z":
            z,

        "wins":
            wins
    }])


    aggregate.to_csv(
        os.path.join(
            OUTDIR,
            "all_positive_aggregate.csv"
        ),
        index=False
    )


print("\nFIN.")

Writing /content/drive/MyDrive/malecns_ai_v0_1/benchmark_all_positive_v03b.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python benchmark_all_positive_v03b.py

/content
MaleCNS-AI v0.3B — ALL POSITIVE

Estado inicial:
Empty DataFrame
Columns: [network, null_id, memory_capacity]
Index: []


MaleCNS
Cargando: adjacency_binary_real_v02.npz
Neuronas: 165,122
Conexiones: 25,563,096

Ejecutando MaleCNS...
MaleCNS: 20/160
MaleCNS: 40/160
MaleCNS: 60/160
MaleCNS: 80/160
MaleCNS: 100/160
MaleCNS: 120/160
MaleCNS: 140/160
MaleCNS: 160/160

MaleCNS MC: 3.031290816847411
/content/drive/MyDrive/malecns_ai_v0_1/benchmark_all_positive_v03b.py:473: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat(
CHECKPOINT GUARDADO.


Null-1
Cargando: adjacency_degree_matched_v02.npz
Neuronas: 165,122
Conexiones: 25,563,096

Ejecutando Null-1...
Null-1: 20/160
Null-1: 40/160
Null-1: 60/160
Null-1: 80/160
Null-1

In [ ]:
!pip install -q tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_sign_shuffle_v04.py

import os
import gc
import time
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from tqdm.auto import tqdm

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# CONFIGURACION
# ============================================================

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

OUTDIR = os.path.join(
    PROJECT,
    "results_v04_sign_shuffle"
)

SIGNDIR = os.path.join(
    OUTDIR,
    "sign_vectors"
)

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(SIGNDIR, exist_ok=True)

os.chdir(PROJECT)


BASE_SEED = 20260914

# Cinco permutaciones independientes
N_SHUFFLES = 5

N_SAMPLES = 160
SEQUENCE_LENGTH = 70

N_INPUT = 256
N_READOUT = 512

MAX_DELAY = 30

TRAIN_FRACTION = 0.70
RIDGE_ALPHA = 1e-2

LEAK = 0.2
GAIN = 1.2


SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v04_summary.csv"
)

PAIR_FILE = os.path.join(
    OUTDIR,
    "v04_pairs.csv"
)

AGGREGATE_FILE = os.path.join(
    OUTDIR,
    "v04_aggregate.csv"
)

LOG_FILE = os.path.join(
    OUTDIR,
    "v04_run.log"
)


# ============================================================
# LOG
# ============================================================

def log(message):

    stamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    line = f"[{stamp}] {message}"

    tqdm.write(line)

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(line + "\n")


log("=" * 78)
log("MaleCNS-AI v0.4 — PAIRED SIGN-SHUFFLE")
log("=" * 78)

log(
    f"Configuración: "
    f"{N_SHUFFLES} shuffles | "
    f"{N_SAMPLES} secuencias | "
    f"{N_READOUT} readout | "
    f"delay máximo {MAX_DELAY}"
)


# ============================================================
# CARGAR DATOS
# ============================================================

log("Cargando MaleCNSGraph...")

base = MaleCNSGraph()

REAL_PATH = os.path.join(
    PROJECT,
    "adjacency_binary_real_v02.npz"
)

if not os.path.exists(REAL_PATH):

    raise FileNotFoundError(
        REAL_PATH
    )


log("Cargando topología MaleCNS...")

A_real = sparse.load_npz(
    REAL_PATH
).tocsr().astype(np.float32)

n = A_real.shape[0]

log(
    f"Neuronas: {n:,}"
)

log(
    f"Conexiones MaleCNS: {A_real.nnz:,}"
)


# ============================================================
# GRAPH PROXY
# ============================================================

class GraphProxy:

    def __init__(
        self,
        A,
        nodes
    ):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)


    @property
    def n_nodes(self):

        return self.A.shape[0]


    @property
    def n_edges(self):

        return self.A.nnz


real_graph = GraphProxy(
    A_real,
    base.nodes
)


# ============================================================
# OBTENER SIGNOS BIOLOGICOS EXACTAMENTE DESDE dynamics.py
# ============================================================

log(
    "Extrayendo vector biological_fast "
    "directamente desde dynamics.py..."
)

reference_reservoir = ConnectomeReservoir(
    real_graph,
    leak=LEAK,
    gain=GAIN,
    sign_mode="biological_fast"
)

biological_sign = (
    reference_reservoir
    .sign
    .copy()
    .astype(np.float32)
)

del reference_reservoir
gc.collect()


values, counts = np.unique(
    biological_sign,
    return_counts=True
)

sign_counts = dict(
    zip(
        values.tolist(),
        counts.tolist()
    )
)

log(
    f"Signos originales: {sign_counts}"
)

log(
    "Cada shuffle preservará estos conteos EXACTAMENTE."
)


# ============================================================
# RESERVOIR CON VECTOR DE SIGNOS PERSONALIZADO
# ============================================================

class SignedReservoir:

    def __init__(
        self,
        graph,
        sign_vector,
        leak=0.2,
        gain=1.2
    ):

        self.g = graph

        self.leak = float(leak)
        self.gain = float(gain)

        self.state = np.zeros(
            graph.n_nodes,
            dtype=np.float32
        )

        self.sign = np.asarray(
            sign_vector,
            dtype=np.float32
        )

        if len(self.sign) != graph.n_nodes:

            raise ValueError(
                "El vector de signos no coincide "
                "con el número de neuronas."
            )


        self.denom = np.maximum(
            graph.in_strength,
            1.0
        ).astype(np.float32)


    def reset(self):

        self.state.fill(0.0)


    def step(
        self,
        external=None
    ):

        signed = (
            self.state
            * self.sign
        )

        recurrent = (
            self.g.A.T.dot(signed)
            .astype(
                np.float32,
                copy=False
            )
            / self.denom
        )

        z = (
            self.gain
            * recurrent
        )

        if external is not None:

            z = (
                z
                + external
            )


        proposal = np.tanh(
            z
        ).astype(
            np.float32,
            copy=False
        )


        self.state += (
            self.leak
            * (
                proposal
                - self.state
            )
        )

        return self.state


# ============================================================
# MISMAS ENTRADAS / READOUT / SECUENCIAS QUE v0.2-v0.3
# ============================================================

log(
    "Reconstruyendo exactamente "
    "la interfaz experimental previa..."
)

rng = np.random.default_rng(
    BASE_SEED
)

input_nodes = rng.choice(
    n,
    size=N_INPUT,
    replace=False
)

remaining = np.setdiff1d(
    np.arange(n),
    input_nodes
)

readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)

sequences = rng.choice(
    [-1.0, 1.0],
    size=(
        N_SAMPLES,
        SEQUENCE_LENGTH
    )
).astype(np.float32)


n_train = int(
    N_SAMPLES
    * TRAIN_FRACTION
)

train_ids = np.arange(
    n_train
)

test_ids = np.arange(
    n_train,
    N_SAMPLES
)


log(
    f"Train sequences: {len(train_ids)}"
)

log(
    f"Test sequences: {len(test_ids)}"
)


# ============================================================
# GENERAR / CARGAR SHUFFLES
# ============================================================

def get_shuffled_sign(
    shuffle_id
):

    path = os.path.join(
        SIGNDIR,
        f"sign_shuffle_{shuffle_id:02d}.npy"
    )


    if os.path.exists(path):

        sign = np.load(
            path
        )

        log(
            f"Shuffle-{shuffle_id}: "
            "vector ya guardado; reutilizando."
        )

    else:

        seed = (
            BASE_SEED
            + shuffle_id * 10000
        )

        local_rng = (
            np.random.default_rng(seed)
        )

        sign = local_rng.permutation(
            biological_sign
        ).astype(np.float32)

        np.save(
            path,
            sign
        )

        log(
            f"Shuffle-{shuffle_id}: "
            f"creado con seed={seed}"
        )


    # Validar conteos
    v, c = np.unique(
        sign,
        return_counts=True
    )

    shuffled_counts = dict(
        zip(
            v.tolist(),
            c.tolist()
        )
    )

    if shuffled_counts != sign_counts:

        raise RuntimeError(
            f"Shuffle-{shuffle_id}: "
            "los conteos de signos cambiaron."
        )


    return sign


# ============================================================
# SIMULACION CON BARRA DE PROGRESO
# ============================================================

def generate_states(
    graph,
    sign_vector,
    task_name
):

    reservoir = SignedReservoir(
        graph,
        sign_vector,
        leak=LEAK,
        gain=GAIN
    )


    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )


    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )


    start = time.time()


    log(
        f"{task_name}: "
        f"iniciando {N_SAMPLES} secuencias."
    )


    progress = tqdm(
        total=N_SAMPLES,
        desc=task_name,
        unit="seq",
        dynamic_ncols=True,
        leave=False
    )


    for sample in range(
        N_SAMPLES
    ):

        reservoir.reset()


        for t, value in enumerate(
            sequences[sample]
        ):

            external.fill(0.0)

            external[
                input_nodes
            ] = value


            x = reservoir.step(
                external
            )


            states[
                sample,
                t,
                :
            ] = x[
                readout_nodes
            ]


        progress.update(1)


        # Log cada 20 secuencias
        if (
            sample + 1
        ) % 20 == 0:

            elapsed = (
                time.time()
                - start
            )

            pct = (
                (sample + 1)
                / N_SAMPLES
                * 100
            )

            rate = (
                elapsed
                / (sample + 1)
            )

            remaining_samples = (
                N_SAMPLES
                - (sample + 1)
            )

            eta_seconds = (
                rate
                * remaining_samples
            )

            log(
                f"{task_name}: "
                f"{sample+1}/{N_SAMPLES} "
                f"({pct:.1f}%) | "
                f"ETA ~ {eta_seconds/60:.1f} min"
            )


    progress.close()


    elapsed = (
        time.time()
        - start
    )


    log(
        f"{task_name}: "
        f"simulación terminada "
        f"en {elapsed/60:.1f} min."
    )


    return states


# ============================================================
# DATASET
# ============================================================

def build_dataset(
    states,
    ids
):

    X = []
    Y = []

    delays = np.arange(
        1,
        MAX_DELAY + 1
    )


    for sample in ids:

        seq = sequences[
            sample
        ]


        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[
                    sample,
                    t,
                    :
                ]
            )

            Y.append(
                seq[
                    t - delays
                ]
            )


    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            Y,
            dtype=np.float32
        )
    )


# ============================================================
# EVALUACION
# ============================================================

def evaluate(
    states,
    task_name
):

    log(
        f"{task_name}: "
        "construyendo dataset..."
    )


    X_train, Y_train = (
        build_dataset(
            states,
            train_ids
        )
    )

    X_test, Y_test = (
        build_dataset(
            states,
            test_ids
        )
    )


    log(
        f"{task_name}: "
        f"train={X_train.shape}, "
        f"test={X_test.shape}"
    )


    mean = X_train.mean(
        axis=0
    )

    std = (
        X_train.std(
            axis=0
        )
        + 1e-6
    )


    X_train = (
        X_train
        - mean
    ) / std

    X_test = (
        X_test
        - mean
    ) / std


    log(
        f"{task_name}: "
        "entrenando ridge readout..."
    )


    XTX = (
        X_train.T
        @ X_train
    ).astype(np.float64)

    XTY = (
        X_train.T
        @ Y_train
    ).astype(np.float64)


    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA


    W = np.linalg.solve(
        XTX,
        XTY
    )


    log(
        f"{task_name}: "
        "evaluando test..."
    )


    pred = (
        X_test.astype(np.float64)
        @ W
    )


    rows = []


    for d in range(
        MAX_DELAY
    ):

        y = Y_test[:, d]
        p = pred[:, d]


        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )


        acc = float(
            np.mean(
                binary == y
            )
        )


        corr = float(
            np.corrcoef(
                p,
                y
            )[0, 1]
        )


        if not np.isfinite(
            corr
        ):

            corr = 0.0


        rows.append({

            "delay":
                d + 1,

            "accuracy":
                acc,

            "corr":
                corr,

            "memory":
                corr ** 2
        })


    df = pd.DataFrame(
        rows
    )


    MC = float(
        df[
            "memory"
        ].sum()
    )


    log(
        f"{task_name}: "
        f"MEMORY CAPACITY = {MC:.6f}"
    )


    return df, MC


# ============================================================
# BASELINES BIOLOGICOS v0.3A
# ============================================================

BASELINE_FILE = os.path.join(
    PROJECT,
    "results_v03a",
    "null_ensemble_summary.csv"
)


if not os.path.exists(
    BASELINE_FILE
):

    raise FileNotFoundError(
        "No encuentro el resumen de v0.3A:\n"
        + BASELINE_FILE
    )


baseline_df = pd.read_csv(
    BASELINE_FILE
)


baseline = {

    int(row.null_id):
        float(row.memory_capacity)

    for row in baseline_df.itertuples()
}


log(
    f"Baseline biological MaleCNS: "
    f"{baseline[0]:.6f}"
)

log(
    "Baselines biological Null: "
    + str(
        {
            i: round(
                baseline[i],
                6
            )
            for i in range(
                1,
                6
            )
        }
    )
)


# ============================================================
# GRAFOS NULL
# ============================================================

NULL_PATHS = {

    1:
        os.path.join(
            PROJECT,
            "adjacency_degree_matched_v02.npz"
        ),

    2:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_02.npz"
        ),

    3:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_03.npz"
        ),

    4:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_04.npz"
        ),

    5:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_05.npz"
        )
}


for null_id, path in (
    NULL_PATHS.items()
):

    if not os.path.exists(
        path
    ):

        raise FileNotFoundError(
            path
        )


# ============================================================
# CARGAR RESUMEN / REANUDAR
# ============================================================

if os.path.exists(
    SUMMARY_FILE
):

    summary = pd.read_csv(
        SUMMARY_FILE
    )

else:

    summary = pd.DataFrame(
        columns=[
            "task_key",
            "graph",
            "null_id",
            "shuffle_id",
            "seed",
            "memory_capacity",
            "biological_baseline",
            "delta_shuffle_minus_bio",
            "percent_change"
        ]
    )


completed = set(
    summary[
        "task_key"
    ].astype(str)
    .tolist()
)


# ============================================================
# DEFINIR 10 TAREAS
#
# Shuffle-1 -> MaleCNS + Null-1
# Shuffle-2 -> MaleCNS + Null-2
# ...
# ============================================================

tasks = []


for shuffle_id in range(
    1,
    N_SHUFFLES + 1
):

    tasks.append({
        "task_key":
            f"real_s{shuffle_id}",

        "graph":
            "MaleCNS",

        "null_id":
            0,

        "shuffle_id":
            shuffle_id
    })


    tasks.append({
        "task_key":
            f"null{shuffle_id}_s{shuffle_id}",

        "graph":
            f"Null-{shuffle_id}",

        "null_id":
            shuffle_id,

        "shuffle_id":
            shuffle_id
    })


TOTAL_TASKS = len(
    tasks
)


valid_completed = sum(
    t[
        "task_key"
    ] in completed

    for t in tasks
)


log(
    f"Tareas totales: {TOTAL_TASKS}"
)

log(
    f"Tareas ya terminadas: "
    f"{valid_completed}/{TOTAL_TASKS}"
)


# ============================================================
# BARRA GLOBAL
# ============================================================

global_bar = tqdm(
    total=TOTAL_TASKS,
    initial=valid_completed,
    desc="PROGRESO GLOBAL v0.4",
    unit="run",
    dynamic_ncols=True
)


# ============================================================
# EJECUTAR
# ============================================================

for task_number, task in enumerate(
    tasks,
    start=1
):

    task_key = task[
        "task_key"
    ]

    graph_name = task[
        "graph"
    ]

    null_id = int(
        task[
            "null_id"
        ]
    )

    shuffle_id = int(
        task[
            "shuffle_id"
        ]
    )


    if task_key in completed:

        log(
            f"{task_key}: "
            "ya terminado; saltando."
        )

        continue


    log("")
    log("=" * 78)

    log(
        f"TAREA {task_number}/{TOTAL_TASKS}: "
        f"{task_key}"
    )

    log(
        f"Grafo={graph_name} | "
        f"Shuffle={shuffle_id}"
    )

    log("=" * 78)


    sign_vector = (
        get_shuffled_sign(
            shuffle_id
        )
    )


    seed = (
        BASE_SEED
        + shuffle_id * 10000
    )


    # --------------------------------------------------------
    # GRAFO
    # --------------------------------------------------------

    if null_id == 0:

        graph = real_graph

        A_current = None

        log(
            "Usando topología MaleCNS real."
        )

    else:

        path = NULL_PATHS[
            null_id
        ]

        log(
            f"Cargando {graph_name}: "
            f"{path}"
        )

        A_current = (
            sparse.load_npz(
                path
            )
            .tocsr()
            .astype(np.float32)
        )

        graph = GraphProxy(
            A_current,
            base.nodes
        )


        log(
            f"{graph_name}: "
            f"{graph.n_edges:,} conexiones."
        )


    # --------------------------------------------------------
    # SIMULACION
    # --------------------------------------------------------

    states = generate_states(
        graph,
        sign_vector,
        task_key
    )


    # --------------------------------------------------------
    # EVALUAR
    # --------------------------------------------------------

    delay_df, MC = evaluate(
        states,
        task_key
    )


    # --------------------------------------------------------
    # COMPARAR CONTRA BASELINE BIOLOGICO
    # --------------------------------------------------------

    bio_MC = baseline[
        null_id
    ]


    delta = (
        MC
        - bio_MC
    )


    pct_change = (
        delta
        / bio_MC
        * 100
    )


    log(
        f"{task_key}: "
        f"biological={bio_MC:.6f}"
    )

    log(
        f"{task_key}: "
        f"shuffled={MC:.6f}"
    )

    log(
        f"{task_key}: "
        f"Δ shuffled-biological="
        f"{delta:+.6f} "
        f"({pct_change:+.2f}%)"
    )


    # --------------------------------------------------------
    # GUARDAR DELAYS
    # --------------------------------------------------------

    delay_file = os.path.join(
        OUTDIR,
        f"{task_key}_delays.csv"
    )


    delay_df.to_csv(
        delay_file,
        index=False
    )


    # --------------------------------------------------------
    # CHECKPOINT DEL RESUMEN
    # --------------------------------------------------------

    row = pd.DataFrame([{

        "task_key":
            task_key,

        "graph":
            graph_name,

        "null_id":
            null_id,

        "shuffle_id":
            shuffle_id,

        "seed":
            seed,

        "memory_capacity":
            MC,

        "biological_baseline":
            bio_MC,

        "delta_shuffle_minus_bio":
            delta,

        "percent_change":
            pct_change
    }])


    summary = pd.concat(
        [
            summary,
            row
        ],
        ignore_index=True
    )


    # Escritura casi-atómica
    temp_summary = (
        SUMMARY_FILE
        + ".tmp"
    )


    summary.to_csv(
        temp_summary,
        index=False
    )

    os.replace(
        temp_summary,
        SUMMARY_FILE
    )


    completed.add(
        task_key
    )


    log(
        f"{task_key}: "
        "CHECKPOINT GUARDADO."
    )


    # --------------------------------------------------------
    # LIMPIEZA
    # --------------------------------------------------------

    del states
    del delay_df
    del graph

    if A_current is not None:

        del A_current

    gc.collect()


    global_bar.update(1)


# ============================================================
# FINAL
# ============================================================

global_bar.close()


summary = pd.read_csv(
    SUMMARY_FILE
)


log("")
log("=" * 78)
log("ANALISIS FINAL v0.4")
log("=" * 78)


if len(
    summary
) < TOTAL_TASKS:

    log(
        f"Faltan "
        f"{TOTAL_TASKS-len(summary)} "
        "ejecuciones."
    )

    log(
        "Ejecuta nuevamente el mismo script "
        "para continuar."
    )

    raise SystemExit


pair_rows = []


for shuffle_id in range(
    1,
    N_SHUFFLES + 1
):

    real_row = summary.loc[
        summary[
            "task_key"
        ]
        == f"real_s{shuffle_id}"
    ].iloc[0]


    null_row = summary.loc[
        summary[
            "task_key"
        ]
        == f"null{shuffle_id}_s{shuffle_id}"
    ].iloc[0]


    real_shuffle = float(
        real_row[
            "memory_capacity"
        ]
    )

    null_shuffle = float(
        null_row[
            "memory_capacity"
        ]
    )


    real_bio = float(
        baseline[0]
    )

    null_bio = float(
        baseline[
            shuffle_id
        ]
    )


    # Cuánto se pierde al desordenar signos
    real_shuffle_loss = (
        real_bio
        - real_shuffle
    )

    null_shuffle_loss = (
        null_bio
        - null_shuffle
    )


    # Interacción
    interaction = (
        real_shuffle_loss
        - null_shuffle_loss
    )


    pair_rows.append({

        "shuffle_id":
            shuffle_id,

        "real_biological":
            real_bio,

        "real_shuffled":
            real_shuffle,

        "null_biological":
            null_bio,

        "null_shuffled":
            null_shuffle,

        "real_loss_after_shuffle":
            real_shuffle_loss,

        "null_loss_after_shuffle":
            null_shuffle_loss,

        "interaction":
            interaction,

        "real_minus_null_shuffled":
            real_shuffle
            - null_shuffle
    })


pairs = pd.DataFrame(
    pair_rows
)


pairs.to_csv(
    PAIR_FILE,
    index=False
)


real_shuffle_mean = float(
    pairs[
        "real_shuffled"
    ].mean()
)

real_shuffle_sd = float(
    pairs[
        "real_shuffled"
    ].std(
        ddof=1
    )
)

null_shuffle_mean = float(
    pairs[
        "null_shuffled"
    ].mean()
)

null_shuffle_sd = float(
    pairs[
        "null_shuffled"
    ].std(
        ddof=1
    )
)


real_bio = float(
    baseline[0]
)

null_bio_mean = float(
    np.mean(
        [
            baseline[i]
            for i in range(
                1,
                6
            )
        ]
    )
)


real_loss_mean = float(
    pairs[
        "real_loss_after_shuffle"
    ].mean()
)

null_loss_mean = float(
    pairs[
        "null_loss_after_shuffle"
    ].mean()
)


interaction_mean = float(
    pairs[
        "interaction"
    ].mean()
)

interaction_sd = float(
    pairs[
        "interaction"
    ].std(
        ddof=1
    )
)


real_drop_percent = (
    (
        real_bio
        - real_shuffle_mean
    )
    / real_bio
    * 100
)


null_drop_percent = (
    (
        null_bio_mean
        - null_shuffle_mean
    )
    / null_bio_mean
    * 100
)


wins_shuffled = int(
    np.sum(
        pairs[
            "real_minus_null_shuffled"
        ] > 0
    )
)


aggregate = pd.DataFrame([{

    "real_biological":
        real_bio,

    "real_shuffled_mean":
        real_shuffle_mean,

    "real_shuffled_sd":
        real_shuffle_sd,

    "real_drop_percent":
        real_drop_percent,

    "null_biological_mean":
        null_bio_mean,

    "null_shuffled_mean":
        null_shuffle_mean,

    "null_shuffled_sd":
        null_shuffle_sd,

    "null_drop_percent":
        null_drop_percent,

    "real_shuffle_loss":
        real_loss_mean,

    "null_shuffle_loss":
        null_loss_mean,

    "interaction_mean":
        interaction_mean,

    "interaction_sd":
        interaction_sd,

    "shuffled_real_wins":
        wins_shuffled,

    "n_pairs":
        N_SHUFFLES
}])


aggregate.to_csv(
    AGGREGATE_FILE,
    index=False
)


log("")
log("--------------- RESULTADO ---------------")

log(
    f"MaleCNS biological: "
    f"{real_bio:.6f}"
)

log(
    f"MaleCNS shuffled: "
    f"{real_shuffle_mean:.6f} "
    f"± {real_shuffle_sd:.6f}"
)

log(
    f"Caída MaleCNS al barajar signos: "
    f"{real_drop_percent:.2f}%"
)

log("")

log(
    f"Null biological mean: "
    f"{null_bio_mean:.6f}"
)

log(
    f"Null shuffled mean: "
    f"{null_shuffle_mean:.6f} "
    f"± {null_shuffle_sd:.6f}"
)

log(
    f"Caída Null al barajar signos: "
    f"{null_drop_percent:.2f}%"
)

log("")

log(
    f"Interacción media: "
    f"{interaction_mean:+.6f} "
    f"± {interaction_sd:.6f}"
)

log(
    f"MaleCNS-shuffled superior "
    f"al Null-shuffled pareado: "
    f"{wins_shuffled}/{N_SHUFFLES}"
)

log("")

log(
    "Archivos finales:"
)

log(
    SUMMARY_FILE
)

log(
    PAIR_FILE
)

log(
    AGGREGATE_FILE
)

log(
    LOG_FILE
)

log("")
log("FIN v0.4")

Writing /content/drive/MyDrive/malecns_ai_v0_1/benchmark_sign_shuffle_v04.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_sign_shuffle_v04.py

/content
[2026-09-15 19:33:40] ==============================================================================
[2026-09-15 19:33:40] MaleCNS-AI v0.4 — PAIRED SIGN-SHUFFLE
[2026-09-15 19:33:40] ==============================================================================
[2026-09-15 19:33:40] Configuración: 5 shuffles | 160 secuencias | 512 readout | delay máximo 30
[2026-09-15 19:33:40] Cargando MaleCNSGraph...
[2026-09-15 19:33:43] Cargando topología MaleCNS...
[2026-09-15 19:33:45] Neuronas: 165,122
[2026-09-15 19:33:45] Conexiones MaleCNS: 25,563,096
[2026-09-15 19:33:45] Extrayendo vector biological_fast directamente desde dynamics.py...
[2026-09-15 19:33:45] Signos originales: {-1.0: 57261, 0.0: 4143, 1.0: 103718}
[2026-09-15 19:33:45] Cada shuffle preservará estos conteos EXACTAMENTE.
[2026-09-15 19:33:45] Reconstruyendo exactamente la interfaz experimental previa...
[2026-09-15 19:33:45] Train sequences: 112
[2026-09-15 19:33:45] Test sequences: 48
[2026-09-15 19:33:45] Baseline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_sign_shuffle_v05_30.py

import os
import gc
import time
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from tqdm.auto import tqdm

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# CONFIGURACION
# ============================================================

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

OUTDIR = os.path.join(
    PROJECT,
    "results_v05_sign_shuffle_30"
)

SIGNDIR = os.path.join(
    OUTDIR,
    "sign_vectors"
)

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(SIGNDIR, exist_ok=True)

os.chdir(PROJECT)


BASE_SEED = 20260914

N_SHUFFLES = 30

N_SAMPLES = 160
SEQUENCE_LENGTH = 70

N_INPUT = 256
N_READOUT = 512

MAX_DELAY = 30

TRAIN_FRACTION = 0.70
RIDGE_ALPHA = 1e-2

LEAK = 0.2
GAIN = 1.2


SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v05_summary.csv"
)

AGGREGATE_FILE = os.path.join(
    OUTDIR,
    "v05_aggregate.csv"
)

SORTED_FILE = os.path.join(
    OUTDIR,
    "v05_distribution_sorted.csv"
)

LOG_FILE = os.path.join(
    OUTDIR,
    "v05_run.log"
)


# ============================================================
# LOG
# ============================================================

def log(message=""):

    stamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    line = f"[{stamp}] {message}"

    tqdm.write(line)

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(line + "\n")


log("=" * 78)
log("MaleCNS-AI v0.5 — 30 SIGN SHUFFLES")
log("=" * 78)

log(
    f"Objetivo: {N_SHUFFLES} shuffles "
    "sobre la topología MaleCNS fija."
)


# ============================================================
# CARGAR MaleCNS
# ============================================================

log("Cargando MaleCNSGraph...")

base = MaleCNSGraph()


REAL_PATH = os.path.join(
    PROJECT,
    "adjacency_binary_real_v02.npz"
)


A_real = sparse.load_npz(
    REAL_PATH
).tocsr().astype(np.float32)


n = A_real.shape[0]


log(
    f"Neuronas: {n:,}"
)

log(
    f"Conexiones: {A_real.nnz:,}"
)


# ============================================================
# GRAPH PROXY
# ============================================================

class GraphProxy:

    def __init__(
        self,
        A,
        nodes
    ):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)


    @property
    def n_nodes(self):

        return self.A.shape[0]


    @property
    def n_edges(self):

        return self.A.nnz


graph = GraphProxy(
    A_real,
    base.nodes
)


# ============================================================
# VECTOR BIOLOGICO
# ============================================================

log(
    "Extrayendo biological_fast..."
)


reference = ConnectomeReservoir(
    graph,
    leak=LEAK,
    gain=GAIN,
    sign_mode="biological_fast"
)


biological_sign = (
    reference.sign
    .copy()
    .astype(np.float32)
)


values, counts = np.unique(
    biological_sign,
    return_counts=True
)


SIGN_COUNTS = dict(
    zip(
        values.tolist(),
        counts.tolist()
    )
)


log(
    f"Conteos de signos: {SIGN_COUNTS}"
)


del reference
gc.collect()


# ============================================================
# BASELINE BIOLOGICO
# ============================================================

BASELINE_FILE = os.path.join(
    PROJECT,
    "results_v03a",
    "null_ensemble_summary.csv"
)


baseline_df = pd.read_csv(
    BASELINE_FILE
)


BIOLOGICAL_MC = float(
    baseline_df.loc[
        baseline_df["null_id"] == 0,
        "memory_capacity"
    ].iloc[0]
)


log(
    f"MaleCNS biological MC = "
    f"{BIOLOGICAL_MC:.6f}"
)


# ============================================================
# MISMA INTERFAZ EXPERIMENTAL
# ============================================================

rng = np.random.default_rng(
    BASE_SEED
)


input_nodes = rng.choice(
    n,
    size=N_INPUT,
    replace=False
)


remaining = np.setdiff1d(
    np.arange(n),
    input_nodes
)


readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)


sequences = rng.choice(
    [-1.0, 1.0],
    size=(
        N_SAMPLES,
        SEQUENCE_LENGTH
    )
).astype(np.float32)


n_train = int(
    N_SAMPLES
    * TRAIN_FRACTION
)


train_ids = np.arange(
    n_train
)

test_ids = np.arange(
    n_train,
    N_SAMPLES
)


# ============================================================
# SIGN SHUFFLE
# ============================================================

def get_sign_vector(
    shuffle_id
):

    # Primero buscamos los vectores ya generados por v0.4
    old_path = os.path.join(
        PROJECT,
        "results_v04_sign_shuffle",
        "sign_vectors",
        f"sign_shuffle_{shuffle_id:02d}.npy"
    )


    new_path = os.path.join(
        SIGNDIR,
        f"sign_shuffle_{shuffle_id:02d}.npy"
    )


    if os.path.exists(old_path):

        sign = np.load(
            old_path
        ).astype(np.float32)

        log(
            f"Shuffle-{shuffle_id}: "
            "reutilizando vector v0.4."
        )


    elif os.path.exists(new_path):

        sign = np.load(
            new_path
        ).astype(np.float32)

        log(
            f"Shuffle-{shuffle_id}: "
            "reutilizando vector v0.5."
        )


    else:

        seed = (
            BASE_SEED
            + shuffle_id * 10000
        )

        local_rng = (
            np.random.default_rng(seed)
        )


        sign = local_rng.permutation(
            biological_sign
        ).astype(np.float32)


        np.save(
            new_path,
            sign
        )


        log(
            f"Shuffle-{shuffle_id}: "
            f"creado | seed={seed}"
        )


    # Validación
    v, c = np.unique(
        sign,
        return_counts=True
    )


    counts_now = dict(
        zip(
            v.tolist(),
            c.tolist()
        )
    )


    if counts_now != SIGN_COUNTS:

        raise RuntimeError(
            f"Shuffle-{shuffle_id}: "
            "conteos de signos incorrectos."
        )


    return sign


# ============================================================
# SIMULACION
# ============================================================

def generate_states(
    sign_vector,
    shuffle_id
):

    name = (
        f"Shuffle-{shuffle_id:02d}"
    )


    reservoir = ConnectomeReservoir(
        graph,
        leak=LEAK,
        gain=GAIN,
        sign_mode="biological_fast"
    )


    # Sustituimos únicamente la localización
    # de los signos.
    reservoir.sign = (
        sign_vector.copy()
    )


    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )


    external = np.zeros(
        n,
        dtype=np.float32
    )


    start = time.time()


    log(
        f"{name}: iniciando simulación."
    )


    bar = tqdm(
        total=N_SAMPLES,
        desc=name,
        unit="seq",
        dynamic_ncols=True,
        leave=False
    )


    for sample in range(
        N_SAMPLES
    ):

        reservoir.reset()


        for t, value in enumerate(
            sequences[sample]
        ):

            external.fill(
                0.0
            )

            external[
                input_nodes
            ] = value


            state = reservoir.step(
                external
            )


            states[
                sample,
                t,
                :
            ] = state[
                readout_nodes
            ]


        bar.update(1)


        if (
            sample + 1
        ) % 20 == 0:

            elapsed = (
                time.time()
                - start
            )

            completed = (
                sample + 1
            )

            pct = (
                completed
                / N_SAMPLES
                * 100
            )


            seconds_per_sample = (
                elapsed
                / completed
            )


            eta = (
                N_SAMPLES
                - completed
            ) * seconds_per_sample


            log(
                f"{name}: "
                f"{completed}/{N_SAMPLES} "
                f"({pct:.1f}%) | "
                f"ETA ~ {eta/60:.1f} min"
            )


    bar.close()


    elapsed = (
        time.time()
        - start
    )


    log(
        f"{name}: simulación terminada "
        f"en {elapsed/60:.1f} min."
    )


    return states


# ============================================================
# DATASET
# ============================================================

def build_dataset(
    states,
    ids
):

    X = []
    Y = []


    delays = np.arange(
        1,
        MAX_DELAY + 1
    )


    for sample in ids:

        seq = sequences[
            sample
        ]


        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[
                    sample,
                    t,
                    :
                ]
            )


            Y.append(
                seq[
                    t - delays
                ]
            )


    return (
        np.asarray(
            X,
            dtype=np.float32
        ),

        np.asarray(
            Y,
            dtype=np.float32
        )
    )


# ============================================================
# EVALUACION
# ============================================================

def evaluate(
    states,
    shuffle_id
):

    name = (
        f"Shuffle-{shuffle_id:02d}"
    )


    log(
        f"{name}: preparando train/test..."
    )


    X_train, Y_train = (
        build_dataset(
            states,
            train_ids
        )
    )


    X_test, Y_test = (
        build_dataset(
            states,
            test_ids
        )
    )


    mean = X_train.mean(
        axis=0
    )


    std = (
        X_train.std(
            axis=0
        )
        + 1e-6
    )


    X_train = (
        X_train
        - mean
    ) / std


    X_test = (
        X_test
        - mean
    ) / std


    log(
        f"{name}: entrenando ridge..."
    )


    XTX = (
        X_train.T
        @ X_train
    ).astype(np.float64)


    XTY = (
        X_train.T
        @ Y_train
    ).astype(np.float64)


    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA


    W = np.linalg.solve(
        XTX,
        XTY
    )


    pred = (
        X_test.astype(np.float64)
        @ W
    )


    rows = []


    for d in range(
        MAX_DELAY
    ):

        y = Y_test[:, d]
        p = pred[:, d]


        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )


        accuracy = float(
            np.mean(
                binary == y
            )
        )


        corr = float(
            np.corrcoef(
                p,
                y
            )[0, 1]
        )


        if not np.isfinite(
            corr
        ):

            corr = 0.0


        rows.append({

            "delay":
                d + 1,

            "accuracy":
                accuracy,

            "corr":
                corr,

            "memory":
                corr ** 2
        })


    df = pd.DataFrame(
        rows
    )


    MC = float(
        df[
            "memory"
        ].sum()
    )


    log(
        f"{name}: MC = {MC:.6f}"
    )


    log(
        f"{name}: Δ vs biological = "
        f"{MC-BIOLOGICAL_MC:+.6f}"
    )


    return df, MC


# ============================================================
# CARGAR RESULTADOS EXISTENTES
# ============================================================

if os.path.exists(
    SUMMARY_FILE
):

    summary = pd.read_csv(
        SUMMARY_FILE
    )

else:

    summary = pd.DataFrame(
        columns=[
            "shuffle_id",
            "seed",
            "memory_capacity",
            "delta_vs_biological",
            "percent_vs_biological",
            "source"
        ]
    )


# ============================================================
# REUTILIZAR LOS 5 SHUFFLES DE v0.4
# ============================================================

V04_FILE = os.path.join(
    PROJECT,
    "results_v04_sign_shuffle",
    "v04_summary.csv"
)


if os.path.exists(
    V04_FILE
):

    v04 = pd.read_csv(
        V04_FILE
    )


    existing_ids = set(
        summary[
            "shuffle_id"
        ].astype(int)
        .tolist()
    )


    for shuffle_id in range(
        1,
        6
    ):

        if shuffle_id in existing_ids:
            continue


        task_key = (
            f"real_s{shuffle_id}"
        )


        row = v04.loc[
            v04[
                "task_key"
            ] == task_key
        ]


        if len(row) == 1:

            MC = float(
                row[
                    "memory_capacity"
                ].iloc[0]
            )


            new_row = pd.DataFrame([{

                "shuffle_id":
                    shuffle_id,

                "seed":
                    BASE_SEED
                    + shuffle_id
                    * 10000,

                "memory_capacity":
                    MC,

                "delta_vs_biological":
                    MC
                    - BIOLOGICAL_MC,

                "percent_vs_biological":
                    (
                        MC
                        / BIOLOGICAL_MC
                        - 1
                    ) * 100,

                "source":
                    "reused_v04"
            }])


            summary = pd.concat(
                [
                    summary,
                    new_row
                ],
                ignore_index=True
            )


            log(
                f"Shuffle-{shuffle_id:02d}: "
                f"reutilizado de v0.4 | "
                f"MC={MC:.6f}"
            )


summary = (
    summary
    .drop_duplicates(
        subset=[
            "shuffle_id"
        ],
        keep="last"
    )
    .sort_values(
        "shuffle_id"
    )
)


summary.to_csv(
    SUMMARY_FILE,
    index=False
)


# ============================================================
# PROGRESO GLOBAL
# ============================================================

completed_ids = set(
    summary[
        "shuffle_id"
    ].astype(int)
    .tolist()
)


completed_count = len(
    completed_ids
)


log(
    f"Shuffles ya disponibles: "
    f"{completed_count}/{N_SHUFFLES}"
)


global_bar = tqdm(
    total=N_SHUFFLES,
    initial=completed_count,
    desc="PROGRESO GLOBAL v0.5",
    unit="shuffle",
    dynamic_ncols=True
)


# ============================================================
# EJECUTAR FALTANTES
# ============================================================

for shuffle_id in range(
    1,
    N_SHUFFLES + 1
):

    if shuffle_id in completed_ids:

        continue


    log("")
    log("=" * 78)

    log(
        f"SHUFFLE "
        f"{shuffle_id}/{N_SHUFFLES}"
    )

    log("=" * 78)


    sign_vector = (
        get_sign_vector(
            shuffle_id
        )
    )


    states = generate_states(
        sign_vector,
        shuffle_id
    )


    delay_df, MC = evaluate(
        states,
        shuffle_id
    )


    delay_file = os.path.join(
        OUTDIR,
        f"shuffle_{shuffle_id:02d}_delays.csv"
    )


    delay_df.to_csv(
        delay_file,
        index=False
    )


    seed = (
        BASE_SEED
        + shuffle_id * 10000
    )


    row = pd.DataFrame([{

        "shuffle_id":
            shuffle_id,

        "seed":
            seed,

        "memory_capacity":
            MC,

        "delta_vs_biological":
            MC
            - BIOLOGICAL_MC,

        "percent_vs_biological":
            (
                MC
                / BIOLOGICAL_MC
                - 1
            ) * 100,

        "source":
            "v05"
    }])


    summary = pd.concat(
        [
            summary,
            row
        ],
        ignore_index=True
    )


    summary = (
        summary
        .drop_duplicates(
            subset=[
                "shuffle_id"
            ],
            keep="last"
        )
        .sort_values(
            "shuffle_id"
        )
    )


    # Checkpoint casi atómico
    temp_file = (
        SUMMARY_FILE
        + ".tmp"
    )


    summary.to_csv(
        temp_file,
        index=False
    )


    os.replace(
        temp_file,
        SUMMARY_FILE
    )


    completed_ids.add(
        shuffle_id
    )


    log(
        f"Shuffle-{shuffle_id:02d}: "
        "CHECKPOINT GUARDADO."
    )


    del states
    del delay_df
    del sign_vector

    gc.collect()


    global_bar.update(1)


global_bar.close()


# ============================================================
# ANALISIS FINAL
# ============================================================

summary = pd.read_csv(
    SUMMARY_FILE
)


if len(summary) != N_SHUFFLES:

    log(
        f"Faltan "
        f"{N_SHUFFLES-len(summary)} "
        "shuffles."
    )

    raise SystemExit


values = summary[
    "memory_capacity"
].to_numpy(
    dtype=float
)


mean_MC = float(
    np.mean(values)
)


sd_MC = float(
    np.std(
        values,
        ddof=1
    )
)


median_MC = float(
    np.median(values)
)


minimum = float(
    np.min(values)
)


maximum = float(
    np.max(values)
)


q025 = float(
    np.quantile(
        values,
        0.025
    )
)


q975 = float(
    np.quantile(
        values,
        0.975
    )
)


below = int(
    np.sum(
        values
        < BIOLOGICAL_MC
    )
)


above = int(
    np.sum(
        values
        > BIOLOGICAL_MC
    )
)


equal = int(
    np.sum(
        np.isclose(
            values,
            BIOLOGICAL_MC
        )
    )
)


percentile = (
    below
    / N_SHUFFLES
    * 100
)


delta_mean = (
    BIOLOGICAL_MC
    - mean_MC
)


relative_advantage = (
    BIOLOGICAL_MC
    / mean_MC
    - 1
) * 100


z_descriptive = (
    delta_mean
    / sd_MC
    if sd_MC > 0
    else np.nan
)


# Permutation p:
# ¿cuántos shuffles son >= biological?
p_empirical_high = (
    1
    + int(
        np.sum(
            values
            >= BIOLOGICAL_MC
        )
    )
) / (
    N_SHUFFLES
    + 1
)


# ============================================================
# ORDENAR DISTRIBUCION
# ============================================================

sorted_df = (
    summary
    .sort_values(
        "memory_capacity",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


sorted_df[
    "rank_descending"
] = (
    np.arange(
        1,
        len(sorted_df) + 1
    )
)


sorted_df.to_csv(
    SORTED_FILE,
    index=False
)


# ============================================================
# GUARDAR AGREGADO
# ============================================================

aggregate = pd.DataFrame([{

    "biological_MC":
        BIOLOGICAL_MC,

    "shuffle_mean":
        mean_MC,

    "shuffle_sd":
        sd_MC,

    "shuffle_median":
        median_MC,

    "shuffle_min":
        minimum,

    "shuffle_max":
        maximum,

    "shuffle_q025":
        q025,

    "shuffle_q975":
        q975,

    "biological_minus_shuffle_mean":
        delta_mean,

    "relative_advantage_percent":
        relative_advantage,

    "descriptive_z":
        z_descriptive,

    "shuffles_below_biological":
        below,

    "shuffles_above_biological":
        above,

    "shuffles_equal_biological":
        equal,

    "biological_percentile":
        percentile,

    "empirical_one_sided_p":
        p_empirical_high,

    "n_shuffles":
        N_SHUFFLES
}])


aggregate.to_csv(
    AGGREGATE_FILE,
    index=False
)


# ============================================================
# RESULTADO FINAL
# ============================================================

log("")
log("=" * 78)
log("RESULTADO FINAL v0.5")
log("=" * 78)

log(
    f"Biological MC: "
    f"{BIOLOGICAL_MC:.6f}"
)

log(
    f"Shuffle mean: "
    f"{mean_MC:.6f} "
    f"± {sd_MC:.6f}"
)

log(
    f"Median: "
    f"{median_MC:.6f}"
)

log(
    f"Range: "
    f"{minimum:.6f} "
    f"→ {maximum:.6f}"
)

log(
    f"95% empirical interval: "
    f"{q025:.6f} "
    f"→ {q975:.6f}"
)

log("")

log(
    f"Biological - shuffle mean: "
    f"{delta_mean:+.6f}"
)

log(
    f"Relative difference: "
    f"{relative_advantage:+.2f}%"
)

log(
    f"Descriptive Z: "
    f"{z_descriptive:+.3f}"
)

log("")

log(
    f"Shuffles debajo del biological: "
    f"{below}/{N_SHUFFLES}"
)

log(
    f"Shuffles encima del biological: "
    f"{above}/{N_SHUFFLES}"
)

log(
    f"Percentil aproximado biological: "
    f"{percentile:.1f}"
)

log(
    f"Permutation p unilateral: "
    f"{p_empirical_high:.4f}"
)

log("")

log("Top 5 shuffles:")

for row in sorted_df.head(5).itertuples():

    log(
        f"  #{row.rank_descending} "
        f"Shuffle-{int(row.shuffle_id):02d}: "
        f"{row.memory_capacity:.6f}"
    )


log("")
log("Bottom 5 shuffles:")

for row in (
    sorted_df
    .tail(5)
    .sort_values(
        "memory_capacity"
    )
    .itertuples()
):

    log(
        f"  Shuffle-{int(row.shuffle_id):02d}: "
        f"{row.memory_capacity:.6f}"
    )


log("")
log("Archivos:")

log(
    SUMMARY_FILE
)

log(
    AGGREGATE_FILE
)

log(
    SORTED_FILE
)

log(
    LOG_FILE
)

log("")
log("FIN v0.5")

Writing /content/drive/MyDrive/malecns_ai_v0_1/benchmark_sign_shuffle_v05_30.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_sign_shuffle_v05_30.py

Se truncaron las últimas líneas 5000 del resultado de transmisión.
[2026-09-15 21:51:44] MaleCNS biological MC = 5.499105
/content/drive/MyDrive/malecns_ai_v0_1/benchmark_sign_shuffle_v05_30.py:925: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary = pd.concat(
[2026-09-15 21:51:44] Shuffle-01: reutilizado de v0.4 | MC=4.692290
[2026-09-15 21:51:44] Shuffle-02: reutilizado de v0.4 | MC=5.175031
[2026-09-15 21:51:44] Shuffle-03: reutilizado de v0.4 | MC=5.098700
[2026-09-15 21:51:44] Shuffle-04: reutilizado de v0.4 | MC=5.426152
[2026-09-15 21:51:44] Shuffle-05: reutilizado de v0.4 | MC=5.541638
[2026-09-15 21:51:44] Shuffles ya disponibles: 5/30
[2026-09-15 21:51:44] 
[2026-09-15 21:51:44] ===============================================

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/analyze_structure_v06.py

import os
import gc
import time
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# CONFIGURACION
# ============================================================

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

OUTDIR = os.path.join(
    PROJECT,
    "results_v06_structure"
)

os.makedirs(
    OUTDIR,
    exist_ok=True
)

os.chdir(PROJECT)


BASE_SEED = 20260914

LEAK = 0.2
GAIN = 1.2

N_INPUT = 256
N_READOUT = 512


FEATURE_FILE = os.path.join(
    OUTDIR,
    "v06_structural_features.csv"
)

CORR_FILE = os.path.join(
    OUTDIR,
    "v06_correlations.csv"
)

EXTREME_FILE = os.path.join(
    OUTDIR,
    "v06_extremes.csv"
)

LOG_FILE = os.path.join(
    OUTDIR,
    "v06_run.log"
)


# ============================================================
# LOG
# ============================================================

def log(message=""):

    stamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    line = f"[{stamp}] {message}"

    tqdm.write(line)

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            line + "\n"
        )


log("=" * 80)
log("MaleCNS-AI v0.6 — STRUCTURAL MECHANISM ANALYSIS")
log("=" * 80)


# ============================================================
# CARGAR GRAFO
# ============================================================

GRAPH_FILE = os.path.join(
    PROJECT,
    "adjacency_binary_real_v02.npz"
)


log("Cargando topología MaleCNS...")


A = sparse.load_npz(
    GRAPH_FILE
).tocsr().astype(np.float32)


n = A.shape[0]
M = A.nnz


log(
    f"Neuronas: {n:,}"
)

log(
    f"Conexiones: {M:,}"
)


# Transpuesta CSR para acelerar A^T x
log(
    "Construyendo transpuesta CSR..."
)

AT = (
    A.transpose()
    .tocsr()
)


# ============================================================
# GRADOS
# ============================================================

out_degree = np.diff(
    A.indptr
).astype(np.float32)


in_degree = np.diff(
    AT.indptr
).astype(np.float32)


total_degree = (
    in_degree
    + out_degree
)


denom = np.maximum(
    in_degree,
    1.0
).astype(np.float32)


log(
    "Grados calculados."
)


# ============================================================
# CARGAR NODOS / SIGNOS BIOLOGICOS
# ============================================================

base = MaleCNSGraph()


class GraphProxy:

    def __init__(
        self,
        A,
        nodes
    ):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)


    @property
    def n_nodes(self):

        return self.A.shape[0]


graph_proxy = GraphProxy(
    A,
    base.nodes
)


reference = ConnectomeReservoir(
    graph_proxy,
    leak=LEAK,
    gain=GAIN,
    sign_mode="biological_fast"
)


biological_sign = (
    reference.sign
    .copy()
    .astype(np.float32)
)


del reference
gc.collect()


values, counts = np.unique(
    biological_sign,
    return_counts=True
)


BIO_COUNTS = dict(
    zip(
        values.tolist(),
        counts.tolist()
    )
)


log(
    f"Conteos de signos: {BIO_COUNTS}"
)


# ============================================================
# MC BIOLOGICO + 30 SHUFFLES
# ============================================================

BASELINE_FILE = os.path.join(
    PROJECT,
    "results_v03a",
    "null_ensemble_summary.csv"
)


base_df = pd.read_csv(
    BASELINE_FILE
)


BIO_MC = float(
    base_df.loc[
        base_df["null_id"] == 0,
        "memory_capacity"
    ].iloc[0]
)


V05_FILE = os.path.join(
    PROJECT,
    "results_v05_sign_shuffle_30",
    "v05_summary.csv"
)


v05 = pd.read_csv(
    V05_FILE
)


if len(v05) != 30:

    raise RuntimeError(
        f"v05_summary tiene {len(v05)} filas; "
        "se esperaban 30."
    )


mc_by_shuffle = {

    int(row.shuffle_id):
        float(row.memory_capacity)

    for row in v05.itertuples()
}


log(
    f"MC biológico: {BIO_MC:.6f}"
)

log(
    "30 MC de shuffles cargados."
)


# ============================================================
# RECONSTRUIR INPUT / READOUT EXACTOS
# ============================================================

rng = np.random.default_rng(
    BASE_SEED
)


input_nodes = rng.choice(
    n,
    size=N_INPUT,
    replace=False
)


remaining = np.setdiff1d(
    np.arange(n),
    input_nodes
)


readout_nodes = rng.choice(
    remaining,
    size=N_READOUT,
    replace=False
)


log(
    f"Inputs reconstruidos: {len(input_nodes)}"
)

log(
    f"Readouts reconstruidos: {len(readout_nodes)}"
)


# ============================================================
# HUBS
# ============================================================

def top_mask(
    values,
    fraction
):

    k = max(
        1,
        int(
            round(
                len(values)
                * fraction
            )
        )
    )


    indices = np.argpartition(
        values,
        -k
    )[-k:]


    mask = np.zeros(
        len(values),
        dtype=bool
    )


    mask[
        indices
    ] = True


    return mask


hub_masks = {

    "hub_01pct":
        top_mask(
            total_degree,
            0.001
        ),

    "hub_1pct":
        top_mask(
            total_degree,
            0.01
        ),

    "hub_5pct":
        top_mask(
            total_degree,
            0.05
        )
}


for name, mask in hub_masks.items():

    log(
        f"{name}: "
        f"{int(mask.sum()):,} neuronas"
    )


# ============================================================
# PROBES FIJOS DEL OPERADOR RECURRENTE
# ============================================================

probe_rng = np.random.default_rng(
    6062026
)


probe_1 = probe_rng.standard_normal(
    n
).astype(np.float32)


probe_2 = probe_rng.standard_normal(
    n
).astype(np.float32)


probe_1 /= np.linalg.norm(
    probe_1
)

probe_2 /= np.linalg.norm(
    probe_2
)


# ============================================================
# UBICACION DE SIGN VECTORS
# ============================================================

def sign_path(
    shuffle_id
):

    old = os.path.join(
        PROJECT,
        "results_v04_sign_shuffle",
        "sign_vectors",
        f"sign_shuffle_{shuffle_id:02d}.npy"
    )


    new = os.path.join(
        PROJECT,
        "results_v05_sign_shuffle_30",
        "sign_vectors",
        f"sign_shuffle_{shuffle_id:02d}.npy"
    )


    if os.path.exists(old):

        return old


    if os.path.exists(new):

        return new


    raise FileNotFoundError(
        f"No encuentro vector de signos "
        f"Shuffle-{shuffle_id:02d}"
    )


# ============================================================
# CONFIGURACIONES
# ============================================================

configs = [{

    "config_id":
        "biological",

    "config_type":
        "biological",

    "shuffle_id":
        0,

    "memory_capacity":
        BIO_MC,

    "path":
        None
}]


for shuffle_id in range(
    1,
    31
):

    configs.append({

        "config_id":
            f"shuffle_{shuffle_id:02d}",

        "config_type":
            "shuffle",

        "shuffle_id":
            shuffle_id,

        "memory_capacity":
            mc_by_shuffle[
                shuffle_id
            ],

        "path":
            sign_path(
                shuffle_id
            )
    })


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def frac(
    condition
):

    return float(
        np.mean(
            condition
        )
    )


def safe_mean(
    values
):

    if len(values) == 0:

        return np.nan

    return float(
        np.mean(values)
    )


def safe_std(
    values
):

    if len(values) == 0:

        return np.nan

    return float(
        np.std(values)
    )


# ============================================================
# ANALIZAR UNA CONFIGURACION
# ============================================================

def analyze_configuration(
    config,
    sign
):

    name = config[
        "config_id"
    ]


    inner = tqdm(
        total=6,
        desc=name,
        unit="stage",
        dynamic_ncols=True,
        leave=False
    )


    result = {

        "config_id":
            name,

        "config_type":
            config[
                "config_type"
            ],

        "shuffle_id":
            config[
                "shuffle_id"
            ],

        "memory_capacity":
            config[
                "memory_capacity"
            ]
    }


    # ========================================================
    # STAGE 1 — VALIDAR + INPUT / READOUT
    # ========================================================

    vals, cnt = np.unique(
        sign,
        return_counts=True
    )


    current_counts = dict(
        zip(
            vals.tolist(),
            cnt.tolist()
        )
    )


    if current_counts != BIO_COUNTS:

        raise RuntimeError(
            f"{name}: conteos de signos incorrectos."
        )


    global_pos = frac(
        sign == 1
    )

    global_neg = frac(
        sign == -1
    )

    global_zero = frac(
        sign == 0
    )


    result.update({

        "global_pos_frac":
            global_pos,

        "global_neg_frac":
            global_neg,

        "global_zero_frac":
            global_zero,

        "input_pos_frac":
            frac(
                sign[
                    input_nodes
                ] == 1
            ),

        "input_neg_frac":
            frac(
                sign[
                    input_nodes
                ] == -1
            ),

        "input_zero_frac":
            frac(
                sign[
                    input_nodes
                ] == 0
            ),

        "readout_pos_frac":
            frac(
                sign[
                    readout_nodes
                ] == 1
            ),

        "readout_neg_frac":
            frac(
                sign[
                    readout_nodes
                ] == -1
            ),

        "readout_zero_frac":
            frac(
                sign[
                    readout_nodes
                ] == 0
            )
    })


    inner.update(1)


    # ========================================================
    # STAGE 2 — SIGNOS VS GRADO / HUBS
    # ========================================================

    pos = (
        sign == 1
    )

    neg = (
        sign == -1
    )

    zero = (
        sign == 0
    )

    active = (
        sign != 0
    )


    for label, mask in [

        ("pos", pos),
        ("neg", neg),
        ("zero", zero)

    ]:

        result[
            f"{label}_mean_in_degree"
        ] = safe_mean(
            in_degree[
                mask
            ]
        )

        result[
            f"{label}_mean_out_degree"
        ] = safe_mean(
            out_degree[
                mask
            ]
        )

        result[
            f"{label}_mean_total_degree"
        ] = safe_mean(
            total_degree[
                mask
            ]
        )


    # Fracción de TODAS las conexiones cuyo source tiene cierto signo.
    pos_out_edges = float(
        out_degree[
            pos
        ].sum(
            dtype=np.float64
        )
    )

    neg_out_edges = float(
        out_degree[
            neg
        ].sum(
            dtype=np.float64
        )
    )

    zero_out_edges = float(
        out_degree[
            zero
        ].sum(
            dtype=np.float64
        )
    )


    result.update({

        "positive_source_edge_frac":
            pos_out_edges / M,

        "negative_source_edge_frac":
            neg_out_edges / M,

        "zero_source_edge_frac":
            zero_out_edges / M,

        "active_source_edge_frac":
            (
                pos_out_edges
                + neg_out_edges
            ) / M,

        "neg_to_pos_source_edge_ratio":
            (
                neg_out_edges
                / pos_out_edges
                if pos_out_edges > 0
                else np.nan
            )
    })


    # Hubs
    for hub_name, hub_mask in (
        hub_masks.items()
    ):

        hp = frac(
            sign[
                hub_mask
            ] == 1
        )

        hn = frac(
            sign[
                hub_mask
            ] == -1
        )

        hz = frac(
            sign[
                hub_mask
            ] == 0
        )


        result[
            f"{hub_name}_pos_frac"
        ] = hp

        result[
            f"{hub_name}_neg_frac"
        ] = hn

        result[
            f"{hub_name}_zero_frac"
        ] = hz


        result[
            f"{hub_name}_pos_enrichment"
        ] = (
            hp / global_pos
        )

        result[
            f"{hub_name}_neg_enrichment"
        ] = (
            hn / global_neg
        )

        result[
            f"{hub_name}_zero_enrichment"
        ] = (
            hz / global_zero
        )


    inner.update(1)


    # ========================================================
    # STAGE 3 — EDGE SIGN MOTIFS
    # ========================================================

    sign_defs = [

        ("neg", -1),
        ("zero", 0),
        ("pos", 1)
    ]


    masks = {

        label:
            (
                sign == value
            )

        for label, value
        in sign_defs
    }


    edge_fraction = {}


    for target_label, target_value in (
        sign_defs
    ):

        target_indicator = (
            sign == target_value
        ).astype(np.float32)


        edges_to_target = A.dot(
            target_indicator
        )


        for source_label, source_value in (
            sign_defs
        ):

            count = float(
                edges_to_target[
                    masks[
                        source_label
                    ]
                ].sum(
                    dtype=np.float64
                )
            )


            key = (
                f"edge_"
                f"{source_label}_"
                f"{target_label}_frac"
            )


            value = (
                count / M
            )


            result[
                key
            ] = value


            edge_fraction[
                (
                    source_label,
                    target_label
                )
            ] = value


    result[
        "edge_active_same_sign_frac"
    ] = (
        edge_fraction[
            ("pos", "pos")
        ]
        +
        edge_fraction[
            ("neg", "neg")
        ]
    )


    result[
        "edge_active_opposite_sign_frac"
    ] = (
        edge_fraction[
            ("pos", "neg")
        ]
        +
        edge_fraction[
            ("neg", "pos")
        ]
    )


    inner.update(1)


    # ========================================================
    # STAGE 4 — BALANCE DE ENTRADA POR NEURONA
    # ========================================================

    incoming_signed = (
        AT.dot(
            sign
        )
        .astype(
            np.float32,
            copy=False
        )
    )


    valid = (
        in_degree > 0
    )


    balance = np.zeros(
        n,
        dtype=np.float32
    )


    balance[
        valid
    ] = (
        incoming_signed[
            valid
        ]
        /
        in_degree[
            valid
        ]
    )


    abs_balance = np.abs(
        balance[
            valid
        ]
    )


    result.update({

        "incoming_balance_mean":
            float(
                np.mean(
                    balance[
                        valid
                    ]
                )
            ),

        "incoming_balance_std":
            float(
                np.std(
                    balance[
                        valid
                    ]
                )
            ),

        "incoming_abs_balance_mean":
            float(
                np.mean(
                    abs_balance
                )
            ),

        "incoming_abs_balance_median":
            float(
                np.median(
                    abs_balance
                )
            ),

        "incoming_abs_balance_q90":
            float(
                np.quantile(
                    abs_balance,
                    0.90
                )
            ),

        "incoming_abs_balance_q99":
            float(
                np.quantile(
                    abs_balance,
                    0.99
                )
            ),

        "incoming_abs_gt_050_frac":
            float(
                np.mean(
                    abs_balance
                    > 0.50
                )
            ),

        "incoming_abs_lt_010_frac":
            float(
                np.mean(
                    abs_balance
                    < 0.10
                )
            ),

        "input_incoming_abs_balance_mean":
            float(
                np.mean(
                    np.abs(
                        balance[
                            input_nodes
                        ]
                    )
                )
            ),

        "readout_incoming_abs_balance_mean":
            float(
                np.mean(
                    np.abs(
                        balance[
                            readout_nodes
                        ]
                    )
                )
            )
    })


    inner.update(1)


    # ========================================================
    # STAGE 5 — LINEAR RECURRENT AMPLIFICATION PROXY
    #
    # y = gain * D^-1 * A^T * S * x
    #
    # Misma parte lineal de nuestra dinámica antes de tanh.
    # ========================================================

    amplifications = []


    for probe in [

        probe_1,
        probe_2

    ]:

        y = (
            GAIN
            *
            AT.dot(
                probe
                * sign
            )
            /
            denom
        )


        amp = (
            np.linalg.norm(
                y
            )
            /
            np.linalg.norm(
                probe
            )
        )


        amplifications.append(
            float(amp)
        )


    result[
        "linear_amp_probe1"
    ] = amplifications[0]

    result[
        "linear_amp_probe2"
    ] = amplifications[1]

    result[
        "linear_amp_mean"
    ] = float(
        np.mean(
            amplifications
        )
    )


    inner.update(1)


    # ========================================================
    # STAGE 6 — FINAL
    # ========================================================

    inner.update(1)

    inner.close()


    return result


# ============================================================
# REANUDAR
# ============================================================

if os.path.exists(
    FEATURE_FILE
):

    features = pd.read_csv(
        FEATURE_FILE
    )

else:

    features = pd.DataFrame()


if (
    not features.empty
    and
    "config_id"
    in features.columns
):

    completed = set(
        features[
            "config_id"
        ].astype(str)
    )

else:

    completed = set()


log(
    f"Configuraciones ya analizadas: "
    f"{len(completed)}/31"
)


# ============================================================
# BARRA GLOBAL
# ============================================================

global_bar = tqdm(
    total=len(configs),
    initial=len(completed),
    desc="PROGRESO GLOBAL v0.6",
    unit="config",
    dynamic_ncols=True
)


# ============================================================
# EJECUTAR
# ============================================================

for number, config in enumerate(
    configs,
    start=1
):

    config_id = config[
        "config_id"
    ]


    if config_id in completed:

        log(
            f"{config_id}: "
            "ya analizado; saltando."
        )

        continue


    log("")
    log("=" * 80)

    log(
        f"CONFIG {number}/31: "
        f"{config_id}"
    )

    log(
        f"MC = "
        f"{config['memory_capacity']:.6f}"
    )

    log("=" * 80)


    start = time.time()


    if (
        config[
            "config_type"
        ]
        == "biological"
    ):

        sign = (
            biological_sign
            .copy()
        )

    else:

        sign = np.load(
            config[
                "path"
            ]
        ).astype(np.float32)


    result = analyze_configuration(
        config,
        sign
    )


    row = pd.DataFrame(
        [result]
    )


    if features.empty:

        features = row

    else:

        features = pd.concat(
            [
                features,
                row
            ],
            ignore_index=True
        )


    # Evitar duplicados
    features = (
        features
        .drop_duplicates(
            subset=[
                "config_id"
            ],
            keep="last"
        )
    )


    # CHECKPOINT CASI ATOMICO
    tmp = (
        FEATURE_FILE
        + ".tmp"
    )


    features.to_csv(
        tmp,
        index=False
    )


    os.replace(
        tmp,
        FEATURE_FILE
    )


    completed.add(
        config_id
    )


    elapsed = (
        time.time()
        - start
    )


    log(
        f"{config_id}: "
        f"terminado en "
        f"{elapsed/60:.2f} min"
    )

    log(
        f"{config_id}: "
        "CHECKPOINT GUARDADO."
    )


    del sign
    gc.collect()


    global_bar.update(1)


global_bar.close()


# ============================================================
# VALIDAR FINAL
# ============================================================

features = pd.read_csv(
    FEATURE_FILE
)


if len(features) != 31:

    log(
        f"Hay {len(features)}/31 configuraciones."
    )

    log(
        "Ejecuta nuevamente para continuar."
    )

    raise SystemExit


# ============================================================
# CORRELACIONES
# SOLO 30 SHUFFLES
# ============================================================

shuffled = features.loc[
    features[
        "config_type"
    ] == "shuffle"
].copy()


excluded = {

    "shuffle_id",
    "memory_capacity"
}


numeric_columns = shuffled.select_dtypes(
    include=[
        np.number
    ]
).columns


feature_columns = [

    column

    for column in numeric_columns

    if column
    not in excluded
]


correlation_rows = []


log("")
log("=" * 80)
log("CALCULANDO CORRELACIONES MC ↔ ESTRUCTURA")
log("=" * 80)


for column in tqdm(
    feature_columns,
    desc="Correlaciones",
    unit="feature"
):

    x = shuffled[
        column
    ].to_numpy(
        dtype=float
    )

    y = shuffled[
        "memory_capacity"
    ].to_numpy(
        dtype=float
    )


    valid = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )


    x = x[
        valid
    ]

    y = y[
        valid
    ]


    if (
        len(x) < 5
        or
        np.std(x) == 0
    ):

        continue


    pear_r, pear_p = pearsonr(
        x,
        y
    )


    spear_r, spear_p = spearmanr(
        x,
        y
    )


    correlation_rows.append({

        "feature":
            column,

        "pearson_r":
            float(
                pear_r
            ),

        "pearson_p":
            float(
                pear_p
            ),

        "spearman_r":
            float(
                spear_r
            ),

        "spearman_p":
            float(
                spear_p
            ),

        "n":
            len(x)
    })


corr = pd.DataFrame(
    correlation_rows
)


# ============================================================
# BENJAMINI-HOCHBERG FDR
# ============================================================

def bh_fdr(
    pvalues
):

    pvalues = np.asarray(
        pvalues,
        dtype=float
    )

    m = len(
        pvalues
    )


    order = np.argsort(
        pvalues
    )


    ranked = (
        pvalues[
            order
        ]
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )


    ranked = np.minimum.accumulate(
        ranked[
            ::-1
        ]
    )[
        ::-1
    ]


    ranked = np.clip(
        ranked,
        0,
        1
    )


    q = np.empty(
        m,
        dtype=float
    )


    q[
        order
    ] = ranked


    return q


if len(corr) > 0:

    corr[
        "spearman_q_fdr"
    ] = bh_fdr(
        corr[
            "spearman_p"
        ].to_numpy()
    )


    corr[
        "abs_spearman"
    ] = np.abs(
        corr[
            "spearman_r"
        ]
    )


    corr = corr.sort_values(
        "abs_spearman",
        ascending=False
    )


corr.to_csv(
    CORR_FILE,
    index=False
)


# ============================================================
# EXTREMOS
# ============================================================

extreme_ids = [

    "biological",
    "shuffle_07",
    "shuffle_22",
    "shuffle_14",
    "shuffle_10"
]


extremes = features.loc[
    features[
        "config_id"
    ].isin(
        extreme_ids
    )
].copy()


extremes.to_csv(
    EXTREME_FILE,
    index=False
)


# ============================================================
# RESULTADOS PRINCIPALES
# ============================================================

log("")
log("=" * 80)
log("RESULTADO FINAL v0.6")
log("=" * 80)


bio = features.loc[
    features[
        "config_id"
    ] == "biological"
].iloc[0]


log(
    f"Biological MC: "
    f"{bio['memory_capacity']:.6f}"
)

log(
    f"Biological active-source edges: "
    f"{bio['active_source_edge_frac']*100:.2f}%"
)

log(
    f"Biological ZERO-source edges: "
    f"{bio['zero_source_edge_frac']*100:.2f}%"
)

log(
    f"Biological top-0.1% hubs ZERO: "
    f"{bio['hub_01pct_zero_frac']*100:.2f}%"
)

log(
    f"Biological incoming |balance| mean: "
    f"{bio['incoming_abs_balance_mean']:.4f}"
)

log(
    f"Biological linear amplification: "
    f"{bio['linear_amp_mean']:.4f}"
)


log("")
log("EXTREMOS:")


for config_id in extreme_ids:

    row = features.loc[
        features[
            "config_id"
        ] == config_id
    ].iloc[0]


    log(
        f"{config_id:12s} | "
        f"MC={row['memory_capacity']:.6f} | "
        f"active={row['active_source_edge_frac']*100:.2f}% | "
        f"zeroEdges={row['zero_source_edge_frac']*100:.2f}% | "
        f"hub0={row['hub_01pct_zero_frac']*100:.2f}% | "
        f"|balance|={row['incoming_abs_balance_mean']:.4f} | "
        f"amp={row['linear_amp_mean']:.4f}"
    )


log("")
log("TOP CORRELACIONES CON MEMORY CAPACITY:")


if len(corr) > 0:

    for row in corr.head(
        15
    ).itertuples():

        log(
            f"{row.feature:40s} "
            f"Spearman={row.spearman_r:+.3f} "
            f"p={row.spearman_p:.4g} "
            f"FDR-q={row.spearman_q_fdr:.4g}"
        )


# ============================================================
# CORRELACIONES ESPECIFICAS IMPORTANTES
# ============================================================

important = [

    "zero_source_edge_frac",
    "active_source_edge_frac",

    "hub_01pct_zero_frac",
    "hub_1pct_zero_frac",

    "hub_01pct_neg_frac",
    "hub_1pct_neg_frac",

    "incoming_abs_balance_mean",
    "incoming_balance_std",

    "edge_active_same_sign_frac",
    "edge_active_opposite_sign_frac",

    "input_zero_frac",
    "readout_zero_frac",

    "linear_amp_mean"
]


log("")
log("VARIABLES MECANISTICAS CLAVE:")


for variable in important:

    row = corr.loc[
        corr[
            "feature"
        ] == variable
    ]


    if len(row) == 1:

        r = row.iloc[0]

        log(
            f"{variable:40s} "
            f"rho={r['spearman_r']:+.3f} | "
            f"p={r['spearman_p']:.4g} | "
            f"q={r['spearman_q_fdr']:.4g}"
        )


log("")
log("Archivos guardados:")

log(
    FEATURE_FILE
)

log(
    CORR_FILE
)

log(
    EXTREME_FILE
)

log(
    LOG_FILE
)

log("")
log("FIN v0.6")

Writing /content/drive/MyDrive/malecns_ai_v0_1/analyze_structure_v06.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u analyze_structure_v06.py

/content
[2026-09-16 02:39:27] ================================================================================
[2026-09-16 02:39:27] MaleCNS-AI v0.6 — STRUCTURAL MECHANISM ANALYSIS
[2026-09-16 02:39:27] ================================================================================
[2026-09-16 02:39:27] Cargando topología MaleCNS...
[2026-09-16 02:39:30] Neuronas: 165,122
[2026-09-16 02:39:30] Conexiones: 25,563,096
[2026-09-16 02:39:30] Construyendo transpuesta CSR...
[2026-09-16 02:39:31] Grados calculados.
[2026-09-16 02:39:35] Conteos de signos: {-1.0: 57261, 0.0: 4143, 1.0: 103718}
[2026-09-16 02:39:35] MC biológico: 5.499105
[2026-09-16 02:39:35] 30 MC de shuffles cargados.
[2026-09-16 02:39:35] Inputs reconstruidos: 256
[2026-09-16 02:39:35] Readouts reconstruidos: 512
[2026-09-16 02:39:35] hub_01pct: 165 neuronas
[2026-09-16 02:39:35] hub_1pct: 1,651 neuronas
[2026-09-16 02:39:35] hub_5pct: 8,256 neuronas
[2026-09-16 02:39:35] Configuraciones ya analizadas: 0/31
[2026-09-16 0

In [ ]:
import numpy as np
import pandas as pd

path = "/content/drive/MyDrive/malecns_ai_v0_1/results_v06_structure/v06_correlations.csv"

df = pd.read_csv(path)

p = df["spearman_p"].to_numpy(dtype=float)
valid = np.isfinite(p)

q = np.full(len(df), np.nan)

pv = p[valid]
order = np.argsort(pv)

m = len(pv)

adjusted = (
    pv[order]
    * m
    / np.arange(1, m + 1)
)

adjusted = np.minimum.accumulate(
    adjusted[::-1]
)[::-1]

adjusted = np.clip(
    adjusted,
    0,
    1
)

q_valid = np.empty(m)
q_valid[order] = adjusted

q[valid] = q_valid

df["spearman_q_fdr_fixed"] = q

df = df.sort_values(
    "spearman_q_fdr_fixed"
)

df.to_csv(
    "/content/drive/MyDrive/malecns_ai_v0_1/results_v06_structure/v06_correlations_FDR_FIXED.csv",
    index=False
)

print(
    df[
        [
            "feature",
            "spearman_r",
            "spearman_p",
            "spearman_q_fdr_fixed"
        ]
    ].head(15).to_string(index=False)
)

                          feature  spearman_r  spearman_p  spearman_q_fdr_fixed
                 readout_neg_frac    0.446721    0.013333              0.757854
                 readout_pos_frac   -0.409971    0.024447              0.757854
  input_incoming_abs_balance_mean    0.339711    0.066260              0.915479
   edge_active_opposite_sign_frac   -0.262291    0.161441              0.915479
            neg_mean_total_degree   -0.248943    0.184648              0.915479
                   input_pos_frac    0.248467    0.185517              0.915479
              neg_mean_out_degree   -0.234705    0.211871              0.915479
                edge_neg_neg_frac   -0.234705    0.211871              0.915479
        negative_source_edge_frac   -0.234705    0.211871              0.915479
     neg_to_pos_source_edge_ratio   -0.233370    0.214557              0.915479
             incoming_balance_std    0.212903    0.258654              0.915479
        positive_source_edge_frac    0.2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_interface_robustness_v07.py

import os
import gc
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from tqdm.auto import tqdm

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# CONFIGURACION — NO CAMBIAR DESPUES DE INICIAR
# ============================================================

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

OUTDIR = os.path.join(
    PROJECT,
    "results_v07_interface_robustness"
)

INTERFACE_DIR = os.path.join(
    OUTDIR,
    "interfaces"
)

DELAY_DIR = os.path.join(
    OUTDIR,
    "delays"
)

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(INTERFACE_DIR, exist_ok=True)
os.makedirs(DELAY_DIR, exist_ok=True)

os.chdir(PROJECT)


BASE_SEED = 20260914

N_SAMPLES = 160
SEQUENCE_LENGTH = 70

N_INPUT = 256
N_READOUT = 512

MAX_DELAY = 30

TRAIN_FRACTION = 0.70
RIDGE_ALPHA = 1e-2

LEAK = 0.2
GAIN = 1.2

SIGN_MODE = "biological_fast"


# Interfaz 0 = original.
# 1–4 = nuevas interfaces independientes.
INTERFACE_SEEDS = {
    0: BASE_SEED,
    1: BASE_SEED + 101001,
    2: BASE_SEED + 202002,
    3: BASE_SEED + 303003,
    4: BASE_SEED + 404004,
}


RUN_SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v07_runs.csv"
)

INTERFACE_SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v07_interfaces.csv"
)

FINAL_FILE = os.path.join(
    OUTDIR,
    "v07_final_gate.csv"
)

LOG_FILE = os.path.join(
    OUTDIR,
    "v07_run.log"
)


# ============================================================
# CRITERIOS PRE-REGISTRADOS
# ============================================================

REQUIRED_TOTAL_INTERFACE_WINS = 4   # de 5
REQUIRED_NEW_NO_D1_WINS = 3        # de 4


# ============================================================
# LOG
# ============================================================

def log(message=""):

    stamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    line = f"[{stamp}] {message}"

    tqdm.write(line)

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(line + "\n")


log("=" * 82)
log("MaleCNS-AI v0.7 — INTERFACE ROBUSTNESS / FINAL MEMORY GATE")
log("=" * 82)

log(
    "Criterio primario: MaleCNS > media Null en >= 4/5 interfaces."
)

log(
    "Criterio global: media ΔMC entre interfaces > 0."
)

log(
    "Control delay-1: en interfaces nuevas, "
    "ΔMC delays 2–30 positivo en >=3/4 y media positiva."
)


# ============================================================
# CARGAR DATOS
# ============================================================

log("Cargando MaleCNSGraph...")

base = MaleCNSGraph()


REAL_PATH = os.path.join(
    PROJECT,
    "adjacency_binary_real_v02.npz"
)


log("Cargando topología MaleCNS...")

A_real = sparse.load_npz(
    REAL_PATH
).tocsr().astype(np.float32)


n = A_real.shape[0]


log(
    f"Neuronas: {n:,}"
)

log(
    f"Conexiones reales: {A_real.nnz:,}"
)


# ============================================================
# GRAPH PROXY
# ============================================================

class GraphProxy:

    def __init__(
        self,
        A,
        nodes
    ):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)


    @property
    def n_nodes(self):
        return self.A.shape[0]


    @property
    def n_edges(self):
        return self.A.nnz


real_graph = GraphProxy(
    A_real,
    base.nodes
)


# ============================================================
# REDES
# ============================================================

NETWORK_PATHS = {

    0:
        REAL_PATH,

    1:
        os.path.join(
            PROJECT,
            "adjacency_degree_matched_v02.npz"
        ),

    2:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_02.npz"
        ),

    3:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_03.npz"
        ),

    4:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_04.npz"
        ),

    5:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_05.npz"
        )
}


NETWORK_NAMES = {
    0: "MaleCNS",
    1: "Null-1",
    2: "Null-2",
    3: "Null-3",
    4: "Null-4",
    5: "Null-5"
}


for network_id, path in NETWORK_PATHS.items():

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"No existe {NETWORK_NAMES[network_id]}:\n{path}"
        )


# ============================================================
# RECONSTRUIR EXACTAMENTE LAS SECUENCIAS ORIGINALES
#
# IMPORTANTE:
# v0.2/v0.3 generaban:
# input -> readout -> sequences
#
# Por eso consumimos RNG en exactamente ese orden.
# ============================================================

log(
    "Reconstruyendo las secuencias exactas "
    "del benchmark original..."
)


rng_original = np.random.default_rng(
    BASE_SEED
)


original_input = rng_original.choice(
    n,
    size=N_INPUT,
    replace=False
)


remaining_original = np.setdiff1d(
    np.arange(n),
    original_input
)


original_readout = rng_original.choice(
    remaining_original,
    size=N_READOUT,
    replace=False
)


sequences = rng_original.choice(
    [-1.0, 1.0],
    size=(
        N_SAMPLES,
        SEQUENCE_LENGTH
    )
).astype(np.float32)


n_train = int(
    N_SAMPLES
    * TRAIN_FRACTION
)


train_ids = np.arange(
    n_train
)


test_ids = np.arange(
    n_train,
    N_SAMPLES
)


log(
    f"Train: {len(train_ids)} secuencias"
)

log(
    f"Test: {len(test_ids)} secuencias"
)


# ============================================================
# HASH
# ============================================================

def array_hash(arr):

    return hashlib.sha256(
        np.ascontiguousarray(arr).tobytes()
    ).hexdigest()[:16]


SEQUENCE_HASH = array_hash(
    sequences
)


log(
    f"Sequence hash: {SEQUENCE_HASH}"
)


# ============================================================
# CREAR / CARGAR INTERFACES
# ============================================================

def get_interface(
    interface_id
):

    path = os.path.join(
        INTERFACE_DIR,
        f"interface_{interface_id:02d}.npz"
    )


    if os.path.exists(path):

        data = np.load(path)

        input_nodes = (
            data["input_nodes"]
            .astype(np.int64)
        )

        readout_nodes = (
            data["readout_nodes"]
            .astype(np.int64)
        )

        log(
            f"Interface-{interface_id}: "
            "reutilizando nodos guardados."
        )


    elif interface_id == 0:

        input_nodes = (
            original_input.copy()
        )

        readout_nodes = (
            original_readout.copy()
        )


        np.savez_compressed(
            path,
            input_nodes=input_nodes,
            readout_nodes=readout_nodes
        )


        log(
            "Interface-0: reconstruida "
            "desde el benchmark original."
        )


    else:

        seed = INTERFACE_SEEDS[
            interface_id
        ]


        rng = np.random.default_rng(
            seed
        )


        input_nodes = rng.choice(
            n,
            size=N_INPUT,
            replace=False
        )


        remaining = np.setdiff1d(
            np.arange(n),
            input_nodes
        )


        readout_nodes = rng.choice(
            remaining,
            size=N_READOUT,
            replace=False
        )


        np.savez_compressed(
            path,
            input_nodes=input_nodes,
            readout_nodes=readout_nodes
        )


        log(
            f"Interface-{interface_id}: "
            f"creada con seed={seed}"
        )


    # Validaciones
    if len(np.unique(input_nodes)) != N_INPUT:

        raise RuntimeError(
            "Input nodes duplicados."
        )


    if len(np.unique(readout_nodes)) != N_READOUT:

        raise RuntimeError(
            "Readout nodes duplicados."
        )


    if len(
        np.intersect1d(
            input_nodes,
            readout_nodes
        )
    ) != 0:

        raise RuntimeError(
            "Input y readout se superponen."
        )


    log(
        f"Interface-{interface_id}: "
        f"input hash={array_hash(input_nodes)} | "
        f"readout hash={array_hash(readout_nodes)}"
    )


    return (
        input_nodes,
        readout_nodes
    )


interfaces = {}


for interface_id in range(5):

    interfaces[
        interface_id
    ] = get_interface(
        interface_id
    )


# ============================================================
# DATASET
# ============================================================

def build_dataset(
    states,
    ids
):

    X = []
    Y = []


    delays = np.arange(
        1,
        MAX_DELAY + 1
    )


    for sample in ids:

        seq = sequences[
            sample
        ]


        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[
                    sample,
                    t,
                    :
                ]
            )


            Y.append(
                seq[
                    t - delays
                ]
            )


    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            Y,
            dtype=np.float32
        )
    )


# ============================================================
# SIMULACION
# ============================================================

def generate_states(
    graph,
    input_nodes,
    readout_nodes,
    task_name
):

    reservoir = ConnectomeReservoir(
        graph,
        leak=LEAK,
        gain=GAIN,
        sign_mode=SIGN_MODE
    )


    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )


    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )


    start = time.time()


    log(
        f"{task_name}: "
        f"iniciando {N_SAMPLES} secuencias."
    )


    bar = tqdm(
        total=N_SAMPLES,
        desc=task_name,
        unit="seq",
        dynamic_ncols=True,
        leave=False
    )


    for sample in range(
        N_SAMPLES
    ):

        reservoir.reset()


        for t, value in enumerate(
            sequences[sample]
        ):

            external.fill(
                0.0
            )


            external[
                input_nodes
            ] = value


            state = reservoir.step(
                external
            )


            states[
                sample,
                t,
                :
            ] = state[
                readout_nodes
            ]


        bar.update(1)


        if (
            sample + 1
        ) % 20 == 0:

            elapsed = (
                time.time()
                - start
            )


            done = (
                sample + 1
            )


            pct = (
                done
                / N_SAMPLES
                * 100
            )


            sec_per_seq = (
                elapsed
                / done
            )


            eta = (
                N_SAMPLES
                - done
            ) * sec_per_seq


            log(
                f"{task_name}: "
                f"{done}/{N_SAMPLES} "
                f"({pct:.1f}%) | "
                f"ETA ~ {eta/60:.1f} min"
            )


    bar.close()


    elapsed = (
        time.time()
        - start
    )


    log(
        f"{task_name}: simulación completa "
        f"en {elapsed/60:.1f} min."
    )


    return states


# ============================================================
# EVALUACION
# ============================================================

def evaluate(
    states,
    task_name
):

    log(
        f"{task_name}: "
        "construyendo train/test..."
    )


    X_train, Y_train = build_dataset(
        states,
        train_ids
    )


    X_test, Y_test = build_dataset(
        states,
        test_ids
    )


    log(
        f"{task_name}: "
        f"train={X_train.shape}, "
        f"test={X_test.shape}"
    )


    mean = X_train.mean(
        axis=0
    )


    std = (
        X_train.std(
            axis=0
        )
        + 1e-6
    )


    X_train = (
        X_train
        - mean
    ) / std


    X_test = (
        X_test
        - mean
    ) / std


    log(
        f"{task_name}: entrenando ridge..."
    )


    XTX = (
        X_train.T
        @ X_train
    ).astype(np.float64)


    XTY = (
        X_train.T
        @ Y_train
    ).astype(np.float64)


    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA


    W = np.linalg.solve(
        XTX,
        XTY
    )


    pred = (
        X_test.astype(np.float64)
        @ W
    )


    rows = []


    for d in range(
        MAX_DELAY
    ):

        y = Y_test[:, d]
        p = pred[:, d]


        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )


        acc = float(
            np.mean(
                binary == y
            )
        )


        corr = float(
            np.corrcoef(
                p,
                y
            )[0, 1]
        )


        if not np.isfinite(corr):

            corr = 0.0


        rows.append({

            "delay":
                d + 1,

            "accuracy":
                acc,

            "corr":
                corr,

            "memory":
                corr ** 2
        })


    df = pd.DataFrame(
        rows
    )


    mc_total = float(
        df[
            "memory"
        ].sum()
    )


    mc_no_d1 = float(
        df.loc[
            df[
                "delay"
            ] >= 2,
            "memory"
        ].sum()
    )


    mc_long = float(
        df.loc[
            df[
                "delay"
            ] >= 11,
            "memory"
        ].sum()
    )


    log(
        f"{task_name}: "
        f"MC total={mc_total:.6f} | "
        f"MC d2-30={mc_no_d1:.6f} | "
        f"MC d11-30={mc_long:.6f}"
    )


    return (
        df,
        mc_total,
        mc_no_d1,
        mc_long
    )


# ============================================================
# RESULTADOS EXISTENTES
# ============================================================

if os.path.exists(
    RUN_SUMMARY_FILE
):

    runs = pd.read_csv(
        RUN_SUMMARY_FILE
    )

else:

    runs = pd.DataFrame(
        columns=[
            "task_key",
            "interface_id",
            "interface_seed",
            "network_id",
            "network",
            "memory_capacity",
            "mc_delay_2_30",
            "mc_delay_11_30",
            "source",
            "sequence_hash"
        ]
    )


# ============================================================
# REUTILIZAR INTERFAZ 0 DESDE v0.3A
# ============================================================

V03_FILE = os.path.join(
    PROJECT,
    "results_v03a",
    "null_ensemble_summary.csv"
)


v03 = pd.read_csv(
    V03_FILE
)


existing_keys = set(
    runs[
        "task_key"
    ].astype(str)
    .tolist()
)


for network_id in range(
    6
):

    task_key = (
        f"i00_n{network_id}"
    )


    if task_key in existing_keys:

        continue


    name = NETWORK_NAMES[
        network_id
    ]


    row = v03.loc[
        v03[
            "null_id"
        ] == network_id
    ]


    if len(row) != 1:

        raise RuntimeError(
            f"No pude recuperar {name} "
            "desde v0.3A."
        )


    MC = float(
        row[
            "memory_capacity"
        ].iloc[0]
    )


    new_row = pd.DataFrame([{

        "task_key":
            task_key,

        "interface_id":
            0,

        "interface_seed":
            INTERFACE_SEEDS[0],

        "network_id":
            network_id,

        "network":
            name,

        "memory_capacity":
            MC,

        # No usamos estos valores para el gate
        # anti-delay1 porque fueron generados
        # en experimentos anteriores.
        "mc_delay_2_30":
            np.nan,

        "mc_delay_11_30":
            np.nan,

        "source":
            "reused_v03a",

        "sequence_hash":
            SEQUENCE_HASH
    }])


    if runs.empty:

        runs = new_row

    else:

        runs = pd.concat(
            [
                runs,
                new_row
            ],
            ignore_index=True
        )


    log(
        f"Interface-0 / {name}: "
        f"reutilizado MC={MC:.6f}"
    )


runs = (
    runs
    .drop_duplicates(
        subset=[
            "task_key"
        ],
        keep="last"
    )
)


runs.to_csv(
    RUN_SUMMARY_FILE,
    index=False
)


# ============================================================
# DEFINIR LAS 24 EJECUCIONES NUEVAS
# ============================================================

tasks = []


for interface_id in range(
    1,
    5
):

    for network_id in range(
        6
    ):

        tasks.append({

            "task_key":
                f"i{interface_id:02d}_n{network_id}",

            "interface_id":
                interface_id,

            "network_id":
                network_id
        })


completed_keys = set(
    runs[
        "task_key"
    ].astype(str)
    .tolist()
)


completed_new = sum(
    task[
        "task_key"
    ] in completed_keys

    for task in tasks
)


log("")
log(
    f"Ejecuciones nuevas necesarias: "
    f"{len(tasks)}"
)

log(
    f"Ya terminadas: "
    f"{completed_new}/{len(tasks)}"
)


# ============================================================
# PROGRESO GLOBAL
# ============================================================

global_bar = tqdm(
    total=len(tasks),
    initial=completed_new,
    desc="PROGRESO GLOBAL v0.7",
    unit="run",
    dynamic_ncols=True
)


# ============================================================
# EJECUTAR
# ============================================================

for index, task in enumerate(
    tasks,
    start=1
):

    task_key = task[
        "task_key"
    ]


    if task_key in completed_keys:

        log(
            f"{task_key}: "
            "ya completado; saltando."
        )

        continue


    interface_id = int(
        task[
            "interface_id"
        ]
    )


    network_id = int(
        task[
            "network_id"
        ]
    )


    network_name = NETWORK_NAMES[
        network_id
    ]


    input_nodes, readout_nodes = (
        interfaces[
            interface_id
        ]
    )


    log("")
    log("=" * 82)

    log(
        f"TAREA {index}/{len(tasks)} | "
        f"Interface-{interface_id} | "
        f"{network_name}"
    )

    log("=" * 82)


    # --------------------------------------------------------
    # CARGAR GRAFO
    # --------------------------------------------------------

    if network_id == 0:

        graph = real_graph
        A_current = None


    else:

        path = NETWORK_PATHS[
            network_id
        ]


        log(
            f"Cargando {network_name}..."
        )


        A_current = sparse.load_npz(
            path
        ).tocsr().astype(np.float32)


        graph = GraphProxy(
            A_current,
            base.nodes
        )


    log(
        f"{network_name}: "
        f"{graph.n_edges:,} conexiones"
    )


    # --------------------------------------------------------
    # SIMULAR
    # --------------------------------------------------------

    states = generate_states(
        graph,
        input_nodes,
        readout_nodes,
        task_key
    )


    # --------------------------------------------------------
    # EVALUAR
    # --------------------------------------------------------

    (
        delay_df,
        mc_total,
        mc_no_d1,
        mc_long
    ) = evaluate(
        states,
        task_key
    )


    # --------------------------------------------------------
    # GUARDAR DELAYS
    # --------------------------------------------------------

    delay_file = os.path.join(
        DELAY_DIR,
        f"{task_key}_delays.csv"
    )


    delay_df.to_csv(
        delay_file,
        index=False
    )


    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    row = pd.DataFrame([{

        "task_key":
            task_key,

        "interface_id":
            interface_id,

        "interface_seed":
            INTERFACE_SEEDS[
                interface_id
            ],

        "network_id":
            network_id,

        "network":
            network_name,

        "memory_capacity":
            mc_total,

        "mc_delay_2_30":
            mc_no_d1,

        "mc_delay_11_30":
            mc_long,

        "source":
            "v07",

        "sequence_hash":
            SEQUENCE_HASH
    }])


    if runs.empty:

        runs = row

    else:

        runs = pd.concat(
            [
                runs,
                row
            ],
            ignore_index=True
        )


    runs = (
        runs
        .drop_duplicates(
            subset=[
                "task_key"
            ],
            keep="last"
        )
    )


    temp_file = (
        RUN_SUMMARY_FILE
        + ".tmp"
    )


    runs.to_csv(
        temp_file,
        index=False
    )


    os.replace(
        temp_file,
        RUN_SUMMARY_FILE
    )


    completed_keys.add(
        task_key
    )


    log(
        f"{task_key}: "
        "CHECKPOINT GUARDADO."
    )


    # --------------------------------------------------------
    # LIMPIEZA
    # --------------------------------------------------------

    del states
    del delay_df
    del graph


    if A_current is not None:

        del A_current


    gc.collect()


    global_bar.update(1)


global_bar.close()


# ============================================================
# VALIDAR FINAL
# ============================================================

runs = pd.read_csv(
    RUN_SUMMARY_FILE
)


expected_total = (
    5 * 6
)


if len(runs) != expected_total:

    log(
        f"Hay {len(runs)}/{expected_total} "
        "resultados."
    )

    log(
        "Ejecuta nuevamente el script."
    )

    raise SystemExit


# ============================================================
# RESUMEN POR INTERFAZ
# ============================================================

interface_rows = []


for interface_id in range(
    5
):

    subset = runs.loc[
        runs[
            "interface_id"
        ] == interface_id
    ]


    real_row = subset.loc[
        subset[
            "network_id"
        ] == 0
    ].iloc[0]


    null_rows = subset.loc[
        subset[
            "network_id"
        ] > 0
    ]


    real_mc = float(
        real_row[
            "memory_capacity"
        ]
    )


    null_values = null_rows[
        "memory_capacity"
    ].to_numpy(
        dtype=float
    )


    null_mean = float(
        np.mean(
            null_values
        )
    )


    null_sd = float(
        np.std(
            null_values,
            ddof=1
        )
    )


    delta = (
        real_mc
        - null_mean
    )


    relative = (
        real_mc
        / null_mean
        - 1
    ) * 100


    wins = int(
        np.sum(
            real_mc
            > null_values
        )
    )


    # Delay 2-30:
    # sólo es criterio formal para las cuatro nuevas.
    if interface_id > 0:

        real_no_d1 = float(
            real_row[
                "mc_delay_2_30"
            ]
        )


        null_no_d1 = null_rows[
            "mc_delay_2_30"
        ].to_numpy(
            dtype=float
        )


        null_no_d1_mean = float(
            np.mean(
                null_no_d1
            )
        )


        delta_no_d1 = (
            real_no_d1
            - null_no_d1_mean
        )


        real_long = float(
            real_row[
                "mc_delay_11_30"
            ]
        )


        null_long_mean = float(
            null_rows[
                "mc_delay_11_30"
            ].mean()
        )


        delta_long = (
            real_long
            - null_long_mean
        )


    else:

        real_no_d1 = np.nan
        null_no_d1_mean = np.nan
        delta_no_d1 = np.nan

        real_long = np.nan
        null_long_mean = np.nan
        delta_long = np.nan


    interface_rows.append({

        "interface_id":
            interface_id,

        "interface_seed":
            INTERFACE_SEEDS[
                interface_id
            ],

        "real_MC":
            real_mc,

        "null_mean_MC":
            null_mean,

        "null_sd_MC":
            null_sd,

        "delta_MC":
            delta,

        "relative_advantage_percent":
            relative,

        "real_wins_vs_5_nulls":
            wins,

        "real_MC_d2_30":
            real_no_d1,

        "null_mean_MC_d2_30":
            null_no_d1_mean,

        "delta_MC_d2_30":
            delta_no_d1,

        "real_MC_d11_30":
            real_long,

        "null_mean_MC_d11_30":
            null_long_mean,

        "delta_MC_d11_30":
            delta_long
    })


interfaces_df = pd.DataFrame(
    interface_rows
)


interfaces_df.to_csv(
    INTERFACE_SUMMARY_FILE,
    index=False
)


# ============================================================
# FINAL GATE
# ============================================================

total_interface_wins = int(
    np.sum(
        interfaces_df[
            "delta_MC"
        ] > 0
    )
)


mean_delta = float(
    interfaces_df[
        "delta_MC"
    ].mean()
)


median_delta = float(
    interfaces_df[
        "delta_MC"
    ].median()
)


mean_relative = float(
    interfaces_df[
        "relative_advantage_percent"
    ].mean()
)


new_interfaces = interfaces_df.loc[
    interfaces_df[
        "interface_id"
    ] > 0
]


new_no_d1_wins = int(
    np.sum(
        new_interfaces[
            "delta_MC_d2_30"
        ] > 0
    )
)


mean_no_d1_delta = float(
    new_interfaces[
        "delta_MC_d2_30"
    ].mean()
)


new_long_wins = int(
    np.sum(
        new_interfaces[
            "delta_MC_d11_30"
        ] > 0
    )
)


mean_long_delta = float(
    new_interfaces[
        "delta_MC_d11_30"
    ].mean()
)


primary_pass = (
    total_interface_wins
    >= REQUIRED_TOTAL_INTERFACE_WINS
)


mean_pass = (
    mean_delta > 0
)


delay1_guard_pass = (
    (
        new_no_d1_wins
        >= REQUIRED_NEW_NO_D1_WINS
    )
    and
    (
        mean_no_d1_delta > 0
    )
)


FINAL_PASS = (
    primary_pass
    and
    mean_pass
    and
    delay1_guard_pass
)


final_df = pd.DataFrame([{

    "interfaces_positive":
        total_interface_wins,

    "interfaces_total":
        5,

    "mean_delta_MC":
        mean_delta,

    "median_delta_MC":
        median_delta,

    "mean_relative_advantage_percent":
        mean_relative,

    "new_interfaces_positive_without_delay1":
        new_no_d1_wins,

    "new_interfaces_total":
        4,

    "mean_delta_MC_without_delay1":
        mean_no_d1_delta,

    "new_interfaces_positive_long_memory":
        new_long_wins,

    "mean_delta_MC_delay_11_30":
        mean_long_delta,

    "primary_pass":
        primary_pass,

    "mean_pass":
        mean_pass,

    "delay1_guard_pass":
        delay1_guard_pass,

    "FINAL_MEMORY_GATE_PASS":
        FINAL_PASS,

    "sequence_hash":
        SEQUENCE_HASH
}])


final_df.to_csv(
    FINAL_FILE,
    index=False
)


# ============================================================
# RESULTADOS
# ============================================================

log("")
log("=" * 82)
log("RESULTADO FINAL v0.7")
log("=" * 82)


for row in interfaces_df.itertuples():

    log(
        f"Interface-{int(row.interface_id)} | "
        f"MaleCNS={row.real_MC:.6f} | "
        f"Null={row.null_mean_MC:.6f} "
        f"± {row.null_sd_MC:.6f} | "
        f"Δ={row.delta_MC:+.6f} | "
        f"relative={row.relative_advantage_percent:+.2f}% | "
        f"wins={int(row.real_wins_vs_5_nulls)}/5"
    )


log("")
log(
    f"Interfaces con ΔMC positivo: "
    f"{total_interface_wins}/5"
)

log(
    f"Media ΔMC: "
    f"{mean_delta:+.6f}"
)

log(
    f"Mediana ΔMC: "
    f"{median_delta:+.6f}"
)

log(
    f"Ventaja relativa media: "
    f"{mean_relative:+.2f}%"
)


log("")
log("CONTROL SIN DELAY 1:")


for row in new_interfaces.itertuples():

    log(
        f"Interface-{int(row.interface_id)} | "
        f"ΔMC d2-30="
        f"{row.delta_MC_d2_30:+.6f} | "
        f"ΔMC d11-30="
        f"{row.delta_MC_d11_30:+.6f}"
    )


log(
    f"Interfaces nuevas positivas d2-30: "
    f"{new_no_d1_wins}/4"
)

log(
    f"Media Δ d2-30: "
    f"{mean_no_d1_delta:+.6f}"
)

log(
    f"Interfaces nuevas positivas d11-30: "
    f"{new_long_wins}/4"
)

log(
    f"Media Δ d11-30: "
    f"{mean_long_delta:+.6f}"
)


log("")
log("------------------------------------------")

log(
    f"PRIMARY PASS >=4/5: "
    f"{primary_pass}"
)

log(
    f"MEAN Δ > 0: "
    f"{mean_pass}"
)

log(
    f"DELAY-1 GUARD PASS: "
    f"{delay1_guard_pass}"
)

log("------------------------------------------")

log(
    f"FINAL MEMORY GATE PASS: "
    f"{FINAL_PASS}"
)

log("------------------------------------------")


if FINAL_PASS:

    log(
        "DECISION PRE-REGISTRADA: "
        "cerrar fase de memoria y avanzar "
        "a aprendizaje asociativo."
    )

else:

    log(
        "DECISION PRE-REGISTRADA: "
        "la ventaja no es suficientemente "
        "robusta a la interfaz aleatoria. "
        "No optimizar parámetros; analizar "
        "el acoplamiento input/readout."
    )


log("")
log("Archivos:")

log(
    RUN_SUMMARY_FILE
)

log(
    INTERFACE_SUMMARY_FILE
)

log(
    FINAL_FILE
)

log(
    LOG_FILE
)

log("")
log("FIN v0.7")

Writing /content/drive/MyDrive/malecns_ai_v0_1/benchmark_interface_robustness_v07.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_interface_robustness_v07.py

/content
[2026-09-16 02:49:03] ==================================================================================
[2026-09-16 02:49:03] MaleCNS-AI v0.7 — INTERFACE ROBUSTNESS / FINAL MEMORY GATE
[2026-09-16 02:49:03] ==================================================================================
[2026-09-16 02:49:03] Criterio primario: MaleCNS > media Null en >= 4/5 interfaces.
[2026-09-16 02:49:03] Criterio global: media ΔMC entre interfaces > 0.
[2026-09-16 02:49:03] Control delay-1: en interfaces nuevas, ΔMC delays 2–30 positivo en >=3/4 y media positiva.
[2026-09-16 02:49:03] Cargando MaleCNSGraph...
[2026-09-16 02:49:06] Cargando topología MaleCNS...
[2026-09-16 02:49:09] Neuronas: 165,122
[2026-09-16 02:49:09] Conexiones reales: 25,563,096
[2026-09-16 02:49:09] Reconstruyendo las secuencias exactas del benchmark original...
[2026-09-16 02:49:09] Train: 112 secuencias
[2026-09-16 02:49:09] Test: 48 secuencias
[2026-09-16 02:49:09] Sequence hash: 37023e12a9205ef6
[2026-09-16 02:

In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_interface_robustness_v07.py

[Errno 2] No such file or directory: '/content/drive/MyDrive/malecns_ai_v0_1'
/content
python3: can't open file '/content/benchmark_interface_robustness_v07.py': [Errno 2] No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Mounted at /content/drive
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_interface_robustness_v07.py

import os
import gc
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from tqdm.auto import tqdm

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# CONFIGURACION — NO CAMBIAR DESPUES DE INICIAR
# ============================================================

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

OUTDIR = os.path.join(
    PROJECT,
    "results_v07_interface_robustness"
)

INTERFACE_DIR = os.path.join(
    OUTDIR,
    "interfaces"
)

DELAY_DIR = os.path.join(
    OUTDIR,
    "delays"
)

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(INTERFACE_DIR, exist_ok=True)
os.makedirs(DELAY_DIR, exist_ok=True)

os.chdir(PROJECT)


BASE_SEED = 20260914

N_SAMPLES = 160
SEQUENCE_LENGTH = 70

N_INPUT = 256
N_READOUT = 512

MAX_DELAY = 30

TRAIN_FRACTION = 0.70
RIDGE_ALPHA = 1e-2

LEAK = 0.2
GAIN = 1.2

SIGN_MODE = "biological_fast"


# Interfaz 0 = original.
# 1–4 = nuevas interfaces independientes.
INTERFACE_SEEDS = {
    0: BASE_SEED,
    1: BASE_SEED + 101001,
    2: BASE_SEED + 202002,
    3: BASE_SEED + 303003,
    4: BASE_SEED + 404004,
}


RUN_SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v07_runs.csv"
)

INTERFACE_SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v07_interfaces.csv"
)

FINAL_FILE = os.path.join(
    OUTDIR,
    "v07_final_gate.csv"
)

LOG_FILE = os.path.join(
    OUTDIR,
    "v07_run.log"
)


# ============================================================
# CRITERIOS PRE-REGISTRADOS
# ============================================================

REQUIRED_TOTAL_INTERFACE_WINS = 4   # de 5
REQUIRED_NEW_NO_D1_WINS = 3        # de 4


# ============================================================
# LOG
# ============================================================

def log(message=""):

    stamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    line = f"[{stamp}] {message}"

    tqdm.write(line)

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(line + "\n")


log("=" * 82)
log("MaleCNS-AI v0.7 — INTERFACE ROBUSTNESS / FINAL MEMORY GATE")
log("=" * 82)

log(
    "Criterio primario: MaleCNS > media Null en >= 4/5 interfaces."
)

log(
    "Criterio global: media ΔMC entre interfaces > 0."
)

log(
    "Control delay-1: en interfaces nuevas, "
    "ΔMC delays 2–30 positivo en >=3/4 y media positiva."
)


# ============================================================
# CARGAR DATOS
# ============================================================

log("Cargando MaleCNSGraph...")

base = MaleCNSGraph()


REAL_PATH = os.path.join(
    PROJECT,
    "adjacency_binary_real_v02.npz"
)


log("Cargando topología MaleCNS...")

A_real = sparse.load_npz(
    REAL_PATH
).tocsr().astype(np.float32)


n = A_real.shape[0]


log(
    f"Neuronas: {n:,}"
)

log(
    f"Conexiones reales: {A_real.nnz:,}"
)


# ============================================================
# GRAPH PROXY
# ============================================================

class GraphProxy:

    def __init__(
        self,
        A,
        nodes
    ):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)


    @property
    def n_nodes(self):
        return self.A.shape[0]


    @property
    def n_edges(self):
        return self.A.nnz


real_graph = GraphProxy(
    A_real,
    base.nodes
)


# ============================================================
# REDES
# ============================================================

NETWORK_PATHS = {

    0:
        REAL_PATH,

    1:
        os.path.join(
            PROJECT,
            "adjacency_degree_matched_v02.npz"
        ),

    2:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_02.npz"
        ),

    3:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_03.npz"
        ),

    4:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_04.npz"
        ),

    5:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_05.npz"
        )
}


NETWORK_NAMES = {
    0: "MaleCNS",
    1: "Null-1",
    2: "Null-2",
    3: "Null-3",
    4: "Null-4",
    5: "Null-5"
}


for network_id, path in NETWORK_PATHS.items():

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"No existe {NETWORK_NAMES[network_id]}:\n{path}"
        )


# ============================================================
# RECONSTRUIR EXACTAMENTE LAS SECUENCIAS ORIGINALES
#
# IMPORTANTE:
# v0.2/v0.3 generaban:
# input -> readout -> sequences
#
# Por eso consumimos RNG en exactamente ese orden.
# ============================================================

log(
    "Reconstruyendo las secuencias exactas "
    "del benchmark original..."
)


rng_original = np.random.default_rng(
    BASE_SEED
)


original_input = rng_original.choice(
    n,
    size=N_INPUT,
    replace=False
)


remaining_original = np.setdiff1d(
    np.arange(n),
    original_input
)


original_readout = rng_original.choice(
    remaining_original,
    size=N_READOUT,
    replace=False
)


sequences = rng_original.choice(
    [-1.0, 1.0],
    size=(
        N_SAMPLES,
        SEQUENCE_LENGTH
    )
).astype(np.float32)


n_train = int(
    N_SAMPLES
    * TRAIN_FRACTION
)


train_ids = np.arange(
    n_train
)


test_ids = np.arange(
    n_train,
    N_SAMPLES
)


log(
    f"Train: {len(train_ids)} secuencias"
)

log(
    f"Test: {len(test_ids)} secuencias"
)


# ============================================================
# HASH
# ============================================================

def array_hash(arr):

    return hashlib.sha256(
        np.ascontiguousarray(arr).tobytes()
    ).hexdigest()[:16]


SEQUENCE_HASH = array_hash(
    sequences
)


log(
    f"Sequence hash: {SEQUENCE_HASH}"
)


# ============================================================
# CREAR / CARGAR INTERFACES
# ============================================================

def get_interface(
    interface_id
):

    path = os.path.join(
        INTERFACE_DIR,
        f"interface_{interface_id:02d}.npz"
    )


    if os.path.exists(path):

        data = np.load(path)

        input_nodes = (
            data["input_nodes"]
            .astype(np.int64)
        )

        readout_nodes = (
            data["readout_nodes"]
            .astype(np.int64)
        )

        log(
            f"Interface-{interface_id}: "
            "reutilizando nodos guardados."
        )


    elif interface_id == 0:

        input_nodes = (
            original_input.copy()
        )

        readout_nodes = (
            original_readout.copy()
        )


        np.savez_compressed(
            path,
            input_nodes=input_nodes,
            readout_nodes=readout_nodes
        )


        log(
            "Interface-0: reconstruida "
            "desde el benchmark original."
        )


    else:

        seed = INTERFACE_SEEDS[
            interface_id
        ]


        rng = np.random.default_rng(
            seed
        )


        input_nodes = rng.choice(
            n,
            size=N_INPUT,
            replace=False
        )


        remaining = np.setdiff1d(
            np.arange(n),
            input_nodes
        )


        readout_nodes = rng.choice(
            remaining,
            size=N_READOUT,
            replace=False
        )


        np.savez_compressed(
            path,
            input_nodes=input_nodes,
            readout_nodes=readout_nodes
        )


        log(
            f"Interface-{interface_id}: "
            f"creada con seed={seed}"
        )


    # Validaciones
    if len(np.unique(input_nodes)) != N_INPUT:

        raise RuntimeError(
            "Input nodes duplicados."
        )


    if len(np.unique(readout_nodes)) != N_READOUT:

        raise RuntimeError(
            "Readout nodes duplicados."
        )


    if len(
        np.intersect1d(
            input_nodes,
            readout_nodes
        )
    ) != 0:

        raise RuntimeError(
            "Input y readout se superponen."
        )


    log(
        f"Interface-{interface_id}: "
        f"input hash={array_hash(input_nodes)} | "
        f"readout hash={array_hash(readout_nodes)}"
    )


    return (
        input_nodes,
        readout_nodes
    )


interfaces = {}


for interface_id in range(5):

    interfaces[
        interface_id
    ] = get_interface(
        interface_id
    )


# ============================================================
# DATASET
# ============================================================

def build_dataset(
    states,
    ids
):

    X = []
    Y = []


    delays = np.arange(
        1,
        MAX_DELAY + 1
    )


    for sample in ids:

        seq = sequences[
            sample
        ]


        for t in range(
            MAX_DELAY,
            SEQUENCE_LENGTH
        ):

            X.append(
                states[
                    sample,
                    t,
                    :
                ]
            )


            Y.append(
                seq[
                    t - delays
                ]
            )


    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            Y,
            dtype=np.float32
        )
    )


# ============================================================
# SIMULACION
# ============================================================

def generate_states(
    graph,
    input_nodes,
    readout_nodes,
    task_name
):

    reservoir = ConnectomeReservoir(
        graph,
        leak=LEAK,
        gain=GAIN,
        sign_mode=SIGN_MODE
    )


    states = np.empty(
        (
            N_SAMPLES,
            SEQUENCE_LENGTH,
            N_READOUT
        ),
        dtype=np.float32
    )


    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )


    start = time.time()


    log(
        f"{task_name}: "
        f"iniciando {N_SAMPLES} secuencias."
    )


    bar = tqdm(
        total=N_SAMPLES,
        desc=task_name,
        unit="seq",
        dynamic_ncols=True,
        leave=False
    )


    for sample in range(
        N_SAMPLES
    ):

        reservoir.reset()


        for t, value in enumerate(
            sequences[sample]
        ):

            external.fill(
                0.0
            )


            external[
                input_nodes
            ] = value


            state = reservoir.step(
                external
            )


            states[
                sample,
                t,
                :
            ] = state[
                readout_nodes
            ]


        bar.update(1)


        if (
            sample + 1
        ) % 20 == 0:

            elapsed = (
                time.time()
                - start
            )


            done = (
                sample + 1
            )


            pct = (
                done
                / N_SAMPLES
                * 100
            )


            sec_per_seq = (
                elapsed
                / done
            )


            eta = (
                N_SAMPLES
                - done
            ) * sec_per_seq


            log(
                f"{task_name}: "
                f"{done}/{N_SAMPLES} "
                f"({pct:.1f}%) | "
                f"ETA ~ {eta/60:.1f} min"
            )


    bar.close()


    elapsed = (
        time.time()
        - start
    )


    log(
        f"{task_name}: simulación completa "
        f"en {elapsed/60:.1f} min."
    )


    return states


# ============================================================
# EVALUACION
# ============================================================

def evaluate(
    states,
    task_name
):

    log(
        f"{task_name}: "
        "construyendo train/test..."
    )


    X_train, Y_train = build_dataset(
        states,
        train_ids
    )


    X_test, Y_test = build_dataset(
        states,
        test_ids
    )


    log(
        f"{task_name}: "
        f"train={X_train.shape}, "
        f"test={X_test.shape}"
    )


    mean = X_train.mean(
        axis=0
    )


    std = (
        X_train.std(
            axis=0
        )
        + 1e-6
    )


    X_train = (
        X_train
        - mean
    ) / std


    X_test = (
        X_test
        - mean
    ) / std


    log(
        f"{task_name}: entrenando ridge..."
    )


    XTX = (
        X_train.T
        @ X_train
    ).astype(np.float64)


    XTY = (
        X_train.T
        @ Y_train
    ).astype(np.float64)


    XTX.flat[
        ::XTX.shape[0] + 1
    ] += RIDGE_ALPHA


    W = np.linalg.solve(
        XTX,
        XTY
    )


    pred = (
        X_test.astype(np.float64)
        @ W
    )


    rows = []


    for d in range(
        MAX_DELAY
    ):

        y = Y_test[:, d]
        p = pred[:, d]


        binary = np.where(
            p >= 0,
            1.0,
            -1.0
        )


        acc = float(
            np.mean(
                binary == y
            )
        )


        corr = float(
            np.corrcoef(
                p,
                y
            )[0, 1]
        )


        if not np.isfinite(corr):

            corr = 0.0


        rows.append({

            "delay":
                d + 1,

            "accuracy":
                acc,

            "corr":
                corr,

            "memory":
                corr ** 2
        })


    df = pd.DataFrame(
        rows
    )


    mc_total = float(
        df[
            "memory"
        ].sum()
    )


    mc_no_d1 = float(
        df.loc[
            df[
                "delay"
            ] >= 2,
            "memory"
        ].sum()
    )


    mc_long = float(
        df.loc[
            df[
                "delay"
            ] >= 11,
            "memory"
        ].sum()
    )


    log(
        f"{task_name}: "
        f"MC total={mc_total:.6f} | "
        f"MC d2-30={mc_no_d1:.6f} | "
        f"MC d11-30={mc_long:.6f}"
    )


    return (
        df,
        mc_total,
        mc_no_d1,
        mc_long
    )


# ============================================================
# RESULTADOS EXISTENTES
# ============================================================

if os.path.exists(
    RUN_SUMMARY_FILE
):

    runs = pd.read_csv(
        RUN_SUMMARY_FILE
    )

else:

    runs = pd.DataFrame(
        columns=[
            "task_key",
            "interface_id",
            "interface_seed",
            "network_id",
            "network",
            "memory_capacity",
            "mc_delay_2_30",
            "mc_delay_11_30",
            "source",
            "sequence_hash"
        ]
    )


# ============================================================
# REUTILIZAR INTERFAZ 0 DESDE v0.3A
# ============================================================

V03_FILE = os.path.join(
    PROJECT,
    "results_v03a",
    "null_ensemble_summary.csv"
)


v03 = pd.read_csv(
    V03_FILE
)


existing_keys = set(
    runs[
        "task_key"
    ].astype(str)
    .tolist()
)


for network_id in range(
    6
):

    task_key = (
        f"i00_n{network_id}"
    )


    if task_key in existing_keys:

        continue


    name = NETWORK_NAMES[
        network_id
    ]


    row = v03.loc[
        v03[
            "null_id"
        ] == network_id
    ]


    if len(row) != 1:

        raise RuntimeError(
            f"No pude recuperar {name} "
            "desde v0.3A."
        )


    MC = float(
        row[
            "memory_capacity"
        ].iloc[0]
    )


    new_row = pd.DataFrame([{

        "task_key":
            task_key,

        "interface_id":
            0,

        "interface_seed":
            INTERFACE_SEEDS[0],

        "network_id":
            network_id,

        "network":
            name,

        "memory_capacity":
            MC,

        # No usamos estos valores para el gate
        # anti-delay1 porque fueron generados
        # en experimentos anteriores.
        "mc_delay_2_30":
            np.nan,

        "mc_delay_11_30":
            np.nan,

        "source":
            "reused_v03a",

        "sequence_hash":
            SEQUENCE_HASH
    }])


    if runs.empty:

        runs = new_row

    else:

        runs = pd.concat(
            [
                runs,
                new_row
            ],
            ignore_index=True
        )


    log(
        f"Interface-0 / {name}: "
        f"reutilizado MC={MC:.6f}"
    )


runs = (
    runs
    .drop_duplicates(
        subset=[
            "task_key"
        ],
        keep="last"
    )
)


runs.to_csv(
    RUN_SUMMARY_FILE,
    index=False
)


# ============================================================
# DEFINIR LAS 24 EJECUCIONES NUEVAS
# ============================================================

tasks = []


for interface_id in range(
    1,
    5
):

    for network_id in range(
        6
    ):

        tasks.append({

            "task_key":
                f"i{interface_id:02d}_n{network_id}",

            "interface_id":
                interface_id,

            "network_id":
                network_id
        })


completed_keys = set(
    runs[
        "task_key"
    ].astype(str)
    .tolist()
)


completed_new = sum(
    task[
        "task_key"
    ] in completed_keys

    for task in tasks
)


log("")
log(
    f"Ejecuciones nuevas necesarias: "
    f"{len(tasks)}"
)

log(
    f"Ya terminadas: "
    f"{completed_new}/{len(tasks)}"
)


# ============================================================
# PROGRESO GLOBAL
# ============================================================

global_bar = tqdm(
    total=len(tasks),
    initial=completed_new,
    desc="PROGRESO GLOBAL v0.7",
    unit="run",
    dynamic_ncols=True
)


# ============================================================
# EJECUTAR
# ============================================================

for index, task in enumerate(
    tasks,
    start=1
):

    task_key = task[
        "task_key"
    ]


    if task_key in completed_keys:

        log(
            f"{task_key}: "
            "ya completado; saltando."
        )

        continue


    interface_id = int(
        task[
            "interface_id"
        ]
    )


    network_id = int(
        task[
            "network_id"
        ]
    )


    network_name = NETWORK_NAMES[
        network_id
    ]


    input_nodes, readout_nodes = (
        interfaces[
            interface_id
        ]
    )


    log("")
    log("=" * 82)

    log(
        f"TAREA {index}/{len(tasks)} | "
        f"Interface-{interface_id} | "
        f"{network_name}"
    )

    log("=" * 82)


    # --------------------------------------------------------
    # CARGAR GRAFO
    # --------------------------------------------------------

    if network_id == 0:

        graph = real_graph
        A_current = None


    else:

        path = NETWORK_PATHS[
            network_id
        ]


        log(
            f"Cargando {network_name}..."
        )


        A_current = sparse.load_npz(
            path
        ).tocsr().astype(np.float32)


        graph = GraphProxy(
            A_current,
            base.nodes
        )


    log(
        f"{network_name}: "
        f"{graph.n_edges:,} conexiones"
    )


    # --------------------------------------------------------
    # SIMULAR
    # --------------------------------------------------------

    states = generate_states(
        graph,
        input_nodes,
        readout_nodes,
        task_key
    )


    # --------------------------------------------------------
    # EVALUAR
    # --------------------------------------------------------

    (
        delay_df,
        mc_total,
        mc_no_d1,
        mc_long
    ) = evaluate(
        states,
        task_key
    )


    # --------------------------------------------------------
    # GUARDAR DELAYS
    # --------------------------------------------------------

    delay_file = os.path.join(
        DELAY_DIR,
        f"{task_key}_delays.csv"
    )


    delay_df.to_csv(
        delay_file,
        index=False
    )


    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    row = pd.DataFrame([{

        "task_key":
            task_key,

        "interface_id":
            interface_id,

        "interface_seed":
            INTERFACE_SEEDS[
                interface_id
            ],

        "network_id":
            network_id,

        "network":
            network_name,

        "memory_capacity":
            mc_total,

        "mc_delay_2_30":
            mc_no_d1,

        "mc_delay_11_30":
            mc_long,

        "source":
            "v07",

        "sequence_hash":
            SEQUENCE_HASH
    }])


    if runs.empty:

        runs = row

    else:

        runs = pd.concat(
            [
                runs,
                row
            ],
            ignore_index=True
        )


    runs = (
        runs
        .drop_duplicates(
            subset=[
                "task_key"
            ],
            keep="last"
        )
    )


    temp_file = (
        RUN_SUMMARY_FILE
        + ".tmp"
    )


    runs.to_csv(
        temp_file,
        index=False
    )


    os.replace(
        temp_file,
        RUN_SUMMARY_FILE
    )


    completed_keys.add(
        task_key
    )


    log(
        f"{task_key}: "
        "CHECKPOINT GUARDADO."
    )


    # --------------------------------------------------------
    # LIMPIEZA
    # --------------------------------------------------------

    del states
    del delay_df
    del graph


    if A_current is not None:

        del A_current


    gc.collect()


    global_bar.update(1)


global_bar.close()


# ============================================================
# VALIDAR FINAL
# ============================================================

runs = pd.read_csv(
    RUN_SUMMARY_FILE
)


expected_total = (
    5 * 6
)


if len(runs) != expected_total:

    log(
        f"Hay {len(runs)}/{expected_total} "
        "resultados."
    )

    log(
        "Ejecuta nuevamente el script."
    )

    raise SystemExit


# ============================================================
# RESUMEN POR INTERFAZ
# ============================================================

interface_rows = []


for interface_id in range(
    5
):

    subset = runs.loc[
        runs[
            "interface_id"
        ] == interface_id
    ]


    real_row = subset.loc[
        subset[
            "network_id"
        ] == 0
    ].iloc[0]


    null_rows = subset.loc[
        subset[
            "network_id"
        ] > 0
    ]


    real_mc = float(
        real_row[
            "memory_capacity"
        ]
    )


    null_values = null_rows[
        "memory_capacity"
    ].to_numpy(
        dtype=float
    )


    null_mean = float(
        np.mean(
            null_values
        )
    )


    null_sd = float(
        np.std(
            null_values,
            ddof=1
        )
    )


    delta = (
        real_mc
        - null_mean
    )


    relative = (
        real_mc
        / null_mean
        - 1
    ) * 100


    wins = int(
        np.sum(
            real_mc
            > null_values
        )
    )


    # Delay 2-30:
    # sólo es criterio formal para las cuatro nuevas.
    if interface_id > 0:

        real_no_d1 = float(
            real_row[
                "mc_delay_2_30"
            ]
        )


        null_no_d1 = null_rows[
            "mc_delay_2_30"
        ].to_numpy(
            dtype=float
        )


        null_no_d1_mean = float(
            np.mean(
                null_no_d1
            )
        )


        delta_no_d1 = (
            real_no_d1
            - null_no_d1_mean
        )


        real_long = float(
            real_row[
                "mc_delay_11_30"
            ]
        )


        null_long_mean = float(
            null_rows[
                "mc_delay_11_30"
            ].mean()
        )


        delta_long = (
            real_long
            - null_long_mean
        )


    else:

        real_no_d1 = np.nan
        null_no_d1_mean = np.nan
        delta_no_d1 = np.nan

        real_long = np.nan
        null_long_mean = np.nan
        delta_long = np.nan


    interface_rows.append({

        "interface_id":
            interface_id,

        "interface_seed":
            INTERFACE_SEEDS[
                interface_id
            ],

        "real_MC":
            real_mc,

        "null_mean_MC":
            null_mean,

        "null_sd_MC":
            null_sd,

        "delta_MC":
            delta,

        "relative_advantage_percent":
            relative,

        "real_wins_vs_5_nulls":
            wins,

        "real_MC_d2_30":
            real_no_d1,

        "null_mean_MC_d2_30":
            null_no_d1_mean,

        "delta_MC_d2_30":
            delta_no_d1,

        "real_MC_d11_30":
            real_long,

        "null_mean_MC_d11_30":
            null_long_mean,

        "delta_MC_d11_30":
            delta_long
    })


interfaces_df = pd.DataFrame(
    interface_rows
)


interfaces_df.to_csv(
    INTERFACE_SUMMARY_FILE,
    index=False
)


# ============================================================
# FINAL GATE
# ============================================================

total_interface_wins = int(
    np.sum(
        interfaces_df[
            "delta_MC"
        ] > 0
    )
)


mean_delta = float(
    interfaces_df[
        "delta_MC"
    ].mean()
)


median_delta = float(
    interfaces_df[
        "delta_MC"
    ].median()
)


mean_relative = float(
    interfaces_df[
        "relative_advantage_percent"
    ].mean()
)


new_interfaces = interfaces_df.loc[
    interfaces_df[
        "interface_id"
    ] > 0
]


new_no_d1_wins = int(
    np.sum(
        new_interfaces[
            "delta_MC_d2_30"
        ] > 0
    )
)


mean_no_d1_delta = float(
    new_interfaces[
        "delta_MC_d2_30"
    ].mean()
)


new_long_wins = int(
    np.sum(
        new_interfaces[
            "delta_MC_d11_30"
        ] > 0
    )
)


mean_long_delta = float(
    new_interfaces[
        "delta_MC_d11_30"
    ].mean()
)


primary_pass = (
    total_interface_wins
    >= REQUIRED_TOTAL_INTERFACE_WINS
)


mean_pass = (
    mean_delta > 0
)


delay1_guard_pass = (
    (
        new_no_d1_wins
        >= REQUIRED_NEW_NO_D1_WINS
    )
    and
    (
        mean_no_d1_delta > 0
    )
)


FINAL_PASS = (
    primary_pass
    and
    mean_pass
    and
    delay1_guard_pass
)


final_df = pd.DataFrame([{

    "interfaces_positive":
        total_interface_wins,

    "interfaces_total":
        5,

    "mean_delta_MC":
        mean_delta,

    "median_delta_MC":
        median_delta,

    "mean_relative_advantage_percent":
        mean_relative,

    "new_interfaces_positive_without_delay1":
        new_no_d1_wins,

    "new_interfaces_total":
        4,

    "mean_delta_MC_without_delay1":
        mean_no_d1_delta,

    "new_interfaces_positive_long_memory":
        new_long_wins,

    "mean_delta_MC_delay_11_30":
        mean_long_delta,

    "primary_pass":
        primary_pass,

    "mean_pass":
        mean_pass,

    "delay1_guard_pass":
        delay1_guard_pass,

    "FINAL_MEMORY_GATE_PASS":
        FINAL_PASS,

    "sequence_hash":
        SEQUENCE_HASH
}])


final_df.to_csv(
    FINAL_FILE,
    index=False
)


# ============================================================
# RESULTADOS
# ============================================================

log("")
log("=" * 82)
log("RESULTADO FINAL v0.7")
log("=" * 82)


for row in interfaces_df.itertuples():

    log(
        f"Interface-{int(row.interface_id)} | "
        f"MaleCNS={row.real_MC:.6f} | "
        f"Null={row.null_mean_MC:.6f} "
        f"± {row.null_sd_MC:.6f} | "
        f"Δ={row.delta_MC:+.6f} | "
        f"relative={row.relative_advantage_percent:+.2f}% | "
        f"wins={int(row.real_wins_vs_5_nulls)}/5"
    )


log("")
log(
    f"Interfaces con ΔMC positivo: "
    f"{total_interface_wins}/5"
)

log(
    f"Media ΔMC: "
    f"{mean_delta:+.6f}"
)

log(
    f"Mediana ΔMC: "
    f"{median_delta:+.6f}"
)

log(
    f"Ventaja relativa media: "
    f"{mean_relative:+.2f}%"
)


log("")
log("CONTROL SIN DELAY 1:")


for row in new_interfaces.itertuples():

    log(
        f"Interface-{int(row.interface_id)} | "
        f"ΔMC d2-30="
        f"{row.delta_MC_d2_30:+.6f} | "
        f"ΔMC d11-30="
        f"{row.delta_MC_d11_30:+.6f}"
    )


log(
    f"Interfaces nuevas positivas d2-30: "
    f"{new_no_d1_wins}/4"
)

log(
    f"Media Δ d2-30: "
    f"{mean_no_d1_delta:+.6f}"
)

log(
    f"Interfaces nuevas positivas d11-30: "
    f"{new_long_wins}/4"
)

log(
    f"Media Δ d11-30: "
    f"{mean_long_delta:+.6f}"
)


log("")
log("------------------------------------------")

log(
    f"PRIMARY PASS >=4/5: "
    f"{primary_pass}"
)

log(
    f"MEAN Δ > 0: "
    f"{mean_pass}"
)

log(
    f"DELAY-1 GUARD PASS: "
    f"{delay1_guard_pass}"
)

log("------------------------------------------")

log(
    f"FINAL MEMORY GATE PASS: "
    f"{FINAL_PASS}"
)

log("------------------------------------------")


if FINAL_PASS:

    log(
        "DECISION PRE-REGISTRADA: "
        "cerrar fase de memoria y avanzar "
        "a aprendizaje asociativo."
    )

else:

    log(
        "DECISION PRE-REGISTRADA: "
        "la ventaja no es suficientemente "
        "robusta a la interfaz aleatoria. "
        "No optimizar parámetros; analizar "
        "el acoplamiento input/readout."
    )


log("")
log("Archivos:")

log(
    RUN_SUMMARY_FILE
)

log(
    INTERFACE_SUMMARY_FILE
)

log(
    FINAL_FILE
)

log(
    LOG_FILE
)

log("")
log("FIN v0.7")

Overwriting /content/drive/MyDrive/malecns_ai_v0_1/benchmark_interface_robustness_v07.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_interface_robustness_v07.py

/content
[2026-09-16 05:15:37] ==================================================================================
[2026-09-16 05:15:38] MaleCNS-AI v0.7 — INTERFACE ROBUSTNESS / FINAL MEMORY GATE
[2026-09-16 05:15:38] ==================================================================================
[2026-09-16 05:15:38] Criterio primario: MaleCNS > media Null en >= 4/5 interfaces.
[2026-09-16 05:15:38] Criterio global: media ΔMC entre interfaces > 0.
[2026-09-16 05:15:38] Control delay-1: en interfaces nuevas, ΔMC delays 2–30 positivo en >=3/4 y media positiva.
[2026-09-16 05:15:38] Cargando MaleCNSGraph...
[2026-09-16 05:15:46] Cargando topología MaleCNS...
[2026-09-16 05:15:53] Neuronas: 165,122
[2026-09-16 05:15:53] Conexiones reales: 25,563,096
[2026-09-16 05:15:53] Reconstruyendo las secuencias exactas del benchmark original...
[2026-09-16 05:15:54] Train: 112 secuencias
[2026-09-16 05:15:54] Test: 48 secuencias
[2026-09-16 05:15:54] Sequence hash: 37023e12a9205ef6
[2026-09-16 05:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_associative_learning_v08.py

import os
import gc
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from tqdm.auto import tqdm

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# CONFIGURACION PRE-REGISTRADA
# ============================================================

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

OUTDIR = os.path.join(
    PROJECT,
    "results_v08_associative_learning"
)

TASKDIR = os.path.join(
    OUTDIR,
    "tasks"
)

RUNDIR = os.path.join(
    OUTDIR,
    "runs"
)

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(TASKDIR, exist_ok=True)
os.makedirs(RUNDIR, exist_ok=True)

os.chdir(PROJECT)


# ------------------------------------------------------------
# Grafo / dinámica
# ------------------------------------------------------------

BASE_SEED = 20260914

LEAK = 0.2
GAIN = 1.2

SIGN_MODE = "biological_fast"

N_INPUT = 256
N_READOUT = 512


# ------------------------------------------------------------
# Tarea asociativa
# ------------------------------------------------------------

N_CLASSES = 8

MAX_SHOTS = 16

SHOT_CHECKPOINTS = [
    1,
    2,
    4,
    8,
    16
]

CUE_STEPS = 4
DELAY_STEPS = 8

# Promediamos los últimos estados del periodo sin señal
FEATURE_AVG_STEPS = 3

TRAIN_NOISE = 0.05

TEST_NOISE_LEVELS = [
    0.00,
    0.10,
    0.20
]

TEST_PER_CLASS = 12


TASK_SEEDS = {
    1: 20801001,
    2: 20802002,
    3: 20803003,
}


# ------------------------------------------------------------
# Gate PRE-REGISTRADO
# ------------------------------------------------------------

PRIMARY_NOISE = 0.10

REQUIRED_TASK_WINS = 2  # de 3


# ------------------------------------------------------------
# Archivos
# ------------------------------------------------------------

RESULT_FILE = os.path.join(
    OUTDIR,
    "v08_results.csv"
)

NETWORK_SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v08_network_summary.csv"
)

TASK_SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v08_task_comparison.csv"
)

FINAL_FILE = os.path.join(
    OUTDIR,
    "v08_final_gate.csv"
)

LOG_FILE = os.path.join(
    OUTDIR,
    "v08_run.log"
)


# ============================================================
# LOG
# ============================================================

def log(message=""):

    stamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    line = f"[{stamp}] {message}"

    tqdm.write(line)

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(line + "\n")


log("=" * 86)
log("MaleCNS-AI v0.8 — ONLINE ASSOCIATIVE LEARNING")
log("=" * 86)

log(
    f"Clases={N_CLASSES} | "
    f"shots={SHOT_CHECKPOINTS} | "
    f"cue={CUE_STEPS} pasos | "
    f"delay={DELAY_STEPS} pasos"
)

log(
    f"Train noise={TRAIN_NOISE:.0%} | "
    f"test noise={TEST_NOISE_LEVELS}"
)

log(
    "Criterio primario: associative score con 10% ruido."
)

log(
    "PASS: MaleCNS > media Null en >=2/3 tareas "
    "y Δ promedio > 0."
)


# ============================================================
# CARGAR GRAFO
# ============================================================

log("Cargando información MaleCNS...")

base = MaleCNSGraph()


REAL_PATH = os.path.join(
    PROJECT,
    "adjacency_binary_real_v02.npz"
)


A_real = sparse.load_npz(
    REAL_PATH
).tocsr().astype(np.float32)


n = A_real.shape[0]


log(
    f"Neuronas: {n:,}"
)

log(
    f"Conexiones: {A_real.nnz:,}"
)


# ============================================================
# GRAPH PROXY
# ============================================================

class GraphProxy:

    def __init__(
        self,
        A,
        nodes
    ):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)


    @property
    def n_nodes(self):

        return self.A.shape[0]


    @property
    def n_edges(self):

        return self.A.nnz


real_graph = GraphProxy(
    A_real,
    base.nodes
)


# ============================================================
# GRAFOS NULL
# ============================================================

NETWORK_PATHS = {

    0:
        REAL_PATH,

    1:
        os.path.join(
            PROJECT,
            "adjacency_degree_matched_v02.npz"
        ),

    2:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_02.npz"
        ),

    3:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_03.npz"
        ),

    4:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_04.npz"
        ),

    5:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_05.npz"
        )
}


NETWORK_NAMES = {

    0: "MaleCNS",
    1: "Null-1",
    2: "Null-2",
    3: "Null-3",
    4: "Null-4",
    5: "Null-5"
}


for network_id, path in NETWORK_PATHS.items():

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"No existe {NETWORK_NAMES[network_id]}:\n"
            f"{path}"
        )


# ============================================================
# INTERFACE-0 ORIGINAL
# ============================================================

INTERFACE_FILE = os.path.join(
    PROJECT,
    "results_v07_interface_robustness",
    "interfaces",
    "interface_00.npz"
)


if os.path.exists(
    INTERFACE_FILE
):

    data = np.load(
        INTERFACE_FILE
    )

    input_nodes = (
        data[
            "input_nodes"
        ].astype(np.int64)
    )

    readout_nodes = (
        data[
            "readout_nodes"
        ].astype(np.int64)
    )

    log(
        "Interface-0 cargada desde v0.7."
    )


else:

    log(
        "Interface-0 no encontrada; "
        "reconstruyendo desde BASE_SEED."
    )

    rng = np.random.default_rng(
        BASE_SEED
    )

    input_nodes = rng.choice(
        n,
        size=N_INPUT,
        replace=False
    )

    remaining = np.setdiff1d(
        np.arange(n),
        input_nodes
    )

    readout_nodes = rng.choice(
        remaining,
        size=N_READOUT,
        replace=False
    )


def short_hash(x):

    return hashlib.sha256(
        np.ascontiguousarray(
            x
        ).tobytes()
    ).hexdigest()[:16]


log(
    f"Input hash: "
    f"{short_hash(input_nodes)}"
)

log(
    f"Readout hash: "
    f"{short_hash(readout_nodes)}"
)


if len(
    np.intersect1d(
        input_nodes,
        readout_nodes
    )
) != 0:

    raise RuntimeError(
        "Input y readout se superponen."
    )


# ============================================================
# UTILIDADES DE TAREA
# ============================================================

def balanced_pattern(
    rng
):

    pattern = np.ones(
        N_INPUT,
        dtype=np.float32
    )

    pattern[
        :N_INPUT // 2
    ] = -1.0

    rng.shuffle(
        pattern
    )

    return pattern


def corrupt_pattern(
    pattern,
    fraction,
    rng
):

    result = (
        pattern.copy()
    )

    k = int(
        round(
            fraction
            * len(result)
        )
    )

    if k > 0:

        idx = rng.choice(
            len(result),
            size=k,
            replace=False
        )

        result[
            idx
        ] *= -1.0


    return result


# ============================================================
# CREAR / CARGAR LAS 3 TAREAS
# ============================================================

def get_task(
    task_id
):

    path = os.path.join(
        TASKDIR,
        f"task_{task_id:02d}.npz"
    )


    if os.path.exists(path):

        d = np.load(
            path
        )

        log(
            f"Task-{task_id}: "
            "reutilizando definición."
        )

        return {
            "base_patterns":
                d[
                    "base_patterns"
                ].astype(np.float32),

            "target_map":
                d[
                    "target_map"
                ].astype(np.int64),

            "train_cues":
                d[
                    "train_cues"
                ].astype(np.float32),

            "test_cues":
                d[
                    "test_cues"
                ].astype(np.float32),

            "test_labels":
                d[
                    "test_labels"
                ].astype(np.int64)
        }


    seed = TASK_SEEDS[
        task_id
    ]

    rng = np.random.default_rng(
        seed
    )


    # --------------------------------------------------------
    # 8 patrones distribuidos
    # Todos tienen exactamente 128 +1 y 128 -1
    # --------------------------------------------------------

    base_patterns = np.stack([

        balanced_pattern(
            rng
        )

        for _ in range(
            N_CLASSES
        )

    ])


    # Asociación arbitraria:
    # cue 0 no necesariamente -> respuesta 0
    target_map = rng.permutation(
        N_CLASSES
    ).astype(np.int64)


    # --------------------------------------------------------
    # TRAIN:
    # máximo 16 ejemplos por clase
    # --------------------------------------------------------

    train_cues = np.empty(
        (
            N_CLASSES,
            MAX_SHOTS,
            N_INPUT
        ),
        dtype=np.float32
    )


    for cue_class in range(
        N_CLASSES
    ):

        for shot in range(
            MAX_SHOTS
        ):

            train_cues[
                cue_class,
                shot
            ] = corrupt_pattern(
                base_patterns[
                    cue_class
                ],
                TRAIN_NOISE,
                rng
            )


    # --------------------------------------------------------
    # TEST:
    #
    # [noise, cue_class, example, input]
    # --------------------------------------------------------

    test_cues = np.empty(
        (
            len(
                TEST_NOISE_LEVELS
            ),
            N_CLASSES,
            TEST_PER_CLASS,
            N_INPUT
        ),
        dtype=np.float32
    )


    test_labels = np.empty(
        (
            len(
                TEST_NOISE_LEVELS
            ),
            N_CLASSES,
            TEST_PER_CLASS
        ),
        dtype=np.int64
    )


    for noise_idx, noise in enumerate(
        TEST_NOISE_LEVELS
    ):

        for cue_class in range(
            N_CLASSES
        ):

            label = target_map[
                cue_class
            ]


            for example in range(
                TEST_PER_CLASS
            ):

                test_cues[
                    noise_idx,
                    cue_class,
                    example
                ] = corrupt_pattern(
                    base_patterns[
                        cue_class
                    ],
                    noise,
                    rng
                )


                test_labels[
                    noise_idx,
                    cue_class,
                    example
                ] = label


    np.savez_compressed(
        path,
        base_patterns=base_patterns,
        target_map=target_map,
        train_cues=train_cues,
        test_cues=test_cues,
        test_labels=test_labels
    )


    log(
        f"Task-{task_id}: "
        f"creada seed={seed} | "
        f"target map={target_map.tolist()}"
    )


    return {
        "base_patterns":
            base_patterns,

        "target_map":
            target_map,

        "train_cues":
            train_cues,

        "test_cues":
            test_cues,

        "test_labels":
            test_labels
    }


tasks_data = {}


for task_id in TASK_SEEDS:

    tasks_data[
        task_id
    ] = get_task(
        task_id
    )


# ============================================================
# FEATURE NORMALIZATION
# ============================================================

def normalize_vector(
    x
):

    norm = np.linalg.norm(
        x
    )

    if norm < 1e-12:

        return np.zeros_like(
            x,
            dtype=np.float32
        )

    return (
        x / norm
    ).astype(np.float32)


def normalize_rows(
    X
):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    norms = np.linalg.norm(
        X,
        axis=1,
        keepdims=True
    )

    norms = np.maximum(
        norms,
        1e-12
    )

    return (
        X / norms
    ).astype(np.float32)


# ============================================================
# SIMULAR UN TRIAL
# ============================================================

def simulate_trial(
    reservoir,
    cue,
    external
):

    reservoir.reset()


    feature_buffer = []


    total_steps = (
        CUE_STEPS
        +
        DELAY_STEPS
    )


    capture_from = (
        total_steps
        -
        FEATURE_AVG_STEPS
    )


    for t in range(
        total_steps
    ):

        external.fill(
            0.0
        )


        if t < CUE_STEPS:

            external[
                input_nodes
            ] = cue


        state = reservoir.step(
            external
        )


        if t >= capture_from:

            feature_buffer.append(
                state[
                    readout_nodes
                ].copy()
            )


    feature = np.mean(
        feature_buffer,
        axis=0
    ).astype(np.float32)


    return feature


# ============================================================
# SIMULAR TODOS LOS EJEMPLOS DE UNA TAREA
# ============================================================

def generate_features(
    graph,
    task,
    run_name
):

    reservoir = ConnectomeReservoir(
        graph,
        leak=LEAK,
        gain=GAIN,
        sign_mode=SIGN_MODE
    )


    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )


    train_features = np.empty(
        (
            N_CLASSES,
            MAX_SHOTS,
            N_READOUT
        ),
        dtype=np.float32
    )


    test_features = np.empty(
        (
            len(
                TEST_NOISE_LEVELS
            ),
            N_CLASSES,
            TEST_PER_CLASS,
            N_READOUT
        ),
        dtype=np.float32
    )


    train_trials = (
        N_CLASSES
        * MAX_SHOTS
    )


    test_trials = (
        len(
            TEST_NOISE_LEVELS
        )
        * N_CLASSES
        * TEST_PER_CLASS
    )


    total_trials = (
        train_trials
        +
        test_trials
    )


    log(
        f"{run_name}: "
        f"{total_trials} trials | "
        f"{CUE_STEPS + DELAY_STEPS} pasos/trial"
    )


    bar = tqdm(
        total=total_trials,
        desc=run_name,
        unit="trial",
        dynamic_ncols=True,
        leave=False
    )


    start = time.time()

    completed = 0


    # --------------------------------------------------------
    # TRAIN FEATURES
    # --------------------------------------------------------

    for cue_class in range(
        N_CLASSES
    ):

        for shot in range(
            MAX_SHOTS
        ):

            cue = task[
                "train_cues"
            ][
                cue_class,
                shot
            ]


            train_features[
                cue_class,
                shot
            ] = simulate_trial(
                reservoir,
                cue,
                external
            )


            completed += 1
            bar.update(1)


            if completed % 50 == 0:

                elapsed = (
                    time.time()
                    - start
                )

                sec_trial = (
                    elapsed
                    / completed
                )

                eta = (
                    total_trials
                    - completed
                ) * sec_trial


                log(
                    f"{run_name}: "
                    f"{completed}/{total_trials} "
                    f"({100*completed/total_trials:.1f}%) | "
                    f"ETA ~ {eta/60:.1f} min"
                )


    # --------------------------------------------------------
    # TEST FEATURES
    # --------------------------------------------------------

    for noise_idx in range(
        len(
            TEST_NOISE_LEVELS
        )
    ):

        for cue_class in range(
            N_CLASSES
        ):

            for example in range(
                TEST_PER_CLASS
            ):

                cue = task[
                    "test_cues"
                ][
                    noise_idx,
                    cue_class,
                    example
                ]


                test_features[
                    noise_idx,
                    cue_class,
                    example
                ] = simulate_trial(
                    reservoir,
                    cue,
                    external
                )


                completed += 1
                bar.update(1)


                if completed % 50 == 0:

                    elapsed = (
                        time.time()
                        - start
                    )

                    sec_trial = (
                        elapsed
                        / completed
                    )

                    eta = (
                        total_trials
                        - completed
                    ) * sec_trial


                    log(
                        f"{run_name}: "
                        f"{completed}/{total_trials} "
                        f"({100*completed/total_trials:.1f}%) | "
                        f"ETA ~ {eta/60:.1f} min"
                    )


    bar.close()


    elapsed = (
        time.time()
        - start
    )


    log(
        f"{run_name}: "
        f"simulación terminada "
        f"en {elapsed/60:.1f} min."
    )


    return (
        train_features,
        test_features
    )


# ============================================================
# HEBBIAN ASSOCIATIVE READOUT
#
# W[label] += normalized reservoir state
#
# Se aprende ONLINE.
# No Ridge.
# No optimización iterativa posterior.
# ============================================================

def evaluate_associative_learning(
    task,
    train_features,
    test_features,
    task_id,
    network_id,
    network_name
):

    prototypes = np.zeros(
        (
            N_CLASSES,
            N_READOUT
        ),
        dtype=np.float32
    )


    counts = np.zeros(
        N_CLASSES,
        dtype=np.int64
    )


    rows = []


    # Pre-normalizar test
    normalized_tests = {}


    for noise_idx, noise in enumerate(
        TEST_NOISE_LEVELS
    ):

        X = (
            test_features[
                noise_idx
            ]
            .reshape(
                -1,
                N_READOUT
            )
        )


        y = (
            task[
                "test_labels"
            ][
                noise_idx
            ]
            .reshape(-1)
        )


        normalized_tests[
            noise
        ] = (
            normalize_rows(
                X
            ),
            y
        )


    # --------------------------------------------------------
    # Aprendizaje realmente incremental
    # --------------------------------------------------------

    for shot_idx in range(
        MAX_SHOTS
    ):

        # Una nueva exposición por cada cue
        for cue_class in range(
            N_CLASSES
        ):

            label = int(
                task[
                    "target_map"
                ][
                    cue_class
                ]
            )


            z = normalize_vector(
                train_features[
                    cue_class,
                    shot_idx
                ]
            )


            # Hebbian update
            prototypes[
                label
            ] += z

            counts[
                label
            ] += 1


        shots = (
            shot_idx
            + 1
        )


        if shots not in SHOT_CHECKPOINTS:

            continue


        # Normalizar prototipos acumulados
        proto_norm = normalize_rows(
            prototypes
        )


        for noise in TEST_NOISE_LEVELS:

            X_test, y_test = (
                normalized_tests[
                    noise
                ]
            )


            similarities = (
                X_test
                @ proto_norm.T
            )


            predictions = np.argmax(
                similarities,
                axis=1
            )


            accuracy = float(
                np.mean(
                    predictions
                    == y_test
                )
            )


            # ------------------------------------------------
            # Margin:
            # similitud target correcto - mejor incorrecto
            # ------------------------------------------------

            row_idx = np.arange(
                len(y_test)
            )


            correct_score = similarities[
                row_idx,
                y_test
            ]


            wrong_scores = (
                similarities.copy()
            )


            wrong_scores[
                row_idx,
                y_test
            ] = -np.inf


            best_wrong = np.max(
                wrong_scores,
                axis=1
            )


            margin = float(
                np.mean(
                    correct_score
                    - best_wrong
                )
            )


            rows.append({

                "task_id":
                    task_id,

                "task_seed":
                    TASK_SEEDS[
                        task_id
                    ],

                "network_id":
                    network_id,

                "network":
                    network_name,

                "shots":
                    shots,

                "noise":
                    noise,

                "accuracy":
                    accuracy,

                "margin":
                    margin,

                "chance":
                    1.0 / N_CLASSES
            })


            log(
                f"Task-{task_id} | "
                f"{network_name} | "
                f"shots={shots:2d} | "
                f"noise={noise:.0%} | "
                f"acc={accuracy:.4f} | "
                f"margin={margin:+.4f}"
            )


    return pd.DataFrame(
        rows
    )


# ============================================================
# RESULTADOS PREVIOS / RESUME
# ============================================================

if os.path.exists(
    RESULT_FILE
):

    results = pd.read_csv(
        RESULT_FILE
    )

else:

    results = pd.DataFrame()


if not results.empty:

    completed_pairs = set(

        (
            int(task_id),
            int(network_id)
        )

        for task_id, network_id
        in zip(
            results[
                "task_id"
            ],
            results[
                "network_id"
            ]
        )
    )

else:

    completed_pairs = set()


# ============================================================
# DEFINIR 18 EJECUCIONES
# ============================================================

runs = []


for task_id in TASK_SEEDS:

    for network_id in range(
        6
    ):

        runs.append(
            (
                task_id,
                network_id
            )
        )


completed_count = sum(
    pair in completed_pairs

    for pair in runs
)


log("")
log(
    f"Runs totales: {len(runs)}"
)

log(
    f"Runs completados: "
    f"{completed_count}/{len(runs)}"
)


global_bar = tqdm(
    total=len(runs),
    initial=completed_count,
    desc="PROGRESO GLOBAL v0.8",
    unit="run",
    dynamic_ncols=True
)


# ============================================================
# EJECUTAR
# ============================================================

for run_number, (
    task_id,
    network_id
) in enumerate(
    runs,
    start=1
):

    if (
        task_id,
        network_id
    ) in completed_pairs:

        log(
            f"Task-{task_id} / "
            f"{NETWORK_NAMES[network_id]}: "
            "ya terminado."
        )

        continue


    network_name = NETWORK_NAMES[
        network_id
    ]


    run_name = (
        f"T{task_id}_{network_name}"
    )


    log("")
    log("=" * 86)

    log(
        f"RUN {run_number}/{len(runs)} | "
        f"Task-{task_id} | "
        f"{network_name}"
    )

    log("=" * 86)


    # --------------------------------------------------------
    # Cargar grafo
    # --------------------------------------------------------

    if network_id == 0:

        graph = real_graph

        A_current = None


    else:

        log(
            f"Cargando {network_name}..."
        )

        A_current = sparse.load_npz(
            NETWORK_PATHS[
                network_id
            ]
        ).tocsr().astype(np.float32)


        graph = GraphProxy(
            A_current,
            base.nodes
        )


    log(
        f"{network_name}: "
        f"{graph.n_edges:,} conexiones"
    )


    # --------------------------------------------------------
    # Simular
    # --------------------------------------------------------

    task = tasks_data[
        task_id
    ]


    (
        train_features,
        test_features
    ) = generate_features(
        graph,
        task,
        run_name
    )


    # --------------------------------------------------------
    # Aprendizaje online
    # --------------------------------------------------------

    run_df = evaluate_associative_learning(
        task,
        train_features,
        test_features,
        task_id,
        network_id,
        network_name
    )


    # Guardar copia individual
    run_file = os.path.join(
        RUNDIR,
        f"task_{task_id:02d}_network_{network_id}.csv"
    )


    run_df.to_csv(
        run_file,
        index=False
    )


    # --------------------------------------------------------
    # Añadir a resultados
    # --------------------------------------------------------

    if results.empty:

        results = run_df.copy()

    else:

        results = pd.concat(
            [
                results,
                run_df
            ],
            ignore_index=True
        )


    # Quitar duplicados
    results = results.drop_duplicates(
        subset=[
            "task_id",
            "network_id",
            "shots",
            "noise"
        ],
        keep="last"
    )


    # Checkpoint casi atómico
    temp_file = (
        RESULT_FILE
        + ".tmp"
    )


    results.to_csv(
        temp_file,
        index=False
    )


    os.replace(
        temp_file,
        RESULT_FILE
    )


    completed_pairs.add(
        (
            task_id,
            network_id
        )
    )


    log(
        f"{run_name}: "
        "CHECKPOINT GUARDADO."
    )


    # --------------------------------------------------------
    # Limpieza
    # --------------------------------------------------------

    del train_features
    del test_features
    del run_df
    del graph


    if A_current is not None:

        del A_current


    gc.collect()


    global_bar.update(1)


global_bar.close()


# ============================================================
# VALIDAR EJECUCION COMPLETA
# ============================================================

results = pd.read_csv(
    RESULT_FILE
)


expected_rows = (
    len(TASK_SEEDS)
    * 6
    * len(SHOT_CHECKPOINTS)
    * len(TEST_NOISE_LEVELS)
)


if len(results) != expected_rows:

    log(
        f"Resultados: "
        f"{len(results)}/{expected_rows} filas."
    )

    log(
        "Todavía faltan runs. "
        "Ejecuta nuevamente el script."
    )

    raise SystemExit


# ============================================================
# RESUMEN POR RED / TAREA
# ============================================================

summary_rows = []


for task_id in TASK_SEEDS:

    for network_id in range(
        6
    ):

        subset = results.loc[
            (
                results[
                    "task_id"
                ] == task_id
            )
            &
            (
                results[
                    "network_id"
                ] == network_id
            )
        ]


        primary = subset.loc[
            np.isclose(
                subset[
                    "noise"
                ],
                PRIMARY_NOISE
            )
        ]


        assoc_score_10 = float(
            primary[
                "accuracy"
            ].mean()
        )


        fewshot_score_10 = float(
            primary.loc[
                primary[
                    "shots"
                ].isin(
                    [
                        1,
                        2,
                        4
                    ]
                ),
                "accuracy"
            ].mean()
        )


        final16_10 = float(
            primary.loc[
                primary[
                    "shots"
                ] == 16,
                "accuracy"
            ].iloc[0]
        )


        noise20 = subset.loc[
            np.isclose(
                subset[
                    "noise"
                ],
                0.20
            )
        ]


        assoc_score_20 = float(
            noise20[
                "accuracy"
            ].mean()
        )


        final16_20 = float(
            noise20.loc[
                noise20[
                    "shots"
                ] == 16,
                "accuracy"
            ].iloc[0]
        )


        clean = subset.loc[
            np.isclose(
                subset[
                    "noise"
                ],
                0.00
            )
        ]


        assoc_score_clean = float(
            clean[
                "accuracy"
            ].mean()
        )


        summary_rows.append({

            "task_id":
                task_id,

            "network_id":
                network_id,

            "network":
                NETWORK_NAMES[
                    network_id
                ],

            "assoc_score_clean":
                assoc_score_clean,

            "assoc_score_10":
                assoc_score_10,

            "fewshot_score_10":
                fewshot_score_10,

            "final16_acc_10":
                final16_10,

            "assoc_score_20":
                assoc_score_20,

            "final16_acc_20":
                final16_20
        })


network_summary = pd.DataFrame(
    summary_rows
)


network_summary.to_csv(
    NETWORK_SUMMARY_FILE,
    index=False
)


# ============================================================
# MALECNS VS NULLS POR TAREA
# ============================================================

task_rows = []


for task_id in TASK_SEEDS:

    task_df = network_summary.loc[
        network_summary[
            "task_id"
        ] == task_id
    ]


    real = task_df.loc[
        task_df[
            "network_id"
        ] == 0
    ].iloc[0]


    nulls = task_df.loc[
        task_df[
            "network_id"
        ] > 0
    ]


    null_primary = float(
        nulls[
            "assoc_score_10"
        ].mean()
    )


    delta_primary = (
        float(
            real[
                "assoc_score_10"
            ]
        )
        -
        null_primary
    )


    null_fewshot = float(
        nulls[
            "fewshot_score_10"
        ].mean()
    )


    delta_fewshot = (
        float(
            real[
                "fewshot_score_10"
            ]
        )
        -
        null_fewshot
    )


    null20 = float(
        nulls[
            "assoc_score_20"
        ].mean()
    )


    delta20 = (
        float(
            real[
                "assoc_score_20"
            ]
        )
        -
        null20
    )


    null16 = float(
        nulls[
            "final16_acc_10"
        ].mean()
    )


    wins_vs_nulls = int(
        np.sum(
            float(
                real[
                    "assoc_score_10"
                ]
            )
            >
            nulls[
                "assoc_score_10"
            ].to_numpy()
        )
    )


    task_rows.append({

        "task_id":
            task_id,

        "real_assoc_score_10":
            float(
                real[
                    "assoc_score_10"
                ]
            ),

        "null_mean_assoc_score_10":
            null_primary,

        "delta_assoc_score_10":
            delta_primary,

        "real_fewshot_score_10":
            float(
                real[
                    "fewshot_score_10"
                ]
            ),

        "null_mean_fewshot_score_10":
            null_fewshot,

        "delta_fewshot_score_10":
            delta_fewshot,

        "real_assoc_score_20":
            float(
                real[
                    "assoc_score_20"
                ]
            ),

        "null_mean_assoc_score_20":
            null20,

        "delta_assoc_score_20":
            delta20,

        "real_final16_acc_10":
            float(
                real[
                    "final16_acc_10"
                ]
            ),

        "null_mean_final16_acc_10":
            null16,

        "wins_vs_5_nulls":
            wins_vs_nulls
    })


task_summary = pd.DataFrame(
    task_rows
)


task_summary.to_csv(
    TASK_SUMMARY_FILE,
    index=False
)


# ============================================================
# GATE FINAL v0.8
# ============================================================

task_wins = int(
    np.sum(
        task_summary[
            "delta_assoc_score_10"
        ] > 0
    )
)


mean_delta_primary = float(
    task_summary[
        "delta_assoc_score_10"
    ].mean()
)


median_delta_primary = float(
    task_summary[
        "delta_assoc_score_10"
    ].median()
)


mean_delta_fewshot = float(
    task_summary[
        "delta_fewshot_score_10"
    ].mean()
)


mean_delta_noise20 = float(
    task_summary[
        "delta_assoc_score_20"
    ].mean()
)


real_final16_mean = float(
    task_summary[
        "real_final16_acc_10"
    ].mean()
)


null_final16_mean = float(
    task_summary[
        "null_mean_final16_acc_10"
    ].mean()
)


PRIMARY_PASS = (
    task_wins
    >= REQUIRED_TASK_WINS
)


MEAN_PASS = (
    mean_delta_primary
    > 0
)


FINAL_ASSOCIATIVE_GATE_PASS = (
    PRIMARY_PASS
    and
    MEAN_PASS
)


final_df = pd.DataFrame([{

    "chance":
        1.0 / N_CLASSES,

    "task_wins":
        task_wins,

    "tasks_total":
        len(TASK_SEEDS),

    "mean_delta_assoc_score_10":
        mean_delta_primary,

    "median_delta_assoc_score_10":
        median_delta_primary,

    "mean_delta_fewshot_score_10":
        mean_delta_fewshot,

    "mean_delta_assoc_score_20":
        mean_delta_noise20,

    "real_mean_final16_acc_10":
        real_final16_mean,

    "null_mean_final16_acc_10":
        null_final16_mean,

    "primary_pass":
        PRIMARY_PASS,

    "mean_pass":
        MEAN_PASS,

    "FINAL_ASSOCIATIVE_GATE_PASS":
        FINAL_ASSOCIATIVE_GATE_PASS
}])


final_df.to_csv(
    FINAL_FILE,
    index=False
)


# ============================================================
# LOG FINAL
# ============================================================

log("")
log("=" * 86)
log("RESULTADO FINAL v0.8")
log("=" * 86)


log(
    f"Chance accuracy: "
    f"{1/N_CLASSES:.3f}"
)


for row in task_summary.itertuples():

    log("")

    log(
        f"Task-{int(row.task_id)}"
    )

    log(
        f"  Associative score 10%:"
        f" MaleCNS={row.real_assoc_score_10:.4f}"
        f" | Null={row.null_mean_assoc_score_10:.4f}"
        f" | Δ={row.delta_assoc_score_10:+.4f}"
        f" | wins={int(row.wins_vs_5_nulls)}/5"
    )

    log(
        f"  Few-shot 1/2/4:"
        f" MaleCNS={row.real_fewshot_score_10:.4f}"
        f" | Null={row.null_mean_fewshot_score_10:.4f}"
        f" | Δ={row.delta_fewshot_score_10:+.4f}"
    )

    log(
        f"  Noise 20%:"
        f" MaleCNS={row.real_assoc_score_20:.4f}"
        f" | Null={row.null_mean_assoc_score_20:.4f}"
        f" | Δ={row.delta_assoc_score_20:+.4f}"
    )

    log(
        f"  16-shot / 10%:"
        f" MaleCNS={row.real_final16_acc_10:.4f}"
        f" | Null={row.null_mean_final16_acc_10:.4f}"
    )


log("")
log("------------------------------------------")

log(
    f"Tareas con ventaja MaleCNS: "
    f"{task_wins}/{len(TASK_SEEDS)}"
)

log(
    f"Media Δ associative score 10%: "
    f"{mean_delta_primary:+.4f}"
)

log(
    f"Mediana Δ associative score 10%: "
    f"{median_delta_primary:+.4f}"
)

log(
    f"Media Δ few-shot 1/2/4: "
    f"{mean_delta_fewshot:+.4f}"
)

log(
    f"Media Δ con 20% ruido: "
    f"{mean_delta_noise20:+.4f}"
)

log(
    f"16-shot 10% promedio:"
    f" MaleCNS={real_final16_mean:.4f}"
    f" | Null={null_final16_mean:.4f}"
)

log("------------------------------------------")

log(
    f"PRIMARY PASS >=2/3: "
    f"{PRIMARY_PASS}"
)

log(
    f"MEAN Δ > 0: "
    f"{MEAN_PASS}"
)

log(
    f"FINAL ASSOCIATIVE GATE PASS: "
    f"{FINAL_ASSOCIATIVE_GATE_PASS}"
)

log("------------------------------------------")


if FINAL_ASSOCIATIVE_GATE_PASS:

    log(
        "DECISION: el sustrato MaleCNS pasa "
        "a la siguiente etapa: plasticidad "
        "interna / aprendizaje adaptativo."
    )

else:

    log(
        "DECISION: no introducir todavía "
        "plasticidad recurrente. Analizar si "
        "la representación asociativa o el "
        "acoplamiento de lectura limita el aprendizaje."
    )


log("")
log("Archivos:")

log(
    RESULT_FILE
)

log(
    NETWORK_SUMMARY_FILE
)

log(
    TASK_SUMMARY_FILE
)

log(
    FINAL_FILE
)

log(
    LOG_FILE
)

log("")
log("FIN v0.8")

Writing /content/drive/MyDrive/malecns_ai_v0_1/benchmark_associative_learning_v08.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_associative_learning_v08.py

Streaming output truncated to the last 5000 lines.
T2_Null-1:  37% 155/416 [02:13<04:06,  1.06trial/s]
T2_Null-1:  38% 156/416 [02:15<04:19,  1.00trial/s]
T2_Null-1:  38% 157/416 [02:15<03:57,  1.09trial/s]
T2_Null-1:  38% 158/416 [02:16<03:47,  1.13trial/s]
T2_Null-1:  38% 159/416 [02:17<03:43,  1.15trial/s]
T2_Null-1:  38% 160/416 [02:18<03:40,  1.16trial/s]
T2_Null-1:  39% 161/416 [02:19<03:39,  1.16trial/s]
T2_Null-1:  39% 162/416 [02:19<03:25,  1.23trial/s]
T2_Null-1:  39% 163/416 [02:20<03:22,  1.25trial/s]
T2_Null-1:  39% 164/416 [02:21<03:20,  1.25trial/s]
T2_Null-1:  40% 165/416 [02:22<03:10,  1.32trial/s]
T2_Null-1:  40% 166/416 [02:22<03:03,  1.36trial/s]
T2_Null-1:  40% 167/416 [02:23<03:04,  1.35trial/s]
T2_Null-1:  40% 168/416 [02:24<03:10,  1.30trial/s]
T2_Null-1:  41% 169/416 [02:25<03:13,  1.27trial/s]
T2_Null-1:  41% 170/416 [02:26<03:50,  1.07trial/s]
T2_Null-1:  41% 171/416 [02:27<04:11,  1.03s/trial]
T2_Null-1:  41% 172/416 [02:28<04:14,  1.04s/trial]
T2_Null-1:  4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%%writefile /content/drive/MyDrive/malecns_ai_v0_1/benchmark_associative_rls_v08b.py

import os
import gc
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd

from scipy import sparse
from tqdm.auto import tqdm

from loader import MaleCNSGraph
from dynamics import ConnectomeReservoir


# ============================================================
# v0.8B — ONLINE RLS DIAGNOSTIC
#
# CAMBIO UNICO RESPECTO A v0.8:
#   Hebbian prototype -> online RLS multiclase
#
# El reservoir permanece CONGELADO.
# ============================================================


PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"

OUTDIR = os.path.join(
    PROJECT,
    "results_v08b_associative_rls"
)

FEATUREDIR = os.path.join(
    OUTDIR,
    "cached_features"
)

RUNDIR = os.path.join(
    OUTDIR,
    "runs"
)

os.makedirs(
    OUTDIR,
    exist_ok=True
)

os.makedirs(
    FEATUREDIR,
    exist_ok=True
)

os.makedirs(
    RUNDIR,
    exist_ok=True
)

os.chdir(PROJECT)


# ============================================================
# CONFIGURACION IDENTICA A v0.8
# ============================================================

BASE_SEED = 20260914

LEAK = 0.2
GAIN = 1.2

SIGN_MODE = "biological_fast"

N_INPUT = 256
N_READOUT = 512

N_CLASSES = 8

MAX_SHOTS = 16

SHOT_CHECKPOINTS = [
    1,
    2,
    4,
    8,
    16
]

CUE_STEPS = 4
DELAY_STEPS = 8

FEATURE_AVG_STEPS = 3

TRAIN_NOISE = 0.05

TEST_NOISE_LEVELS = [
    0.00,
    0.10,
    0.20
]

TEST_PER_CLASS = 12


TASK_SEEDS = {
    1: 20801001,
    2: 20802002,
    3: 20803003,
}


# ============================================================
# CAMBIO v0.8B
# ============================================================

RLS_RIDGE = 1e-2

# No forgetting:
RLS_FORGETTING = 1.0


# ============================================================
# GATE PRE-REGISTRADO
# ============================================================

PRIMARY_NOISE = 0.10

REQUIRED_TASK_WINS = 2


# ============================================================
# ARCHIVOS
# ============================================================

RESULT_FILE = os.path.join(
    OUTDIR,
    "v08b_results.csv"
)

NETWORK_SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v08b_network_summary.csv"
)

TASK_SUMMARY_FILE = os.path.join(
    OUTDIR,
    "v08b_task_comparison.csv"
)

FINAL_FILE = os.path.join(
    OUTDIR,
    "v08b_final_gate.csv"
)

LOG_FILE = os.path.join(
    OUTDIR,
    "v08b_run.log"
)


# ============================================================
# LOG
# ============================================================

def log(message=""):

    stamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    line = f"[{stamp}] {message}"

    tqdm.write(line)

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            line + "\n"
        )


log("=" * 88)

log(
    "MaleCNS-AI v0.8B — "
    "ONLINE RLS ASSOCIATIVE DIAGNOSTIC"
)

log("=" * 88)

log(
    "Único cambio conceptual: "
    "Hebbian readout -> RLS online."
)

log(
    f"RLS ridge lambda={RLS_RIDGE}"
)

log(
    "Reservoir completamente congelado."
)

log(
    "PASS diagnóstico: MaleCNS supera "
    "media Null en >=2/3 tareas y Δ medio > 0."
)


# ============================================================
# HASH
# ============================================================

def array_hash(arr):

    arr = np.ascontiguousarray(
        arr
    )

    return hashlib.sha256(
        arr.tobytes()
    ).hexdigest()[:16]


# ============================================================
# CARGAR MaleCNS
# ============================================================

log(
    "Cargando MaleCNSGraph..."
)

base = MaleCNSGraph()


REAL_PATH = os.path.join(
    PROJECT,
    "adjacency_binary_real_v02.npz"
)


A_real = sparse.load_npz(
    REAL_PATH
).tocsr().astype(np.float32)


n = A_real.shape[0]


log(
    f"Neuronas: {n:,}"
)

log(
    f"Conexiones: {A_real.nnz:,}"
)


# ============================================================
# GRAPH PROXY
# ============================================================

class GraphProxy:

    def __init__(
        self,
        A,
        nodes
    ):

        self.A = A
        self.nodes = nodes

        self.in_strength = np.asarray(
            A.sum(axis=0)
        ).ravel().astype(np.float32)

        self.out_strength = np.asarray(
            A.sum(axis=1)
        ).ravel().astype(np.float32)


    @property
    def n_nodes(self):

        return self.A.shape[0]


    @property
    def n_edges(self):

        return self.A.nnz


real_graph = GraphProxy(
    A_real,
    base.nodes
)


# ============================================================
# NETWORKS
# ============================================================

NETWORK_PATHS = {

    0:
        REAL_PATH,

    1:
        os.path.join(
            PROJECT,
            "adjacency_degree_matched_v02.npz"
        ),

    2:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_02.npz"
        ),

    3:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_03.npz"
        ),

    4:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_04.npz"
        ),

    5:
        os.path.join(
            PROJECT,
            "results_v03a",
            "null_graphs",
            "null_05.npz"
        )
}


NETWORK_NAMES = {

    0: "MaleCNS",
    1: "Null-1",
    2: "Null-2",
    3: "Null-3",
    4: "Null-4",
    5: "Null-5"
}


for network_id, path in (
    NETWORK_PATHS.items()
):

    if not os.path.exists(
        path
    ):

        raise FileNotFoundError(
            f"No existe "
            f"{NETWORK_NAMES[network_id]}:\n"
            f"{path}"
        )


# ============================================================
# INTERFACE-0 EXACTA
# ============================================================

INTERFACE_FILE = os.path.join(
    PROJECT,
    "results_v07_interface_robustness",
    "interfaces",
    "interface_00.npz"
)


if not os.path.exists(
    INTERFACE_FILE
):

    raise FileNotFoundError(
        "No encuentro interface_00.npz "
        "de v0.7."
    )


interface_data = np.load(
    INTERFACE_FILE
)


input_nodes = (
    interface_data[
        "input_nodes"
    ]
    .astype(np.int64)
)


readout_nodes = (
    interface_data[
        "readout_nodes"
    ]
    .astype(np.int64)
)


log(
    f"Input hash: "
    f"{array_hash(input_nodes)}"
)

log(
    f"Readout hash: "
    f"{array_hash(readout_nodes)}"
)


if len(
    np.intersect1d(
        input_nodes,
        readout_nodes
    )
) != 0:

    raise RuntimeError(
        "Input/readout se superponen."
    )


# ============================================================
# CARGAR EXACTAMENTE LAS TAREAS v0.8
# ============================================================

TASKDIR_V08 = os.path.join(
    PROJECT,
    "results_v08_associative_learning",
    "tasks"
)


def load_task(
    task_id
):

    path = os.path.join(
        TASKDIR_V08,
        f"task_{task_id:02d}.npz"
    )


    if not os.path.exists(
        path
    ):

        raise FileNotFoundError(
            f"No encuentro tarea v0.8:\n"
            f"{path}"
        )


    d = np.load(
        path
    )


    task = {

        "base_patterns":
            d[
                "base_patterns"
            ].astype(np.float32),

        "target_map":
            d[
                "target_map"
            ].astype(np.int64),

        "train_cues":
            d[
                "train_cues"
            ].astype(np.float32),

        "test_cues":
            d[
                "test_cues"
            ].astype(np.float32),

        "test_labels":
            d[
                "test_labels"
            ].astype(np.int64)
    }


    log(
        f"Task-{task_id}: cargada desde v0.8 | "
        f"map={task['target_map'].tolist()}"
    )

    log(
        f"Task-{task_id}: train hash="
        f"{array_hash(task['train_cues'])}"
    )

    log(
        f"Task-{task_id}: test hash="
        f"{array_hash(task['test_cues'])}"
    )


    return task


tasks_data = {

    task_id:
        load_task(
            task_id
        )

    for task_id
    in TASK_SEEDS
}


# ============================================================
# NORMALIZACION
# ============================================================

def normalize_vector(
    x
):

    x = np.asarray(
        x,
        dtype=np.float64
    )


    norm = np.linalg.norm(
        x
    )


    if norm < 1e-12:

        return np.zeros_like(
            x,
            dtype=np.float64
        )


    return (
        x / norm
    )


def normalize_rows(
    X
):

    X = np.asarray(
        X,
        dtype=np.float64
    )


    norms = np.linalg.norm(
        X,
        axis=1,
        keepdims=True
    )


    norms = np.maximum(
        norms,
        1e-12
    )


    return (
        X / norms
    )


# ============================================================
# SIMULAR TRIAL
# ============================================================

def simulate_trial(
    reservoir,
    cue,
    external
):

    reservoir.reset()


    feature_buffer = []


    total_steps = (
        CUE_STEPS
        +
        DELAY_STEPS
    )


    capture_from = (
        total_steps
        -
        FEATURE_AVG_STEPS
    )


    for t in range(
        total_steps
    ):

        external.fill(
            0.0
        )


        if t < CUE_STEPS:

            external[
                input_nodes
            ] = cue


        state = reservoir.step(
            external
        )


        if t >= capture_from:

            feature_buffer.append(
                state[
                    readout_nodes
                ].copy()
            )


    return np.mean(
        feature_buffer,
        axis=0
    ).astype(np.float32)


# ============================================================
# FEATURE CACHE
# ============================================================

def feature_path(
    task_id,
    network_id
):

    return os.path.join(
        FEATUREDIR,
        f"task_{task_id:02d}_"
        f"network_{network_id}.npz"
    )


def generate_or_load_features(
    graph,
    task,
    task_id,
    network_id,
    run_name
):

    path = feature_path(
        task_id,
        network_id
    )


    if os.path.exists(
        path
    ):

        log(
            f"{run_name}: "
            "features cacheadas encontradas."
        )


        data = np.load(
            path
        )


        train_features = (
            data[
                "train_features"
            ].astype(np.float32)
        )


        test_features = (
            data[
                "test_features"
            ].astype(np.float32)
        )


        log(
            f"{run_name}: train feature hash="
            f"{array_hash(train_features)}"
        )


        return (
            train_features,
            test_features
        )


    reservoir = ConnectomeReservoir(
        graph,
        leak=LEAK,
        gain=GAIN,
        sign_mode=SIGN_MODE
    )


    external = np.zeros(
        graph.n_nodes,
        dtype=np.float32
    )


    train_features = np.empty(
        (
            N_CLASSES,
            MAX_SHOTS,
            N_READOUT
        ),
        dtype=np.float32
    )


    test_features = np.empty(
        (
            len(
                TEST_NOISE_LEVELS
            ),
            N_CLASSES,
            TEST_PER_CLASS,
            N_READOUT
        ),
        dtype=np.float32
    )


    train_trials = (
        N_CLASSES
        * MAX_SHOTS
    )


    test_trials = (
        len(
            TEST_NOISE_LEVELS
        )
        * N_CLASSES
        * TEST_PER_CLASS
    )


    total_trials = (
        train_trials
        +
        test_trials
    )


    log(
        f"{run_name}: "
        f"{total_trials} trials | "
        f"{CUE_STEPS + DELAY_STEPS} "
        "pasos/trial"
    )


    bar = tqdm(
        total=total_trials,
        desc=run_name,
        unit="trial",
        dynamic_ncols=True,
        leave=False
    )


    start = time.time()

    completed = 0


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    for cue_class in range(
        N_CLASSES
    ):

        for shot in range(
            MAX_SHOTS
        ):

            cue = task[
                "train_cues"
            ][
                cue_class,
                shot
            ]


            train_features[
                cue_class,
                shot
            ] = simulate_trial(
                reservoir,
                cue,
                external
            )


            completed += 1

            bar.update(1)


            if completed % 50 == 0:

                elapsed = (
                    time.time()
                    - start
                )


                sec_trial = (
                    elapsed
                    / completed
                )


                eta = (
                    total_trials
                    - completed
                ) * sec_trial


                log(
                    f"{run_name}: "
                    f"{completed}/{total_trials} "
                    f"({100*completed/total_trials:.1f}%) | "
                    f"ETA ~ {eta/60:.1f} min"
                )


    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    for noise_idx in range(
        len(
            TEST_NOISE_LEVELS
        )
    ):

        for cue_class in range(
            N_CLASSES
        ):

            for example in range(
                TEST_PER_CLASS
            ):

                cue = task[
                    "test_cues"
                ][
                    noise_idx,
                    cue_class,
                    example
                ]


                test_features[
                    noise_idx,
                    cue_class,
                    example
                ] = simulate_trial(
                    reservoir,
                    cue,
                    external
                )


                completed += 1

                bar.update(1)


                if completed % 50 == 0:

                    elapsed = (
                        time.time()
                        - start
                    )


                    sec_trial = (
                        elapsed
                        / completed
                    )


                    eta = (
                        total_trials
                        - completed
                    ) * sec_trial


                    log(
                        f"{run_name}: "
                        f"{completed}/{total_trials} "
                        f"({100*completed/total_trials:.1f}%) | "
                        f"ETA ~ {eta/60:.1f} min"
                    )


    bar.close()


    elapsed = (
        time.time()
        - start
    )


    log(
        f"{run_name}: "
        f"simulación terminada "
        f"en {elapsed/60:.1f} min."
    )


    # Guardar features antes del clasificador
    temp_path = (
        path
        + ".tmp.npz"
    )


    np.savez_compressed(
        temp_path,
        train_features=train_features,
        test_features=test_features
    )


    os.replace(
        temp_path,
        path
    )


    log(
        f"{run_name}: "
        "FEATURE CACHE GUARDADO."
    )


    log(
        f"{run_name}: train hash="
        f"{array_hash(train_features)}"
    )


    return (
        train_features,
        test_features
    )


# ============================================================
# RLS ONLINE
# ============================================================

class OnlineRLSMulticlass:

    def __init__(
        self,
        n_features,
        n_classes,
        ridge=1e-2,
        forgetting=1.0
    ):

        self.n_features = (
            n_features
        )

        self.n_classes = (
            n_classes
        )

        self.ridge = float(
            ridge
        )

        self.forgetting = float(
            forgetting
        )


        # P0 = (lambda I)^-1
        self.P = (
            np.eye(
                n_features,
                dtype=np.float64
            )
            /
            self.ridge
        )


        self.W = np.zeros(
            (
                n_features,
                n_classes
            ),
            dtype=np.float64
        )


    def update(
        self,
        x,
        label
    ):

        x = np.asarray(
            x,
            dtype=np.float64
        )


        y = np.zeros(
            self.n_classes,
            dtype=np.float64
        )


        y[
            int(label)
        ] = 1.0


        Px = (
            self.P
            @ x
        )


        denom = (
            self.forgetting
            +
            x @ Px
        )


        k = (
            Px / denom
        )


        prediction = (
            x @ self.W
        )


        error = (
            y
            -
            prediction
        )


        self.W += (
            np.outer(
                k,
                error
            )
        )


        xTP = (
            x @ self.P
        )


        self.P = (
            self.P
            -
            np.outer(
                k,
                xTP
            )
        ) / self.forgetting


        # Corregir pequeñas asimetrías numéricas
        self.P = (
            0.5
            * (
                self.P
                +
                self.P.T
            )
        )


    def predict_scores(
        self,
        X
    ):

        return (
            X @ self.W
        )


# ============================================================
# EVALUAR RLS
# ============================================================

def evaluate_rls(
    task,
    train_features,
    test_features,
    task_id,
    network_id,
    network_name
):

    learner = OnlineRLSMulticlass(
        n_features=N_READOUT,
        n_classes=N_CLASSES,
        ridge=RLS_RIDGE,
        forgetting=RLS_FORGETTING
    )


    rows = []


    # --------------------------------------------------------
    # Test pre-normalizado
    # --------------------------------------------------------

    normalized_tests = {}


    for noise_idx, noise in enumerate(
        TEST_NOISE_LEVELS
    ):

        X = (
            test_features[
                noise_idx
            ]
            .reshape(
                -1,
                N_READOUT
            )
        )


        y = (
            task[
                "test_labels"
            ][
                noise_idx
            ]
            .reshape(-1)
        )


        normalized_tests[
            noise
        ] = (
            normalize_rows(
                X
            ),
            y
        )


    # --------------------------------------------------------
    # Incremental: 1 nueva exposición por clase / ronda
    # --------------------------------------------------------

    for shot_idx in range(
        MAX_SHOTS
    ):

        for cue_class in range(
            N_CLASSES
        ):

            label = int(
                task[
                    "target_map"
                ][
                    cue_class
                ]
            )


            x = normalize_vector(
                train_features[
                    cue_class,
                    shot_idx
                ]
            )


            learner.update(
                x,
                label
            )


        shots = (
            shot_idx
            + 1
        )


        if shots not in (
            SHOT_CHECKPOINTS
        ):

            continue


        for noise in (
            TEST_NOISE_LEVELS
        ):

            X_test, y_test = (
                normalized_tests[
                    noise
                ]
            )


            scores = (
                learner.predict_scores(
                    X_test
                )
            )


            prediction = np.argmax(
                scores,
                axis=1
            )


            accuracy = float(
                np.mean(
                    prediction
                    == y_test
                )
            )


            row_ids = np.arange(
                len(y_test)
            )


            correct_score = scores[
                row_ids,
                y_test
            ]


            wrong = (
                scores.copy()
            )


            wrong[
                row_ids,
                y_test
            ] = -np.inf


            best_wrong = np.max(
                wrong,
                axis=1
            )


            margin = float(
                np.mean(
                    correct_score
                    -
                    best_wrong
                )
            )


            rows.append({

                "task_id":
                    task_id,

                "task_seed":
                    TASK_SEEDS[
                        task_id
                    ],

                "network_id":
                    network_id,

                "network":
                    network_name,

                "shots":
                    shots,

                "noise":
                    noise,

                "accuracy":
                    accuracy,

                "margin":
                    margin,

                "chance":
                    1.0 / N_CLASSES,

                "learner":
                    "online_RLS",

                "ridge":
                    RLS_RIDGE
            })


            log(
                f"Task-{task_id} | "
                f"{network_name} | "
                f"shots={shots:2d} | "
                f"noise={noise:.0%} | "
                f"acc={accuracy:.4f} | "
                f"margin={margin:+.4f}"
            )


    return pd.DataFrame(
        rows
    )


# ============================================================
# RESUME
# ============================================================

if os.path.exists(
    RESULT_FILE
):

    results = pd.read_csv(
        RESULT_FILE
    )

else:

    results = pd.DataFrame()


if not results.empty:

    completed_pairs = set(

        (
            int(task_id),
            int(network_id)
        )

        for task_id, network_id
        in zip(
            results[
                "task_id"
            ],
            results[
                "network_id"
            ]
        )
    )

else:

    completed_pairs = set()


# ============================================================
# 18 RUNS
# ============================================================

runs = []


for task_id in TASK_SEEDS:

    for network_id in range(
        6
    ):

        runs.append(
            (
                task_id,
                network_id
            )
        )


completed_count = sum(
    run in completed_pairs

    for run in runs
)


log("")
log(
    f"Runs completos: "
    f"{completed_count}/18"
)


global_bar = tqdm(
    total=len(runs),
    initial=completed_count,
    desc="PROGRESO GLOBAL v0.8B",
    unit="run",
    dynamic_ncols=True
)


# ============================================================
# EJECUTAR
# ============================================================

for run_number, (
    task_id,
    network_id
) in enumerate(
    runs,
    start=1
):

    pair = (
        task_id,
        network_id
    )


    if pair in completed_pairs:

        log(
            f"Task-{task_id}/"
            f"{NETWORK_NAMES[network_id]} "
            "ya completado."
        )

        continue


    network_name = (
        NETWORK_NAMES[
            network_id
        ]
    )


    run_name = (
        f"T{task_id}_{network_name}"
    )


    log("")
    log("=" * 88)

    log(
        f"RUN {run_number}/18 | "
        f"Task-{task_id} | "
        f"{network_name}"
    )

    log("=" * 88)


    # --------------------------------------------------------
    # GRAPH
    # --------------------------------------------------------

    if network_id == 0:

        graph = real_graph

        A_current = None


    else:

        log(
            f"Cargando "
            f"{network_name}..."
        )


        A_current = sparse.load_npz(
            NETWORK_PATHS[
                network_id
            ]
        ).tocsr().astype(np.float32)


        graph = GraphProxy(
            A_current,
            base.nodes
        )


    log(
        f"{network_name}: "
        f"{graph.n_edges:,} conexiones"
    )


    # --------------------------------------------------------
    # EXACT SAME TASK
    # --------------------------------------------------------

    task = tasks_data[
        task_id
    ]


    # --------------------------------------------------------
    # FEATURES
    # --------------------------------------------------------

    (
        train_features,
        test_features
    ) = generate_or_load_features(
        graph,
        task,
        task_id,
        network_id,
        run_name
    )


    # --------------------------------------------------------
    # ONLINE RLS
    # --------------------------------------------------------

    run_df = evaluate_rls(
        task,
        train_features,
        test_features,
        task_id,
        network_id,
        network_name
    )


    run_path = os.path.join(
        RUNDIR,
        f"task_{task_id:02d}_"
        f"network_{network_id}.csv"
    )


    run_df.to_csv(
        run_path,
        index=False
    )


    if results.empty:

        results = (
            run_df.copy()
        )

    else:

        results = pd.concat(
            [
                results,
                run_df
            ],
            ignore_index=True
        )


    results = (
        results
        .drop_duplicates(
            subset=[
                "task_id",
                "network_id",
                "shots",
                "noise"
            ],
            keep="last"
        )
    )


    temp = (
        RESULT_FILE
        + ".tmp"
    )


    results.to_csv(
        temp,
        index=False
    )


    os.replace(
        temp,
        RESULT_FILE
    )


    completed_pairs.add(
        pair
    )


    log(
        f"{run_name}: "
        "CHECKPOINT GUARDADO."
    )


    del train_features
    del test_features
    del run_df
    del graph


    if A_current is not None:

        del A_current


    gc.collect()


    global_bar.update(1)


global_bar.close()


# ============================================================
# VALIDACION FINAL
# ============================================================

results = pd.read_csv(
    RESULT_FILE
)


expected_rows = (
    3
    * 6
    * len(
        SHOT_CHECKPOINTS
    )
    * len(
        TEST_NOISE_LEVELS
    )
)


if len(results) != expected_rows:

    log(
        f"Resultados incompletos: "
        f"{len(results)}/{expected_rows}"
    )

    log(
        "Ejecutar nuevamente "
        "para continuar."
    )

    raise SystemExit


# ============================================================
# NETWORK SUMMARY
# ============================================================

summary_rows = []


for task_id in TASK_SEEDS:

    for network_id in range(
        6
    ):

        subset = results.loc[
            (
                results[
                    "task_id"
                ]
                == task_id
            )
            &
            (
                results[
                    "network_id"
                ]
                == network_id
            )
        ]


        primary = subset.loc[
            np.isclose(
                subset[
                    "noise"
                ],
                0.10
            )
        ]


        clean = subset.loc[
            np.isclose(
                subset[
                    "noise"
                ],
                0.00
            )
        ]


        noise20 = subset.loc[
            np.isclose(
                subset[
                    "noise"
                ],
                0.20
            )
        ]


        assoc10 = float(
            primary[
                "accuracy"
            ].mean()
        )


        few10 = float(
            primary.loc[
                primary[
                    "shots"
                ].isin(
                    [
                        1,
                        2,
                        4
                    ]
                ),
                "accuracy"
            ].mean()
        )


        final16_10 = float(
            primary.loc[
                primary[
                    "shots"
                ] == 16,
                "accuracy"
            ].iloc[0]
        )


        assoc20 = float(
            noise20[
                "accuracy"
            ].mean()
        )


        final16_20 = float(
            noise20.loc[
                noise20[
                    "shots"
                ] == 16,
                "accuracy"
            ].iloc[0]
        )


        assoc_clean = float(
            clean[
                "accuracy"
            ].mean()
        )


        summary_rows.append({

            "task_id":
                task_id,

            "network_id":
                network_id,

            "network":
                NETWORK_NAMES[
                    network_id
                ],

            "assoc_score_clean":
                assoc_clean,

            "assoc_score_10":
                assoc10,

            "fewshot_score_10":
                few10,

            "final16_acc_10":
                final16_10,

            "assoc_score_20":
                assoc20,

            "final16_acc_20":
                final16_20
        })


network_summary = pd.DataFrame(
    summary_rows
)


network_summary.to_csv(
    NETWORK_SUMMARY_FILE,
    index=False
)


# ============================================================
# TASK COMPARISON
# ============================================================

task_rows = []


for task_id in TASK_SEEDS:

    subset = network_summary.loc[
        network_summary[
            "task_id"
        ] == task_id
    ]


    real = subset.loc[
        subset[
            "network_id"
        ] == 0
    ].iloc[0]


    nulls = subset.loc[
        subset[
            "network_id"
        ] > 0
    ]


    real_primary = float(
        real[
            "assoc_score_10"
        ]
    )


    null_primary = float(
        nulls[
            "assoc_score_10"
        ].mean()
    )


    real_few = float(
        real[
            "fewshot_score_10"
        ]
    )


    null_few = float(
        nulls[
            "fewshot_score_10"
        ].mean()
    )


    real20 = float(
        real[
            "assoc_score_20"
        ]
    )


    null20 = float(
        nulls[
            "assoc_score_20"
        ].mean()
    )


    real16 = float(
        real[
            "final16_acc_10"
        ]
    )


    null16 = float(
        nulls[
            "final16_acc_10"
        ].mean()
    )


    wins = int(
        np.sum(
            real_primary
            >
            nulls[
                "assoc_score_10"
            ].to_numpy()
        )
    )


    task_rows.append({

        "task_id":
            task_id,

        "real_assoc_score_10":
            real_primary,

        "null_mean_assoc_score_10":
            null_primary,

        "delta_assoc_score_10":
            real_primary
            -
            null_primary,

        "real_fewshot_score_10":
            real_few,

        "null_mean_fewshot_score_10":
            null_few,

        "delta_fewshot_score_10":
            real_few
            -
            null_few,

        "real_assoc_score_20":
            real20,

        "null_mean_assoc_score_20":
            null20,

        "delta_assoc_score_20":
            real20
            -
            null20,

        "real_final16_acc_10":
            real16,

        "null_mean_final16_acc_10":
            null16,

        "delta_final16_acc_10":
            real16
            -
            null16,

        "wins_vs_5_nulls":
            wins
    })


task_summary = pd.DataFrame(
    task_rows
)


task_summary.to_csv(
    TASK_SUMMARY_FILE,
    index=False
)


# ============================================================
# FINAL DIAGNOSTIC GATE
# ============================================================

task_wins = int(
    np.sum(
        task_summary[
            "delta_assoc_score_10"
        ] > 0
    )
)


mean_delta = float(
    task_summary[
        "delta_assoc_score_10"
    ].mean()
)


median_delta = float(
    task_summary[
        "delta_assoc_score_10"
    ].median()
)


mean_few_delta = float(
    task_summary[
        "delta_fewshot_score_10"
    ].mean()
)


mean_noise20_delta = float(
    task_summary[
        "delta_assoc_score_20"
    ].mean()
)


mean_final16_delta = float(
    task_summary[
        "delta_final16_acc_10"
    ].mean()
)


PRIMARY_PASS = (
    task_wins
    >= REQUIRED_TASK_WINS
)


MEAN_PASS = (
    mean_delta
    > 0
)


FINAL_RLS_GATE_PASS = (
    PRIMARY_PASS
    and
    MEAN_PASS
)


final_df = pd.DataFrame([{

    "chance":
        1.0 / N_CLASSES,

    "rls_ridge":
        RLS_RIDGE,

    "task_wins":
        task_wins,

    "tasks_total":
        3,

    "mean_delta_assoc_score_10":
        mean_delta,

    "median_delta_assoc_score_10":
        median_delta,

    "mean_delta_fewshot_score_10":
        mean_few_delta,

    "mean_delta_assoc_score_20":
        mean_noise20_delta,

    "mean_delta_final16_acc_10":
        mean_final16_delta,

    "primary_pass":
        PRIMARY_PASS,

    "mean_pass":
        MEAN_PASS,

    "FINAL_RLS_GATE_PASS":
        FINAL_RLS_GATE_PASS
}])


final_df.to_csv(
    FINAL_FILE,
    index=False
)


# ============================================================
# FINAL LOG
# ============================================================

log("")
log("=" * 88)
log("RESULTADO FINAL v0.8B")
log("=" * 88)


for row in task_summary.itertuples():

    log("")

    log(
        f"Task-{int(row.task_id)}"
    )


    log(
        f"  RLS associative 10%: "
        f"MaleCNS="
        f"{row.real_assoc_score_10:.4f} | "
        f"Null="
        f"{row.null_mean_assoc_score_10:.4f} | "
        f"Δ="
        f"{row.delta_assoc_score_10:+.4f} | "
        f"wins="
        f"{int(row.wins_vs_5_nulls)}/5"
    )


    log(
        f"  Few-shot 1/2/4: "
        f"MaleCNS="
        f"{row.real_fewshot_score_10:.4f} | "
        f"Null="
        f"{row.null_mean_fewshot_score_10:.4f} | "
        f"Δ="
        f"{row.delta_fewshot_score_10:+.4f}"
    )


    log(
        f"  Noise 20%: "
        f"MaleCNS="
        f"{row.real_assoc_score_20:.4f} | "
        f"Null="
        f"{row.null_mean_assoc_score_20:.4f} | "
        f"Δ="
        f"{row.delta_assoc_score_20:+.4f}"
    )


    log(
        f"  16-shot 10%: "
        f"MaleCNS="
        f"{row.real_final16_acc_10:.4f} | "
        f"Null="
        f"{row.null_mean_final16_acc_10:.4f} | "
        f"Δ="
        f"{row.delta_final16_acc_10:+.4f}"
    )


log("")
log("-------------------------------------------")

log(
    f"Tareas favorables: "
    f"{task_wins}/3"
)

log(
    f"Media Δ associative 10%: "
    f"{mean_delta:+.4f}"
)

log(
    f"Mediana Δ associative 10%: "
    f"{median_delta:+.4f}"
)

log(
    f"Media Δ few-shot: "
    f"{mean_few_delta:+.4f}"
)

log(
    f"Media Δ ruido 20%: "
    f"{mean_noise20_delta:+.4f}"
)

log(
    f"Media Δ 16-shot: "
    f"{mean_final16_delta:+.4f}"
)

log("-------------------------------------------")

log(
    f"PRIMARY PASS >=2/3: "
    f"{PRIMARY_PASS}"
)

log(
    f"MEAN Δ > 0: "
    f"{MEAN_PASS}"
)

log(
    f"FINAL RLS GATE PASS: "
    f"{FINAL_RLS_GATE_PASS}"
)

log("-------------------------------------------")


if FINAL_RLS_GATE_PASS:

    log(
        "INTERPRETACION PRE-REGISTRADA: "
        "la representación MaleCNS contiene "
        "información asociativa explotable; "
        "el Hebbian readout de v0.8 era "
        "demasiado restrictivo."
    )

else:

    log(
        "INTERPRETACION PRE-REGISTRADA: "
        "un readout lineal online más potente "
        "no recupera ventaja asociativa. "
        "El siguiente paso será plasticidad "
        "dentro del núcleo recurrente."
    )


log("")
log("Archivos:")

log(
    RESULT_FILE
)

log(
    NETWORK_SUMMARY_FILE
)

log(
    TASK_SUMMARY_FILE
)

log(
    FINAL_FILE
)

log(
    LOG_FILE
)

log("")
log("FIN v0.8B")

Writing /content/drive/MyDrive/malecns_ai_v0_1/benchmark_associative_rls_v08b.py


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_associative_rls_v08b.py

Streaming output truncated to the last 5000 lines.
T2_Null-1:  42% 176/416 [02:28<03:37,  1.10trial/s]
T2_Null-1:  43% 177/416 [02:29<03:47,  1.05trial/s]
T2_Null-1:  43% 178/416 [02:30<03:56,  1.01trial/s]
T2_Null-1:  43% 179/416 [02:31<03:55,  1.01trial/s]
T2_Null-1:  43% 180/416 [02:32<03:34,  1.10trial/s]
T2_Null-1:  44% 181/416 [02:32<03:20,  1.17trial/s]
T2_Null-1:  44% 182/416 [02:33<03:19,  1.17trial/s]
T2_Null-1:  44% 183/416 [02:34<03:19,  1.17trial/s]
T2_Null-1:  44% 184/416 [02:35<03:16,  1.18trial/s]
T2_Null-1:  44% 185/416 [02:36<03:06,  1.24trial/s]
T2_Null-1:  45% 186/416 [02:36<03:04,  1.25trial/s]
T2_Null-1:  45% 187/416 [02:37<03:08,  1.21trial/s]
T2_Null-1:  45% 188/416 [02:38<03:11,  1.19trial/s]
T2_Null-1:  45% 189/416 [02:39<03:09,  1.20trial/s]
T2_Null-1:  46% 190/416 [02:40<03:00,  1.25trial/s]
T2_Null-1:  46% 191/416 [02:41<03:04,  1.22trial/s]
T2_Null-1:  46% 192/416 [02:42<03:36,  1.03trial/s]
T2_Null-1:  46% 193/416 [02:43<03:47,  1.02s/trial]
T2_Null-1:  4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_internal_plasticity_v09.py

/content
[2026-09-16 16:47:47] ========================================================================================
[2026-09-16 16:47:47] MaleCNS-AI v0.9 — INTERNAL RECURRENT PLASTICITY
[2026-09-16 16:47:47] ========================================================================================
[2026-09-16 16:47:47] No external learned decoder.
[2026-09-16 16:47:47] 512 neuronas internas -> 8 ensembles x 64.
[2026-09-16 16:47:47] eta=0.05 | desired correct=0.5 | other=-0.1
[2026-09-16 16:47:47] weight bounds=[0.25, 2.0]
[2026-09-16 16:47:50] Input hash=e764a7ec893ef04d
[2026-09-16 16:47:50] Association hash=7c0ba86bd6427aa6
[2026-09-16 16:47:50] Task-1: map=[2, 5, 4, 3, 1, 7, 6, 0]
[2026-09-16 16:47:50] Task-2: map=[2, 1, 5, 0, 4, 7, 3, 6]
[2026-09-16 16:47:50] Task-3: map=[5, 1, 3, 0, 4, 7, 6, 2]
[2026-09-16 16:47:50] Task-1/MaleCNS ya completado.
[2026-09-16 16:47:50] Task-1/Null-1 ya completado.
[2026-09-16 16:47:50] 
[2026-09-16 16:47:50] ======================================

In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_internal_plasticity_v09.py

[Errno 2] No such file or directory: '/content/drive/MyDrive/malecns_ai_v0_1'
/content
python3: can't open file '/content/benchmark_internal_plasticity_v09.py': [Errno 2] No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, sys

PROJECT = "/content/drive/MyDrive/malecns_ai_v0_1"
os.chdir(PROJECT)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print("Proyecto listo:", os.getcwd())

Mounted at /content/drive
Proyecto listo: /content/drive/MyDrive/malecns_ai_v0_1


In [ ]:
%cd /content/drive/MyDrive/malecns_ai_v0_1
!python -u benchmark_internal_plasticity_v09.py

Streaming output truncated to the last 5000 lines.
T3_MaleCNS:  63% 568/896 [04:51<02:35,  2.11trial/s]
T3_MaleCNS:  64% 569/896 [04:51<02:35,  2.11trial/s]
T3_MaleCNS:  64% 570/896 [04:52<02:32,  2.14trial/s]
T3_MaleCNS:  64% 571/896 [04:52<02:30,  2.16trial/s]
T3_MaleCNS:  64% 572/896 [04:53<02:33,  2.12trial/s]
T3_MaleCNS:  64% 573/896 [04:53<02:52,  1.88trial/s]
T3_MaleCNS:  64% 574/896 [04:54<03:07,  1.72trial/s]
T3_MaleCNS:  64% 575/896 [04:55<03:17,  1.62trial/s]
T3_MaleCNS:  64% 576/896 [04:55<03:13,  1.65trial/s]
T3_MaleCNS:  64% 577/896 [04:56<02:58,  1.79trial/s]
T3_MaleCNS:  65% 578/896 [04:56<02:48,  1.89trial/s]
T3_MaleCNS:  65% 579/896 [04:57<02:45,  1.91trial/s]
T3_MaleCNS:  65% 580/896 [04:57<02:40,  1.97trial/s]
T3_MaleCNS:  65% 581/896 [04:58<02:39,  1.97trial/s]
T3_MaleCNS:  65% 582/896 [04:58<02:36,  2.00trial/s]
T3_MaleCNS:  65% 583/896 [04:59<02:36,  1.99trial/s]
T3_MaleCNS:  65% 584/896 [04:59<02:34,  2.03trial/s]
T3_MaleCNS:  65% 585/896 [05:00<02:29,  2.08tria